In [3]:
"""
anomaly_cleaning.py
Sensor-level anomaly detection, removal, and forecasting-based detection.

Reuses data loading / anomaly injection / training utilities from
anomlay_pipeline.py (keep both files in the same folder, or on PYTHONPATH).

Both paradigms flag anomalies at the (timestep, sensor) cell level:
  supervised   : per-cell classifier trained on injected labels
  unsupervised : reconstruction / forecasting error vs a per-sensor threshold

Flow: load -> scale -> inject (val/test, + small frac into train for supervised)
      -> train -> score every cell -> flag -> report metrics on the flagged mask
      -> remove flagged readings -> write clean long-format CSV.

Graph usage:
  predefined adjacency : SpatioTemporalGNN, GNNForecaster
  learned graph        : GNN_LearnableGraph, GAT_LearnedGraph, GraphAnomalyVAE
  no graph             : TemporalCNN, StackedLSTM
"""
import os, sys, argparse, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score)

warnings.filterwarnings("ignore")

# --- shared pieces from the original pipeline --------------------------------
try:
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)
except ModuleNotFoundError:
    _here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, _here)
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)

try:
    from tabulate import tabulate
    _TAB = True
except ImportError:
    _TAB = False


# =============================================================================
# DATA
# =============================================================================
def _norm_adj(adj_matrix):
    """Symmetric-normalised adjacency with self-loops: D^-1/2 (A+I) D^-1/2."""
    A = (torch.tensor(np.asarray(adj_matrix), dtype=torch.float32) > 0).float()
    A = A + torch.eye(A.shape[0])
    d = A.sum(1).clamp(min=1e-6).pow(-0.5)
    return d[:, None] * A * d[None, :]


def inject(values, frac, atype, adj):
    """Inject anomalies and return (corrupted, cell_mask) where mask[t,s]=1 if
    that exact reading was altered. Mask is derived by diffing the clean array."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr, _ = inject_anomalies(values, frac, atype, "", adj)
    return corr, (corr != values).astype(int)


class WindowDS(Dataset):
    """Sliding windows of (seq_len, N) with matching per-cell labels."""
    def __init__(self, values, cell_labels, seq_len, stride):
        self.x, self.y = [], []
        T = len(values)
        starts = list(range(0, T - seq_len + 1, stride))
        if starts and starts[-1] != T - seq_len:
            starts.append(T - seq_len)
        for s in starts:
            self.x.append(torch.tensor(values[s:s + seq_len], dtype=torch.float32))
            self.y.append(torch.tensor(cell_labels[s:s + seq_len], dtype=torch.float32))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]


# =============================================================================
# MODELS  (all output per-cell logits (B,T,N) for sup, or expose cell scores)
# =============================================================================
class TemporalCNN(nn.Module):
    def __init__(self, N, seq_len, hidden=32, levels=4, k=3):
        super().__init__()
        chs = [N] + [hidden] * levels
        self.tcn = nn.Sequential(*[TCNBlock(chs[i], chs[i + 1], k, 2 ** i) for i in range(levels)])
        self.head = nn.Conv1d(hidden, N, 1)

    def forward(self, x):                                   # (B,T,N)
        h = self.tcn(x.permute(0, 2, 1))                    # (B,hidden,T)
        return self.head(h).permute(0, 2, 1)                # (B,T,N)


class StackedLSTM(nn.Module):
    def __init__(self, N, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(N, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, N)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out)                               # (B,T,N)


class GCNDetector(nn.Module):
    """GCN + per-node GRU per-cell classifier. Fixed graph if adj given, else learned."""
    def __init__(self, N, adj=None, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N, self.learn = N, adj is None
        if self.learn:
            self.adj_logits = nn.Parameter(torch.randn(N, N))
        else:
            self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def _A(self, device):
        if not self.learn:
            return self.A_hat
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def forward(self, x):
        B, T, N = x.shape
        feats = self.W(x.unsqueeze(-1))                     # (B,T,N,H)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device), feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)                              # (B*N,T,rnn)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GATDetector(nn.Module):
    """Learned graph: per-timestep multi-head attention over sensors (GAT on a
    fully-connected graph), then per-node GRU, then per-cell head."""
    def __init__(self, N, hidden=16, heads=2, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.proj = nn.Linear(1, hidden)
        self.attn = nn.MultiheadAttention(hidden, heads, batch_first=True, dropout=0.1)
        self.rnn = nn.GRU(hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):
        B, T, N = x.shape
        f = self.proj(x.reshape(B * T, N, 1))               # (B*T,N,hidden)
        a, _ = self.attn(f, f, f)                           # attention = learned graph
        a = torch.relu(a).reshape(B, T, N, -1)
        seq = a.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GraphAnomalyVAE(nn.Module):
    """Unsupervised graph VAE (learned adjacency). Cell score = reconstruction MSE."""
    def __init__(self, N, seq_len, gcn_hidden=16, rnn_hidden=32, latent=16, beta=1.0):
        super().__init__()
        self.N, self.seq_len, self.beta = N, seq_len, beta
        self.adj_logits = nn.Parameter(torch.randn(N, N))
        self.enc_proj = nn.Linear(1, gcn_hidden, bias=False)
        self.enc_rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.fc_mu = nn.Linear(rnn_hidden, latent)
        self.fc_lv = nn.Linear(rnn_hidden, latent)
        self.dec_fc = nn.Linear(latent, rnn_hidden)
        self.dec_rnn = nn.GRU(rnn_hidden, rnn_hidden, batch_first=True)
        self.dec_W = nn.Linear(rnn_hidden, gcn_hidden, bias=False)
        self.dec_out = nn.Linear(gcn_hidden, 1)

    def _A(self, device):
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def encode(self, x):
        B, T, N = x.shape
        f = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device),
                                    self.enc_proj(x.unsqueeze(-1))))
        seq = f.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        _, h = self.enc_rnn(seq)
        h = h.squeeze(0).reshape(B, N, -1).mean(1)
        return self.fc_mu(h), self.fc_lv(h)

    def decode(self, z):
        h = self.dec_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.dec_rnn(h)
        node = self.dec_W(out).unsqueeze(2).repeat(1, 1, self.N, 1)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(z.device), node))
        return self.dec_out(prop).squeeze(-1)               # (B,T,N)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return self.decode(z), mu, lv

    def loss(self, x):
        recon, mu, lv = self.forward(x)
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        return F.mse_loss(recon, x) + self.beta * kl

    def cell_score(self, x):
        recon, _, _ = self.forward(x)
        return (x - recon).pow(2)                           # (B,T,N)


class GNNForecaster(nn.Module):
    """Unsupervised one-step forecaster on the predefined graph. Trained on clean
    data; per-sensor score = squared next-step prediction error."""
    def __init__(self, N, adj, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):                                   # predict step T from steps 0..T-2
        B, T, N = x.shape
        xi = x[:, :-1, :]
        feats = self.W(xi.unsqueeze(-1))
        prop = torch.relu(torch.einsum("ij,btjh->btih", self.A_hat, feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
        out, _ = self.rnn(seq)
        return self.head(out[:, -1, :]).reshape(B, N)       # (B,N)

    def loss(self, x):
        return F.mse_loss(self.forward(x), x[:, -1, :])

    def step_score(self, x):
        return (self.forward(x) - x[:, -1, :]).pow(2)       # (B,N) at last step


# =============================================================================
# TRAIN / SCORE / FLAG / EVAL
# =============================================================================
def train_model(model, train_loader, val_loader, mode, device,
                lr=1e-3, epochs=50, patience=10, pos_weight=None):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device) if pos_weight is not None else None)
    stop = EarlyStopping(patience=patience, mode="f1" if mode == "sup" else "loss")

    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(device)
            opt.zero_grad()
            loss = crit(model(x), y.to(device)) if mode == "sup" else model.loss(x)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            if mode == "sup":
                ps, ys = [], []
                for x, y in val_loader:
                    ps.append(torch.sigmoid(model(x.to(device))).cpu().numpy().reshape(-1))
                    ys.append(y.numpy().reshape(-1))
                ys = np.concatenate(ys)
                # Ranking metric (AUPRC): tracks learning even when no fixed 0.5
                # cut separates the classes. F1@0.5 stays 0 on rare-positive data
                # and would stop training immediately.
                metric = average_precision_score(ys, np.concatenate(ps)) if ys.any() else 0.0
            else:
                metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in val_loader]))
        if stop.step(metric, model):
            stopped = ep + 1
            break
    else:
        stopped = epochs
    stop.restore(model)
    print(f"   trained {stopped} epoch(s) | best {'val_AUPRC' if mode == 'sup' else 'val_loss'}={stop.best:.4f}")
    return model


def score_series(model, values, seq_len, stride, mode, device):
    """Return a (T, N) anomaly-score map, averaging overlapping windows."""
    model.eval()
    T, N = values.shape
    if mode == "forecast":
        stride = 1                                # score every incoming step
    acc = np.zeros((T, N)); cnt = np.zeros((T, N))
    starts = list(range(0, T - seq_len + 1, stride))
    if starts and starts[-1] != T - seq_len:
        starts.append(T - seq_len)
    with torch.no_grad():
        for s in starts:
            x = torch.tensor(values[s:s + seq_len], dtype=torch.float32)[None].to(device)
            if mode == "sup":
                sc = torch.sigmoid(model(x))[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            elif mode == "vae":
                sc = model.cell_score(x)[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            else:                                            # forecast: last step only
                t = s + seq_len - 1
                acc[t] += model.step_score(x)[0].cpu().numpy(); cnt[t] += 1
    cnt[cnt == 0] = 1
    return acc / cnt


def flag_supervised(val_scores, val_mask, test_scores):
    """Global probability threshold tuned for F1 on the labelled val set."""
    thr, best = float(np.median(val_scores)), -1.0
    for t in np.linspace(val_scores.min(), val_scores.max(), 100):
        f = f1_score(val_mask.reshape(-1), (val_scores >= t).reshape(-1).astype(int), zero_division=0)
        if f > best:
            best, thr = f, t
    return (test_scores >= thr).astype(int), float(thr)


def flag_unsupervised(clean_scores, test_scores, pct):
    """Per-sensor threshold = pct-th percentile of clean-train scores for that sensor."""
    thr = np.percentile(clean_scores, pct, axis=0)           # (N,)
    return (test_scores >= thr).astype(int), thr


def cell_metrics(pred_mask, true_mask):
    p = pred_mask.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    return {"Accuracy":  round(accuracy_score(t, p), 4),
            "Precision": round(precision_score(t, p, zero_division=0), 4),
            "Recall":    round(recall_score(t, p, zero_division=0), 4),
            "F1":        round(f1_score(t, p, zero_division=0), 4)}


def ranking_metrics(scores, true_mask):
    """Threshold-free quality of the score map (does it rank anomalies high?)."""
    s = scores.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    if t.sum() == 0 or t.sum() == len(t):
        return {"AUROC": float("nan"), "AUPRC": float("nan")}
    return {"AUROC": round(roc_auc_score(t, s), 4),
            "AUPRC": round(average_precision_score(t, s), 4)}


# =============================================================================
# CLEAN OUTPUT
# =============================================================================
def _sensor_meta(csv_path):
    try:
        df = pd.read_csv(csv_path)
        cols = [c for c in ("thing_name", "thing_ip") if c in df.columns]
        if not cols:
            return {}
        g = df.drop_duplicates("sensor_name").set_index("sensor_name")[cols]
        return {s: row.to_dict() for s, row in g.iterrows()}
    except Exception:
        return {}


def export_clean(corrupted, scaler, pred_mask, dt_index, sensors, meta, out_path):
    """Drop flagged (datetime, sensor) readings; write the rest as long-format CSV."""
    inv = scaler.inverse_transform(corrupted)
    rows = []
    for ti, dt in enumerate(dt_index):
        dt = pd.Timestamp(dt)
        for sj, sname in enumerate(sensors):
            if pred_mask[ti, sj]:
                continue
            m = meta.get(sname, {})
            rows.append({
                "datetime": dt, "date": dt.strftime("%Y-%m-%d"),
                "time": dt.strftime("%H:%M"), "seconds": dt.timestamp(),
                "state": float(inv[ti, sj]), "sensor_name": sname,
                "thing_name": m.get("thing_name"), "thing_ip": m.get("thing_ip"),
            })
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    return df


# =============================================================================
# REGISTRY & ORCHESTRATION
# =============================================================================
def build_registry(N, seq_len, adj, beta):
    return [
        {"name": "TemporalCNN",        "mode": "sup",      "graph": "none",
         "build": lambda: TemporalCNN(N, seq_len)},
        {"name": "StackedLSTM",        "mode": "sup",      "graph": "none",
         "build": lambda: StackedLSTM(N)},
        {"name": "SpatioTemporalGNN",  "mode": "sup",      "graph": "predefined",
         "build": lambda: GCNDetector(N, adj=adj)},
        {"name": "GNN_LearnableGraph", "mode": "sup",      "graph": "learned",
         "build": lambda: GCNDetector(N, adj=None)},
        {"name": "GAT_LearnedGraph",   "mode": "sup",      "graph": "learned",
         "build": lambda: GATDetector(N)},
        {"name": "GraphAnomalyVAE",    "mode": "vae",      "graph": "learned",
         "build": lambda: GraphAnomalyVAE(N, seq_len, beta=beta)},
        {"name": "GNNForecaster",      "mode": "forecast", "graph": "predefined",
         "build": lambda: GNNForecaster(N, adj=adj)},
    ]


def run(csv_path, adj_csv_path,
        seq_len=24, stride=4, batch_size=64, max_epochs=50, lr=1e-3, patience=15,
        anomaly_frac=0.10, train_anomaly_frac=0.05, anomaly_type="all",
        unsup_percentile=99.0, pos_weight_cap=10.0, train_frac=0.70, val_frac=0.15,
        out_dir="cleaned_output", models=None, beta=1.0, seed=42):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*64}\n  Sensor-level anomaly cleaning  |  device={device}\n{'='*64}")
    wide = load_and_pivot(csv_path)
    sensors = list(wide.columns); N = len(sensors)
    raw = wide.values.astype(np.float32); T = len(raw)
    n_tr, n_va = int(T * train_frac), int(T * val_frac)

    scaler = MinMaxScaler()
    tr = scaler.fit_transform(raw[:n_tr])
    va = scaler.transform(raw[n_tr:n_tr + n_va])
    te = scaler.transform(raw[n_tr + n_va:])
    dt_test = wide.index[n_tr + n_va:]

    edge_index, adj = load_edge_index_from_csv(adj_csv_path, sensors)

    print("\nInjecting anomalies (val / test, and a small fraction into train):")
    va_c, va_m = inject(va, anomaly_frac, anomaly_type, adj)
    te_c, te_m = inject(te, anomaly_frac, anomaly_type, adj)
    tr_c, tr_m = inject(tr, train_anomaly_frac, anomaly_type, adj)
    meta = _sensor_meta(csv_path)

    # data loaders
    zeros_tr = np.zeros_like(tr, dtype=int)
    sup_tr = DataLoader(WindowDS(tr_c, tr_m, seq_len, stride), batch_size=batch_size, shuffle=True)
    sup_va = DataLoader(WindowDS(va_c, va_m, seq_len, stride), batch_size=batch_size, shuffle=False)
    uns_tr = DataLoader(WindowDS(tr, zeros_tr, seq_len, stride), batch_size=batch_size, shuffle=True)
    uns_va = DataLoader(WindowDS(va_c, np.zeros_like(va, int), seq_len, stride), batch_size=batch_size, shuffle=False)

    pos = tr_m.sum(); neg = tr_m.size - pos
    pw = min(neg / max(pos, 1), pos_weight_cap)
    pos_weight = torch.tensor([pw], dtype=torch.float32)
    print(f"\nTrain cell anomaly rate: {100 * pos / tr_m.size:.2f}%  |  pos_weight={pw:.1f} "
          f"(capped at {pos_weight_cap})")

    registry = build_registry(N, seq_len, adj, beta)
    if models:
        registry = [m for m in registry if m["name"] in models]

    results = []
    for entry in registry:
        name, mode = entry["name"], entry["mode"]
        if mode == "sup" and train_anomaly_frac <= 0:
            print(f"\n[skip] {name}: supervised model needs train_anomaly_frac > 0")
            continue
        print(f"\n{'-'*64}\n  {name}  (mode={mode}, graph={entry['graph']})\n{'-'*64}")
        model = entry["build"]()

        if mode == "sup":
            model = train_model(model, sup_tr, sup_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience, pos_weight=pos_weight)
            val_scores = score_series(model, va_c, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_supervised(val_scores, va_m, test_scores)
        else:
            model = train_model(model, uns_tr, uns_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience)
            clean_scores = score_series(model, tr, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_unsupervised(clean_scores, test_scores, unsup_percentile)

        m = cell_metrics(pred, te_m)              # metrics on flagged mask (before removal)
        m.update(ranking_metrics(test_scores, te_m))   # threshold-free diagnostics
        out_path = os.path.join(out_dir, f"clean_{name}.csv")
        clean_df = export_clean(te_c, scaler, pred, dt_test, sensors, meta, out_path)

        removed = int(pred.sum())
        print(f"   F1={m['F1']:.4f}  Prec={m['Precision']:.4f}  Rec={m['Recall']:.4f}  "
              f"Acc={m['Accuracy']:.4f}  | removed {removed} readings -> {out_path}")
        m.update({"Model": name, "Mode": mode, "Removed": removed, "CleanFile": out_path})
        results.append(m)

    _print_table(results, te_m)
    return results


def _print_table(results, true_mask):
    inj = int((true_mask > 0).sum())
    cols = ["Model", "Mode", "AUPRC", "AUROC", "Precision", "Recall", "F1", "Removed"]
    rows = sorted([[r.get(c) for c in cols] for r in results],
                  key=lambda r: (r[2] if r[2] == r[2] else -1), reverse=True)   # by AUPRC
    print(f"\n{'='*72}\n  CELL-LEVEL DETECTION  (injected anomalous readings: {inj})\n{'='*72}")
    if _TAB:
        print(tabulate(rows, headers=cols, tablefmt="fancy_grid", floatfmt=".4f"))
    else:
        print("  ".join(f"{c:<14}" for c in cols))
        for r in rows:
            print("  ".join(f"{str(v):<14}" for v in r))
    print("\nAUPRC/AUROC : threshold-free ranking quality (AUPRC baseline = anomaly rate).")
    print("            If AUPRC >> baseline but F1 is low, it's a threshold problem,")
    print("            not a learning problem. Raise unsup_percentile to cut false flags.")
    print("Recall    = fraction of injected anomalies removed.")
    print("Precision = fraction of removed readings that were truly anomalous.")


# =============================================================================
# ENTRY POINT  (terminal:  python anomaly_cleaning.py --csv ... --adj_csv ...)
#              (jupyter  :  edit the call below and %run the file)
# =============================================================================
if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        run(
            csv_path="Data\\Lab_Data\\sead_anomaly_free_lab_data_1h.csv",
            adj_csv_path="Data\\adjacency_matrices\\lab_fne_matrix_norm.csv",
            seq_len=24, stride=4, batch_size=64, max_epochs=50,
            anomaly_frac=0.10, train_anomaly_frac=0.05, anomaly_type="all",
            unsup_percentile=99.0,
        )
    else:
        p = argparse.ArgumentParser(description="Sensor-level anomaly cleaning")
        p.add_argument("--csv", required=True)
        p.add_argument("--adj_csv", required=True)
        p.add_argument("--seq_len", type=int, default=24)
        p.add_argument("--stride", type=int, default=4)
        p.add_argument("--batch_size", type=int, default=64)
        p.add_argument("--max_epochs", type=int, default=50)
        p.add_argument("--anomaly_frac", type=float, default=0.10)
        p.add_argument("--train_anomaly_frac", type=float, default=0.05)
        p.add_argument("--anomaly_type", type=str, default="all")
        p.add_argument("--unsup_percentile", type=float, default=99.0)
        p.add_argument("--out_dir", type=str, default="cleaned_output")
        a = p.parse_args()
        run(csv_path=a.csv, adj_csv_path=a.adj_csv, seq_len=a.seq_len, stride=a.stride,
            batch_size=a.batch_size, max_epochs=a.max_epochs, anomaly_frac=a.anomaly_frac,
            train_anomaly_frac=a.train_anomaly_frac, anomaly_type=a.anomaly_type,
            unsup_percentile=a.unsup_percentile, out_dir=a.out_dir)

# # =============================================================================
# # ENTRY POINT  (terminal:  python anomaly_cleaning.py --csv ... --adj_csv ...)
# #              (jupyter  :  edit the call below and %run the file)
# # =============================================================================
# if __name__ == "__main__":
#     if "ipykernel" in sys.modules:
#         run(
#             csv_path="Data\\Lab_Data\\sead_anomaly_free_lab_data_1h.csv",
#             adj_csv_path="Data\\adjacency_matrices\\lab_fne_matrix_norm.csv",
#             seq_len=24, stride=4, batch_size=64, max_epochs=100,
#             anomaly_frac=0.10, train_anomaly_frac=0.05, anomaly_type="mixed",
#             unsup_percentile=99.0,
#         )
#     else:
#         p = argparse.ArgumentParser(description="Sensor-level anomaly cleaning")
#         p.add_argument("--csv", required=True)
#         p.add_argument("--adj_csv", required=True)
#         p.add_argument("--seq_len", type=int, default=24)
#         p.add_argument("--stride", type=int, default=4)
#         p.add_argument("--batch_size", type=int, default=64)
#         p.add_argument("--max_epochs", type=int, default=50)
#         p.add_argument("--anomaly_frac", type=float, default=0.10)
#         p.add_argument("--train_anomaly_frac", type=float, default=0.05)
#         p.add_argument("--anomaly_type", type=str, default="all")
#         p.add_argument("--unsup_percentile", type=float, default=99.0)
#         p.add_argument("--out_dir", type=str, default="cleaned_output")
#         a = p.parse_args()
#         run(csv_path=a.csv, adj_csv_path=a.adj_csv, seq_len=a.seq_len, stride=a.stride,
#             batch_size=a.batch_size, max_epochs=a.max_epochs, anomaly_frac=a.anomaly_frac,
#             train_anomaly_frac=a.train_anomaly_frac, anomaly_type=a.anomaly_type,
#             unsup_percentile=a.unsup_percentile, out_dir=a.out_dir)


  Sensor-level anomaly cleaning  |  device=cuda
  Loaded 33199 timestamps x 10 sensors
   Sensors: ['Back Door Sensor', 'Back Motion Sensor', 'Front Motion Sensor', 'Lab Door Motion Sensor', 'Lab Door Sensor', 'Lab Light Sensor', 'Left Motion Sensor', 'Right Motion Sensor', 'Sonar Sensor', 'Vibration Sensor']
   Loaded labelled adjacency matrix - aligned to 10 sensors
   Matrix not symmetric - symmetrising with max(A, A^T)
   88 directed edges | graph density: 97.8%

Injecting anomalies (val / test, and a small fraction into train):
Anomaly types available: ['all', 'collective', 'correlation_violation', 'dropout', 'graph_swap', 'mixed', 'neighbour_inconsistency', 'propagation_delay', 'relational', 'spike']
Relational anomaly types: ['collective', 'correlation_violation', 'graph_swap', 'neighbour_inconsistency', 'propagation_delay']
   Injected 497 anomalies (10.0%)
   Breakdown: {'dropout': 85, 'spike': 84, 'neighbour_inconsistency': 59, 'collective': 57, 'propagation_delay': 67, 'gra

In [8]:
"""
anomaly_cleaning.py
Sensor-level anomaly detection, removal, and forecasting-based detection.

Reuses data loading / anomaly injection / training utilities from
anomlay_pipeline.py (keep both files in the same folder, or on PYTHONPATH).

Both paradigms flag anomalies at the (timestep, sensor) cell level:
  supervised   : per-cell classifier trained on injected labels
  unsupervised : reconstruction / forecasting error vs a per-sensor threshold

Flow: load -> scale -> inject (val/test, + small frac into train for supervised)
      -> train -> score every cell -> flag -> report metrics on the flagged mask
      -> remove flagged readings -> write clean long-format CSV.

Graph usage:
  predefined adjacency : SpatioTemporalGNN, GNNForecaster
  learned graph        : GNN_LearnableGraph, GAT_LearnedGraph, GraphAnomalyVAE
  no graph             : TemporalCNN, StackedLSTM
"""
import os, sys, argparse, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score)

warnings.filterwarnings("ignore")

# --- shared pieces from the original pipeline --------------------------------
try:
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)
except ModuleNotFoundError:
    _here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, _here)
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)

try:
    from tabulate import tabulate
    _TAB = True
except ImportError:
    _TAB = False


# =============================================================================
# DATA
# =============================================================================
def _norm_adj(adj_matrix):
    """Symmetric-normalised adjacency with self-loops: D^-1/2 (A+I) D^-1/2."""
    A = (torch.tensor(np.asarray(adj_matrix), dtype=torch.float32) > 0).float()
    A = A + torch.eye(A.shape[0])
    d = A.sum(1).clamp(min=1e-6).pow(-0.5)
    return d[:, None] * A * d[None, :]


def inject(values, frac, atype, adj):
    """Inject anomalies and return (corrupted, cell_mask) where mask[t,s]=1 if
    that exact reading was altered. Mask is derived by diffing the clean array."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr, _ = inject_anomalies(values, frac, atype, "", adj)
    return corr, (corr != values).astype(int)


def inject_binary(values, frac, run_len=1):
    """Binary bit-flip anomalies: flip 0<->1 in a fraction of (timestep, sensor)
    cells, optionally for run_len consecutive steps. A flip contradicts the learned
    temporal/neighbour pattern, so it raises both reconstruction and forecast error
    (no signal-simplifying inversion). Returns (corrupted, cell_mask)."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr = values.copy()
    mask = np.zeros_like(values, dtype=int)
    T, N = values.shape
    n = max(1, int(T * N * frac / max(run_len, 1)))
    for _ in range(n):
        t = np.random.randint(0, T - run_len + 1)
        s = np.random.randint(0, N)
        sl = slice(t, t + run_len)
        corr[sl, s] = 1.0 - corr[sl, s]
        mask[sl, s] = 1
    return corr, mask


class WindowDS(Dataset):
    """Sliding windows of (seq_len, N) with matching per-cell labels."""
    def __init__(self, values, cell_labels, seq_len, stride):
        self.x, self.y = [], []
        T = len(values)
        starts = list(range(0, T - seq_len + 1, stride))
        if starts and starts[-1] != T - seq_len:
            starts.append(T - seq_len)
        for s in starts:
            self.x.append(torch.tensor(values[s:s + seq_len], dtype=torch.float32))
            self.y.append(torch.tensor(cell_labels[s:s + seq_len], dtype=torch.float32))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]


# =============================================================================
# MODELS  (all output per-cell logits (B,T,N) for sup, or expose cell scores)
# =============================================================================
class TemporalCNN(nn.Module):
    def __init__(self, N, seq_len, hidden=32, levels=4, k=3):
        super().__init__()
        chs = [N] + [hidden] * levels
        self.tcn = nn.Sequential(*[TCNBlock(chs[i], chs[i + 1], k, 2 ** i) for i in range(levels)])
        self.head = nn.Conv1d(hidden, N, 1)

    def forward(self, x):                                   # (B,T,N)
        h = self.tcn(x.permute(0, 2, 1))                    # (B,hidden,T)
        return self.head(h).permute(0, 2, 1)                # (B,T,N)


class StackedLSTM(nn.Module):
    def __init__(self, N, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(N, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, N)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out)                               # (B,T,N)


class GCNDetector(nn.Module):
    """GCN + per-node GRU per-cell classifier. Fixed graph if adj given, else learned."""
    def __init__(self, N, adj=None, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N, self.learn = N, adj is None
        if self.learn:
            self.adj_logits = nn.Parameter(torch.randn(N, N))
        else:
            self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def _A(self, device):
        if not self.learn:
            return self.A_hat
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def forward(self, x):
        B, T, N = x.shape
        feats = self.W(x.unsqueeze(-1))                     # (B,T,N,H)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device), feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)                              # (B*N,T,rnn)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GATDetector(nn.Module):
    """Learned graph: per-timestep multi-head attention over sensors (GAT on a
    fully-connected graph), then per-node GRU, then per-cell head."""
    def __init__(self, N, hidden=16, heads=2, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.proj = nn.Linear(1, hidden)
        self.attn = nn.MultiheadAttention(hidden, heads, batch_first=True, dropout=0.1)
        self.rnn = nn.GRU(hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):
        B, T, N = x.shape
        f = self.proj(x.reshape(B * T, N, 1))               # (B*T,N,hidden)
        a, _ = self.attn(f, f, f)                           # attention = learned graph
        a = torch.relu(a).reshape(B, T, N, -1)
        seq = a.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GraphAnomalyVAE(nn.Module):
    """Unsupervised graph VAE (learned adjacency). Cell score = reconstruction MSE."""
    def __init__(self, N, seq_len, gcn_hidden=16, rnn_hidden=32, latent=16, beta=1.0):
        super().__init__()
        self.N, self.seq_len, self.beta = N, seq_len, beta
        self.adj_logits = nn.Parameter(torch.randn(N, N))
        self.enc_proj = nn.Linear(1, gcn_hidden, bias=False)
        self.enc_rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.fc_mu = nn.Linear(rnn_hidden, latent)
        self.fc_lv = nn.Linear(rnn_hidden, latent)
        self.dec_fc = nn.Linear(latent, rnn_hidden)
        self.dec_rnn = nn.GRU(rnn_hidden, rnn_hidden, batch_first=True)
        self.dec_W = nn.Linear(rnn_hidden, gcn_hidden, bias=False)
        self.dec_out = nn.Linear(gcn_hidden, 1)

    def _A(self, device):
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def encode(self, x):
        B, T, N = x.shape
        f = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device),
                                    self.enc_proj(x.unsqueeze(-1))))
        seq = f.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        _, h = self.enc_rnn(seq)
        h = h.squeeze(0).reshape(B, N, -1).mean(1)
        return self.fc_mu(h), self.fc_lv(h)

    def decode(self, z):
        h = self.dec_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.dec_rnn(h)
        node = self.dec_W(out).unsqueeze(2).repeat(1, 1, self.N, 1)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(z.device), node))
        return self.dec_out(prop).squeeze(-1)               # (B,T,N)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return self.decode(z), mu, lv

    def loss(self, x):
        recon, mu, lv = self.forward(x)
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        return F.mse_loss(recon, x) + self.beta * kl

    def cell_score(self, x):
        recon, _, _ = self.forward(x)
        return (x - recon).pow(2)                           # (B,T,N)


class GNNForecaster(nn.Module):
    """Unsupervised one-step forecaster on the predefined graph. Trained on clean
    data; per-sensor score = squared next-step prediction error."""
    def __init__(self, N, adj, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):                                   # predict step T from steps 0..T-2
        B, T, N = x.shape
        xi = x[:, :-1, :]
        feats = self.W(xi.unsqueeze(-1))
        prop = torch.relu(torch.einsum("ij,btjh->btih", self.A_hat, feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
        out, _ = self.rnn(seq)
        return self.head(out[:, -1, :]).reshape(B, N)       # (B,N)

    def loss(self, x):
        return F.mse_loss(self.forward(x), x[:, -1, :])

    def step_score(self, x):
        return (self.forward(x) - x[:, -1, :]).pow(2)       # (B,N) at last step


# =============================================================================
# TRAIN / SCORE / FLAG / EVAL
# =============================================================================
def train_model(model, train_loader, val_loader, mode, device,
                lr=1e-3, epochs=50, patience=10, pos_weight=None):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device) if pos_weight is not None else None)
    stop = EarlyStopping(patience=patience, mode="f1" if mode == "sup" else "loss")

    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(device)
            opt.zero_grad()
            loss = crit(model(x), y.to(device)) if mode == "sup" else model.loss(x)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            if mode == "sup":
                ps, ys = [], []
                for x, y in val_loader:
                    ps.append(torch.sigmoid(model(x.to(device))).cpu().numpy().reshape(-1))
                    ys.append(y.numpy().reshape(-1))
                ys = np.concatenate(ys)
                # Ranking metric (AUPRC): tracks learning even when no fixed 0.5
                # cut separates the classes. F1@0.5 stays 0 on rare-positive data
                # and would stop training immediately.
                metric = average_precision_score(ys, np.concatenate(ps)) if ys.any() else 0.0
            else:
                metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in val_loader]))
        if stop.step(metric, model):
            stopped = ep + 1
            break
    else:
        stopped = epochs
    stop.restore(model)
    print(f"   trained {stopped} epoch(s) | best {'val_AUPRC' if mode == 'sup' else 'val_loss'}={stop.best:.4f}")
    return model


def score_series(model, values, seq_len, stride, mode, device):
    """Return a (T, N) anomaly-score map, averaging overlapping windows."""
    model.eval()
    T, N = values.shape
    if mode == "forecast":
        stride = 1                                # score every incoming step
    acc = np.zeros((T, N)); cnt = np.zeros((T, N))
    starts = list(range(0, T - seq_len + 1, stride))
    if starts and starts[-1] != T - seq_len:
        starts.append(T - seq_len)
    with torch.no_grad():
        for s in starts:
            x = torch.tensor(values[s:s + seq_len], dtype=torch.float32)[None].to(device)
            if mode == "sup":
                sc = torch.sigmoid(model(x))[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            elif mode == "vae":
                sc = model.cell_score(x)[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            else:                                            # forecast: last step only
                t = s + seq_len - 1
                acc[t] += model.step_score(x)[0].cpu().numpy(); cnt[t] += 1
    cnt[cnt == 0] = 1
    return acc / cnt


def flag_supervised(val_scores, val_mask, test_scores, objective="cell", min_overlap=1):
    """Tune one global probability cut on val. objective='cell' maximises cell F1;
    'event' maximises point-adjust F1 (aligns the cut with run-level scoring)."""
    thr, best = float(np.median(val_scores)), -1.0
    for t in np.linspace(val_scores.min(), val_scores.max(), 100):
        vp = (val_scores >= t).astype(int)
        if objective == "event":
            f = event_metrics(vp, val_mask, min_overlap)["PA_F1"]
        else:
            f = f1_score(val_mask.reshape(-1), vp.reshape(-1).astype(int), zero_division=0)
        if f > best:
            best, thr = f, t
    return (test_scores >= thr).astype(int), float(thr)


def flag_unsupervised(clean_scores, test_scores, method="percentile", pct=99.0, k=4.0):
    """Per-sensor threshold (higher score = more anomalous).
    percentile : pct-th percentile of clean-TRAIN scores (assumes train ~= test).
    median     : median + k*MAD of the scored set itself (adapts to its distribution).
    mean       : mean + k*std of the scored set itself.
    Robust methods need the score to rank anomalies ABOVE normals (AUROC > 0.5)."""
    if method == "percentile":
        thr = np.percentile(clean_scores, pct, axis=0)
    elif method == "median":
        med = np.median(test_scores, axis=0)
        mad = np.median(np.abs(test_scores - med), axis=0)
        thr = med + k * (mad + 1e-9)
    elif method == "mean":
        thr = test_scores.mean(0) + k * (test_scores.std(0) + 1e-9)
    else:
        raise ValueError(f"unknown unsup_threshold '{method}'")
    return (test_scores >= thr).astype(int), thr


def cell_metrics(pred_mask, true_mask):
    p = pred_mask.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    return {"Accuracy":  round(accuracy_score(t, p), 4),
            "Precision": round(precision_score(t, p, zero_division=0), 4),
            "Recall":    round(recall_score(t, p, zero_division=0), 4),
            "F1":        round(f1_score(t, p, zero_division=0), 4)}


def ranking_metrics(scores, true_mask):
    """Threshold-free quality of the score map (does it rank anomalies high?)."""
    s = scores.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    if t.sum() == 0 or t.sum() == len(t):
        return {"AUROC": float("nan"), "AUPRC": float("nan")}
    return {"AUROC": round(roc_auc_score(t, s), 4),
            "AUPRC": round(average_precision_score(t, s), 4)}


def _segments(col):
    """Contiguous runs of 1s in a 1D bool array -> list of (start, end_exclusive)."""
    segs, i, n = [], 0, len(col)
    while i < n:
        if col[i]:
            j = i
            while j < n and col[j]:
                j += 1
            segs.append((i, j)); i = j
        else:
            i += 1
    return segs


def _filter_short_runs(mask, min_len):
    """Drop predicted runs shorter than min_len per sensor (removes isolated false
    flags; useful when real anomalies are sustained)."""
    if min_len <= 1:
        return mask
    out = np.zeros_like(mask)
    for s in range(mask.shape[1]):
        for a, b in _segments(mask[:, s].astype(bool)):
            if b - a >= min_len:
                out[a:b, s] = 1
    return out


def event_metrics(pred_mask, true_mask, min_overlap=1):
    """Score whole anomalous runs instead of individual cells (per sensor).
    point-adjust (PA): if >=min_overlap cells of a true run are flagged, the WHOLE
        run is credited; P/R/F1 then computed point-wise on the adjusted mask
        (standard time-series AD protocol).
    event-overlap (EV): a true run is a hit if it's flagged at all; a predicted run
        that touches no true run is a false alarm. P/R/F1 over runs."""
    pred = pred_mask.astype(bool); true = (np.asarray(true_mask) > 0)
    T, N = true.shape
    adj = pred.copy()
    tp_ev = fn_ev = fp_ev = pred_ev = 0
    for s in range(N):
        for a, b in _segments(true[:, s]):
            if pred[a:b, s].sum() >= min_overlap:
                adj[a:b, s] = True; tp_ev += 1
            else:
                fn_ev += 1
        for a, b in _segments(pred[:, s]):
            pred_ev += 1
            if not true[a:b, s].any():
                fp_ev += 1
    p, t = adj.reshape(-1), true.reshape(-1)
    ev_p = (pred_ev - fp_ev) / pred_ev if pred_ev else 0.0
    ev_r = tp_ev / (tp_ev + fn_ev) if (tp_ev + fn_ev) else 0.0
    ev_f1 = 2 * ev_p * ev_r / (ev_p + ev_r) if (ev_p + ev_r) else 0.0
    return {"PA_Precision": round(precision_score(t, p, zero_division=0), 4),
            "PA_Recall":    round(recall_score(t, p, zero_division=0), 4),
            "PA_F1":        round(f1_score(t, p, zero_division=0), 4),
            "EV_Recall":    round(ev_r, 4),
            "EV_F1":        round(ev_f1, 4),
            "Events":       tp_ev + fn_ev}


# =============================================================================
# CLEAN OUTPUT
# =============================================================================
def _sensor_meta(csv_path):
    try:
        df = pd.read_csv(csv_path)
        cols = [c for c in ("thing_name", "thing_ip") if c in df.columns]
        if not cols:
            return {}
        g = df.drop_duplicates("sensor_name").set_index("sensor_name")[cols]
        return {s: row.to_dict() for s, row in g.iterrows()}
    except Exception:
        return {}


def export_clean(corrupted, scaler, pred_mask, dt_index, sensors, meta, out_path):
    """Drop flagged (datetime, sensor) readings; write the rest as long-format CSV."""
    inv = scaler.inverse_transform(corrupted)
    rows = []
    for ti, dt in enumerate(dt_index):
        dt = pd.Timestamp(dt)
        for sj, sname in enumerate(sensors):
            if pred_mask[ti, sj]:
                continue
            m = meta.get(sname, {})
            rows.append({
                "datetime": dt, "date": dt.strftime("%Y-%m-%d"),
                "time": dt.strftime("%H:%M"), "seconds": dt.timestamp(),
                "state": float(inv[ti, sj]), "sensor_name": sname,
                "thing_name": m.get("thing_name"), "thing_ip": m.get("thing_ip"),
            })
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    return df


# =============================================================================
# REGISTRY & ORCHESTRATION
# =============================================================================
def build_registry(N, seq_len, adj, beta):
    return [
        {"name": "TemporalCNN",        "mode": "sup",      "graph": "none",
         "build": lambda: TemporalCNN(N, seq_len)},
        {"name": "StackedLSTM",        "mode": "sup",      "graph": "none",
         "build": lambda: StackedLSTM(N)},
        {"name": "SpatioTemporalGNN",  "mode": "sup",      "graph": "predefined",
         "build": lambda: GCNDetector(N, adj=adj)},
        {"name": "GNN_LearnableGraph", "mode": "sup",      "graph": "learned",
         "build": lambda: GCNDetector(N, adj=None)},
        {"name": "GAT_LearnedGraph",   "mode": "sup",      "graph": "learned",
         "build": lambda: GATDetector(N)},
        {"name": "GraphAnomalyVAE",    "mode": "vae",      "graph": "learned",
         "build": lambda: GraphAnomalyVAE(N, seq_len, beta=beta)},
        {"name": "GNNForecaster",      "mode": "forecast", "graph": "predefined",
         "build": lambda: GNNForecaster(N, adj=adj)},
    ]


def run(csv_path, adj_csv_path,
        seq_len=24, stride=4, batch_size=64, max_epochs=50, lr=1e-3, patience=15,
        anomaly_frac=0.10, train_anomaly_frac=0.05, anomaly_type="all",
        binary_anomaly=False, run_len=1,
        unsup_percentile=99.0, unsup_threshold="percentile", robust_k=4.0,
        event_min_overlap=1, min_event_len=1, sup_objective="cell",
        pos_weight_cap=10.0, train_frac=0.70, val_frac=0.15,
        out_dir="cleaned_output", models=None, beta=1.0, seed=42):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*64}\n  Sensor-level anomaly cleaning  |  device={device}\n{'='*64}")
    wide = load_and_pivot(csv_path)
    sensors = list(wide.columns); N = len(sensors)
    raw = wide.values.astype(np.float32); T = len(raw)
    n_tr, n_va = int(T * train_frac), int(T * val_frac)

    scaler = MinMaxScaler()
    tr = scaler.fit_transform(raw[:n_tr])
    va = scaler.transform(raw[n_tr:n_tr + n_va])
    te = scaler.transform(raw[n_tr + n_va:])
    dt_test = wide.index[n_tr + n_va:]

    edge_index, adj = load_edge_index_from_csv(adj_csv_path, sensors)

    print("\nInjecting anomalies (val / test, and a small fraction into train):")
    _inj = (lambda v, f: inject_binary(v, f, run_len)) if binary_anomaly \
        else (lambda v, f: inject(v, f, anomaly_type, adj))
    va_c, va_m = _inj(va, anomaly_frac)
    te_c, te_m = _inj(te, anomaly_frac)
    tr_c, tr_m = _inj(tr, train_anomaly_frac)
    meta = _sensor_meta(csv_path)

    # data loaders
    zeros_tr = np.zeros_like(tr, dtype=int)
    sup_tr = DataLoader(WindowDS(tr_c, tr_m, seq_len, stride), batch_size=batch_size, shuffle=True)
    sup_va = DataLoader(WindowDS(va_c, va_m, seq_len, stride), batch_size=batch_size, shuffle=False)
    uns_tr = DataLoader(WindowDS(tr, zeros_tr, seq_len, stride), batch_size=batch_size, shuffle=True)
    uns_va = DataLoader(WindowDS(va_c, np.zeros_like(va, int), seq_len, stride), batch_size=batch_size, shuffle=False)

    pos = tr_m.sum(); neg = tr_m.size - pos
    pw = min(neg / max(pos, 1), pos_weight_cap)
    pos_weight = torch.tensor([pw], dtype=torch.float32)
    print(f"\nTrain cell anomaly rate: {100 * pos / tr_m.size:.2f}%  |  pos_weight={pw:.1f} "
          f"(capped at {pos_weight_cap})")

    registry = build_registry(N, seq_len, adj, beta)
    if models:
        registry = [m for m in registry if m["name"] in models]

    results = []
    for entry in registry:
        name, mode = entry["name"], entry["mode"]
        if mode == "sup" and train_anomaly_frac <= 0:
            print(f"\n[skip] {name}: supervised model needs train_anomaly_frac > 0")
            continue
        print(f"\n{'-'*64}\n  {name}  (mode={mode}, graph={entry['graph']})\n{'-'*64}")
        model = entry["build"]()

        if mode == "sup":
            model = train_model(model, sup_tr, sup_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience, pos_weight=pos_weight)
            val_scores = score_series(model, va_c, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_supervised(val_scores, va_m, test_scores,
                                        objective=sup_objective, min_overlap=event_min_overlap)
        else:
            model = train_model(model, uns_tr, uns_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience)
            clean_scores = score_series(model, tr, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_unsupervised(clean_scores, test_scores,
                                          method=unsup_threshold, pct=unsup_percentile, k=robust_k)

        pred = _filter_short_runs(pred, min_event_len)
        m = cell_metrics(pred, te_m)              # metrics on flagged mask (before removal)
        m.update(ranking_metrics(test_scores, te_m))   # threshold-free diagnostics
        m.update(event_metrics(pred, te_m, event_min_overlap))   # run/segment-level
        out_path = os.path.join(out_dir, f"clean_{name}.csv")
        clean_df = export_clean(te_c, scaler, pred, dt_test, sensors, meta, out_path)

        removed = int(pred.sum())
        print(f"   F1={m['F1']:.4f}  Prec={m['Precision']:.4f}  Rec={m['Recall']:.4f}  "
              f"Acc={m['Accuracy']:.4f}  | removed {removed} readings -> {out_path}")
        m.update({"Model": name, "Mode": mode, "Removed": removed, "CleanFile": out_path})
        results.append(m)

    _print_table(results, te_m)
    return results


def _print_table(results, true_mask):
    inj = int((true_mask > 0).sum())
    cols = ["Model", "Mode", "AUPRC", "Cell_F1", "PA_F1", "EV_Recall", "EV_F1", "Removed"]
    key = {"Cell_F1": "F1"}                       # map display name -> result key
    rows = [[r.get(key.get(c, c)) for c in cols] for r in results]
    rows.sort(key=lambda r: (r[4] if r[4] == r[4] else -1), reverse=True)   # by PA_F1
    nev = results[0].get("Events", "?") if results else "?"
    print(f"\n{'='*78}\n  DETECTION  (injected cells: {inj}, anomalous runs/events: {nev})\n{'='*78}")
    if _TAB:
        print(tabulate(rows, headers=cols, tablefmt="fancy_grid", floatfmt=".4f"))
    else:
        print("  ".join(f"{c:<12}" for c in cols))
        for r in rows:
            print("  ".join(f"{str(v):<12}" for v in r))
    print("\nCell_F1  : exact per-(timestep,sensor) match (strictest).")
    print("PA_F1    : point-adjust - a run is credited if any of its cells is flagged.")
    print("EV_Recall/EV_F1 : per-run - did we catch each anomalous segment.")
    print("Event scoring is meaningful for sustained anomalies (run_len > 1).")


# =============================================================================
# ENTRY POINT  (terminal:  python anomaly_cleaning.py --csv ... --adj_csv ...)
#              (jupyter  :  edit the call below and %run the file)
# =============================================================================
if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        run(
            csv_path="Data\\Lab_Data\\sead_anomaly_free_lab_data_1h.csv",
            adj_csv_path="Data\\adjacency_matrices\\lab_fne_matrix_norm.csv",
            seq_len=24, stride=4, batch_size=64, max_epochs=100,
            anomaly_frac=0.10, train_anomaly_frac=0.05, anomaly_type="all",
            unsup_percentile=99.0, unsup_threshold="mean", binary_anomaly=True, run_len=5, sup_objective="event"
        )
    else:
        p = argparse.ArgumentParser(description="Sensor-level anomaly cleaning")
        p.add_argument("--csv", required=True)
        p.add_argument("--adj_csv", required=True)
        p.add_argument("--seq_len", type=int, default=24)
        p.add_argument("--stride", type=int, default=4)
        p.add_argument("--batch_size", type=int, default=64)
        p.add_argument("--max_epochs", type=int, default=50)
        p.add_argument("--anomaly_frac", type=float, default=0.10)
        p.add_argument("--train_anomaly_frac", type=float, default=0.05)
        p.add_argument("--anomaly_type", type=str, default="all")
        p.add_argument("--binary_anomaly", action="store_true",
                       help="use binary bit-flip injection instead of the pipeline injectors")
        p.add_argument("--run_len", type=int, default=1, help="bit-flip run length (binary)")
        p.add_argument("--unsup_percentile", type=float, default=99.0)
        p.add_argument("--unsup_threshold", type=str, default="percentile",
                       choices=["percentile", "median", "mean"])
        p.add_argument("--robust_k", type=float, default=4.0)
        p.add_argument("--out_dir", type=str, default="cleaned_output")
        a = p.parse_args()
        run(csv_path=a.csv, adj_csv_path=a.adj_csv, seq_len=a.seq_len, stride=a.stride,
            batch_size=a.batch_size, max_epochs=a.max_epochs, anomaly_frac=a.anomaly_frac,
            train_anomaly_frac=a.train_anomaly_frac, anomaly_type=a.anomaly_type,
            binary_anomaly=a.binary_anomaly, run_len=a.run_len,
            unsup_percentile=a.unsup_percentile, unsup_threshold=a.unsup_threshold,
            robust_k=a.robust_k, out_dir=a.out_dir)


  Sensor-level anomaly cleaning  |  device=cuda
  Loaded 33199 timestamps x 10 sensors
   Sensors: ['Back Door Sensor', 'Back Motion Sensor', 'Front Motion Sensor', 'Lab Door Motion Sensor', 'Lab Door Sensor', 'Lab Light Sensor', 'Left Motion Sensor', 'Right Motion Sensor', 'Sonar Sensor', 'Vibration Sensor']
   Loaded labelled adjacency matrix - aligned to 10 sensors
   Matrix not symmetric - symmetrising with max(A, A^T)
   88 directed edges | graph density: 97.8%

Injecting anomalies (val / test, and a small fraction into train):

Train cell anomaly rate: 4.88%  |  pos_weight=10.0 (capped at 10.0)

----------------------------------------------------------------
  TemporalCNN  (mode=sup, graph=none)
----------------------------------------------------------------
   trained 70 epoch(s) | best val_AUPRC=0.6643
   F1=0.4973  Prec=0.5389  Rec=0.4617  Acc=0.9106  | removed 4088 readings -> cleaned_output\clean_TemporalCNN.csv

-----------------------------------------------------------

In [9]:
import numpy as np
from anomlay_pipeline import load_and_pivot, load_edge_index_from_csv
w = load_and_pivot("Data\\Lab_Data\\sead_anomaly_free_lab_data_1h.csv"); X = w.values
_, adj = load_edge_index_from_csv("Data\\adjacency_matrices\\lab_fne_matrix_norm.csv", list(w.columns))
A = np.asarray(adj) > 0
agree = [ (X[:,i]==X[:,j]).mean() for i in range(X.shape[1]) for j in np.where(A[i])[0] ]
print("connected-pair agreement: mean", round(np.mean(agree),3), " min", round(np.min(agree),3))


  Loaded 33199 timestamps x 10 sensors
   Sensors: ['Back Door Sensor', 'Back Motion Sensor', 'Front Motion Sensor', 'Lab Door Motion Sensor', 'Lab Door Sensor', 'Lab Light Sensor', 'Left Motion Sensor', 'Right Motion Sensor', 'Sonar Sensor', 'Vibration Sensor']
   Loaded labelled adjacency matrix - aligned to 10 sensors
   Matrix not symmetric - symmetrising with max(A, A^T)
   88 directed edges | graph density: 97.8%
connected-pair agreement: mean 0.43  min 0.012


In [22]:
"""
anomaly_cleaning.py
Sensor-level anomaly detection, removal, and forecasting-based detection.

Reuses data loading / anomaly injection / training utilities from
anomlay_pipeline.py (keep both files in the same folder, or on PYTHONPATH).

Both paradigms flag anomalies at the (timestep, sensor) cell level:
  supervised   : per-cell classifier trained on injected labels
  unsupervised : reconstruction / forecasting error vs a per-sensor threshold

Flow: load -> scale -> inject (val/test, + small frac into train for supervised)
      -> train -> score every cell -> flag -> report metrics on the flagged mask
      -> remove flagged readings -> write clean long-format CSV.

Graph usage:
  predefined adjacency : SpatioTemporalGNN, GNNForecaster
  learned graph        : GNN_LearnableGraph, GAT_LearnedGraph, GraphAnomalyVAE
  no graph             : TemporalCNN, StackedLSTM
"""
import os, sys, argparse, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score)

warnings.filterwarnings("ignore")

# --- shared pieces from the original pipeline --------------------------------
try:
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)
except ModuleNotFoundError:
    _here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, _here)
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)

try:
    from tabulate import tabulate
    _TAB = True
except ImportError:
    _TAB = False


# =============================================================================
# DATA
# =============================================================================
def _norm_adj(adj_matrix):
    """Symmetric-normalised adjacency with self-loops: D^-1/2 (A+I) D^-1/2."""
    A = (torch.tensor(np.asarray(adj_matrix), dtype=torch.float32) > 0).float()
    A = A + torch.eye(A.shape[0])
    d = A.sum(1).clamp(min=1e-6).pow(-0.5)
    return d[:, None] * A * d[None, :]


def adjacency_from_groups(groups, sensors):
    """Build a symmetric 0/1 adjacency (N x N) from a {group_name: [sensor,...]} dict.
    Every pair of sensors that share a group is connected (each group is a clique).
    Names not present in `sensors` are ignored; sensors in no group stay isolated."""
    idx = {s: i for i, s in enumerate(sensors)}
    N = len(sensors)
    A = np.zeros((N, N), dtype=np.float32)
    missing = set()
    for members in groups.values():
        ids = [idx[s] for s in members if s in idx]
        missing |= {s for s in members if s not in idx}
        for a in ids:
            for b in ids:
                if a != b:
                    A[a, b] = 1.0
    iso = [sensors[i] for i in range(N) if A[i].sum() == 0]
    print(f"   Built graph from {len(groups)} groups: {int(A.sum())//2} undirected edges "
          f"| density {A.sum() / (N * (N - 1)):.1%}")
    if missing:
        print(f"   [groups] ignored {len(missing)} name(s) not in data: {sorted(missing)}")
    if iso:
        print(f"   [groups] isolated sensors (in no group): {iso}")
    return A


def inject(values, frac, atype, adj):
    """Inject anomalies and return (corrupted, cell_mask) where mask[t,s]=1 if
    that exact reading was altered. Mask is derived by diffing the clean array."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr, _ = inject_anomalies(values, frac, atype, "", adj)
    return corr, (corr != values).astype(int)


def inject_binary(values, frac, run_len=1, mode="flip", adj=None, ood_value=1.5, rate=0.7):
    """Binary anomaly injection. Each anomaly spans run_len consecutive steps on one
    sensor. mask marks only cells whose value actually changed. Modes:
      flip     : invert the bit(s) (in-distribution; needs predictable data to detect).
      ood      : set to an out-of-range value (sensor-fault / garbage reading; the
                 binary analogue of a spike - separable in feature space, easiest).
      stuck    : freeze at a constant 0 or 1 over the run (stuck-at fault).
      neighbor : force disagreement with a graph-neighbour (spatial anomaly; detectable
                 from cross-sensor structure even when the series isn't predictable).
      rate     : flip a 'rate' fraction of the run's steps (a burst / activation-rate
                 shift; a sustained statistical anomaly, more robust than one flip)."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr = values.copy()
    mask = np.zeros_like(values, dtype=int)
    T, N = values.shape
    n = max(1, int(T * N * frac / max(run_len, 1)))
    nbrs = None
    if mode == "neighbor" and adj is not None:
        A = np.asarray(adj) > 0
        nbrs = [np.where(A[s])[0] for s in range(N)]
    for _ in range(n):
        t = np.random.randint(0, T - run_len + 1)
        s = np.random.randint(0, N)
        sl = slice(t, t + run_len)
        if mode == "ood":
            corr[sl, s] = ood_value
        elif mode == "stuck":
            corr[sl, s] = float(np.random.randint(0, 2))
        elif mode == "neighbor" and nbrs is not None and len(nbrs[s]):
            corr[sl, s] = 1.0 - values[sl, np.random.choice(nbrs[s])]
        elif mode == "rate":
            pick = np.where(np.random.rand(run_len) < rate)[0] + t
            corr[pick, s] = 1.0 - corr[pick, s]
        else:                                            # flip
            corr[sl, s] = 1.0 - corr[sl, s]
        mask[sl, s] = (corr[sl, s] != values[sl, s]).astype(int)
    return corr, mask


class WindowDS(Dataset):
    """Sliding windows of (seq_len, N) with matching per-cell labels."""
    def __init__(self, values, cell_labels, seq_len, stride):
        self.x, self.y = [], []
        T = len(values)
        starts = list(range(0, T - seq_len + 1, stride))
        if starts and starts[-1] != T - seq_len:
            starts.append(T - seq_len)
        for s in starts:
            self.x.append(torch.tensor(values[s:s + seq_len], dtype=torch.float32))
            self.y.append(torch.tensor(cell_labels[s:s + seq_len], dtype=torch.float32))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]


# =============================================================================
# MODELS  (all output per-cell logits (B,T,N) for sup, or expose cell scores)
# =============================================================================
class TemporalCNN(nn.Module):
    def __init__(self, N, seq_len, hidden=32, levels=4, k=3):
        super().__init__()
        chs = [N] + [hidden] * levels
        self.tcn = nn.Sequential(*[TCNBlock(chs[i], chs[i + 1], k, 2 ** i) for i in range(levels)])
        self.head = nn.Conv1d(hidden, N, 1)

    def forward(self, x):                                   # (B,T,N)
        h = self.tcn(x.permute(0, 2, 1))                    # (B,hidden,T)
        return self.head(h).permute(0, 2, 1)                # (B,T,N)


class StackedLSTM(nn.Module):
    def __init__(self, N, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(N, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, N)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out)                               # (B,T,N)


class GCNDetector(nn.Module):
    """GCN + per-node GRU per-cell classifier. Fixed graph if adj given, else learned."""
    def __init__(self, N, adj=None, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N, self.learn = N, adj is None
        if self.learn:
            self.adj_logits = nn.Parameter(torch.randn(N, N))
        else:
            self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def _A(self, device):
        if not self.learn:
            return self.A_hat
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def forward(self, x):
        B, T, N = x.shape
        feats = self.W(x.unsqueeze(-1))                     # (B,T,N,H)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device), feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)                              # (B*N,T,rnn)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GATDetector(nn.Module):
    """Learned graph: per-timestep multi-head attention over sensors (GAT on a
    fully-connected graph), then per-node GRU, then per-cell head."""
    def __init__(self, N, hidden=16, heads=2, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.proj = nn.Linear(1, hidden)
        self.attn = nn.MultiheadAttention(hidden, heads, batch_first=True, dropout=0.1)
        self.rnn = nn.GRU(hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):
        B, T, N = x.shape
        f = self.proj(x.reshape(B * T, N, 1))               # (B*T,N,hidden)
        a, _ = self.attn(f, f, f)                           # attention = learned graph
        a = torch.relu(a).reshape(B, T, N, -1)
        seq = a.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GraphAnomalyVAE(nn.Module):
    """Unsupervised graph VAE (learned adjacency). Cell score = reconstruction MSE."""
    def __init__(self, N, seq_len, gcn_hidden=16, rnn_hidden=32, latent=16, beta=1.0):
        super().__init__()
        self.N, self.seq_len, self.beta = N, seq_len, beta
        self.adj_logits = nn.Parameter(torch.randn(N, N))
        self.enc_proj = nn.Linear(1, gcn_hidden, bias=False)
        self.enc_rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.fc_mu = nn.Linear(rnn_hidden, latent)
        self.fc_lv = nn.Linear(rnn_hidden, latent)
        self.dec_fc = nn.Linear(latent, rnn_hidden)
        self.dec_rnn = nn.GRU(rnn_hidden, rnn_hidden, batch_first=True)
        self.dec_W = nn.Linear(rnn_hidden, gcn_hidden, bias=False)
        self.dec_out = nn.Linear(gcn_hidden, 1)

    def _A(self, device):
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def encode(self, x):
        B, T, N = x.shape
        f = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device),
                                    self.enc_proj(x.unsqueeze(-1))))
        seq = f.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        _, h = self.enc_rnn(seq)
        h = h.squeeze(0).reshape(B, N, -1).mean(1)
        return self.fc_mu(h), self.fc_lv(h)

    def decode(self, z):
        h = self.dec_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.dec_rnn(h)
        node = self.dec_W(out).unsqueeze(2).repeat(1, 1, self.N, 1)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(z.device), node))
        return self.dec_out(prop).squeeze(-1)               # (B,T,N)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return self.decode(z), mu, lv

    def loss(self, x):
        recon, mu, lv = self.forward(x)
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        return F.mse_loss(recon, x) + self.beta * kl

    def cell_score(self, x):
        recon, _, _ = self.forward(x)
        return (x - recon).pow(2)                           # (B,T,N)


class GNNForecaster(nn.Module):
    """Unsupervised one-step forecaster on the predefined graph. Trained on clean
    data; per-sensor score = squared next-step prediction error."""
    def __init__(self, N, adj, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):                                   # predict step T from steps 0..T-2
        B, T, N = x.shape
        xi = x[:, :-1, :]
        feats = self.W(xi.unsqueeze(-1))
        prop = torch.relu(torch.einsum("ij,btjh->btih", self.A_hat, feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
        out, _ = self.rnn(seq)
        return self.head(out[:, -1, :]).reshape(B, N)       # (B,N)

    def loss(self, x):
        return F.mse_loss(self.forward(x), x[:, -1, :])

    def step_score(self, x):
        return (self.forward(x) - x[:, -1, :]).pow(2)       # (B,N) at last step


# =============================================================================
# TRAIN / SCORE / FLAG / EVAL
# =============================================================================
def train_model(model, train_loader, val_loader, mode, device,
                lr=1e-3, epochs=50, patience=10, pos_weight=None):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device) if pos_weight is not None else None)
    stop = EarlyStopping(patience=patience, mode="f1" if mode == "sup" else "loss")

    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(device)
            opt.zero_grad()
            loss = crit(model(x), y.to(device)) if mode == "sup" else model.loss(x)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            if mode == "sup":
                ps, ys = [], []
                for x, y in val_loader:
                    ps.append(torch.sigmoid(model(x.to(device))).cpu().numpy().reshape(-1))
                    ys.append(y.numpy().reshape(-1))
                ys = np.concatenate(ys)
                # Ranking metric (AUPRC): tracks learning even when no fixed 0.5
                # cut separates the classes. F1@0.5 stays 0 on rare-positive data
                # and would stop training immediately.
                metric = average_precision_score(ys, np.concatenate(ps)) if ys.any() else 0.0
            else:
                metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in val_loader]))
        if stop.step(metric, model):
            stopped = ep + 1
            break
    else:
        stopped = epochs
    stop.restore(model)
    if mode == "sup":
        print(f"   trained {stopped} epoch(s) | best {'val_AUPRC' if mode == 'sup' else 'val_loss'}={stop.best:.4f}")
    else:
        model.eval()
        with torch.no_grad():
            train_metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in train_loader]))
        print(f"   trained {stopped} epoch(s) | train_loss={train_metric:.4f} | best {'val_AUPRC' if mode == 'sup' else 'val_loss'}={stop.best:.4f}")
    return model


def score_series(model, values, seq_len, stride, mode, device):
    """Return a (T, N) anomaly-score map, averaging overlapping windows."""
    model.eval()
    T, N = values.shape
    if mode == "forecast":
        stride = 1                                # score every incoming step
    acc = np.zeros((T, N)); cnt = np.zeros((T, N))
    starts = list(range(0, T - seq_len + 1, stride))
    if starts and starts[-1] != T - seq_len:
        starts.append(T - seq_len)
    with torch.no_grad():
        for s in starts:
            x = torch.tensor(values[s:s + seq_len], dtype=torch.float32)[None].to(device)
            if mode == "sup":
                sc = torch.sigmoid(model(x))[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            elif mode == "vae":
                sc = model.cell_score(x)[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            else:                                            # forecast: last step only
                t = s + seq_len - 1
                acc[t] += model.step_score(x)[0].cpu().numpy(); cnt[t] += 1
    cnt[cnt == 0] = 1
    return acc / cnt


def flag_supervised(val_scores, val_mask, test_scores, objective="cell", min_overlap=1):
    """Tune one global probability cut on val. objective='cell' maximises cell F1;
    'event' maximises point-adjust F1 (aligns the cut with run-level scoring)."""
    thr, best = float(np.median(val_scores)), -1.0
    for t in np.linspace(val_scores.min(), val_scores.max(), 100):
        vp = (val_scores >= t).astype(int)
        if objective == "event":
            f = event_metrics(vp, val_mask, min_overlap)["PA_F1"]
        else:
            f = f1_score(val_mask.reshape(-1), vp.reshape(-1).astype(int), zero_division=0)
        if f > best:
            best, thr = f, t
    return (test_scores >= thr).astype(int), float(thr)


def flag_unsupervised(clean_scores, test_scores, method="percentile", pct=99.0, k=4.0):
    """Per-sensor threshold (higher score = more anomalous).
    percentile : pct-th percentile of clean-TRAIN scores (assumes train ~= test).
    median     : median + k*MAD of the scored set itself (adapts to its distribution).
    mean       : mean + k*std of the scored set itself.
    Robust methods need the score to rank anomalies ABOVE normals (AUROC > 0.5)."""
    if method == "percentile":
        thr = np.percentile(clean_scores, pct, axis=0)
    elif method == "median":
        med = np.median(test_scores, axis=0)
        mad = np.median(np.abs(test_scores - med), axis=0)
        thr = med + k * (mad + 1e-9)
    elif method == "mean":
        thr = test_scores.mean(0) + k * (test_scores.std(0) + 1e-9)
    else:
        raise ValueError(f"unknown unsup_threshold '{method}'")
    return (test_scores >= thr).astype(int), thr


def cell_metrics(pred_mask, true_mask):
    p = pred_mask.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    return {"Accuracy":  round(accuracy_score(t, p), 4),
            "Precision": round(precision_score(t, p, zero_division=0), 4),
            "Recall":    round(recall_score(t, p, zero_division=0), 4),
            "F1":        round(f1_score(t, p, zero_division=0), 4)}


def ranking_metrics(scores, true_mask):
    """Threshold-free quality of the score map (does it rank anomalies high?)."""
    s = scores.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    if t.sum() == 0 or t.sum() == len(t):
        return {"AUROC": float("nan"), "AUPRC": float("nan")}
    return {"AUROC": round(roc_auc_score(t, s), 4),
            "AUPRC": round(average_precision_score(t, s), 4)}


def _segments(col):
    """Contiguous runs of 1s in a 1D bool array -> list of (start, end_exclusive)."""
    segs, i, n = [], 0, len(col)
    while i < n:
        if col[i]:
            j = i
            while j < n and col[j]:
                j += 1
            segs.append((i, j)); i = j
        else:
            i += 1
    return segs


def _filter_short_runs(mask, min_len):
    """Drop predicted runs shorter than min_len per sensor (removes isolated false
    flags; useful when real anomalies are sustained)."""
    if min_len <= 1:
        return mask
    out = np.zeros_like(mask)
    for s in range(mask.shape[1]):
        for a, b in _segments(mask[:, s].astype(bool)):
            if b - a >= min_len:
                out[a:b, s] = 1
    return out


def event_metrics(pred_mask, true_mask, min_overlap=1):
    """Score whole anomalous runs instead of individual cells (per sensor).
    point-adjust (PA): if >=min_overlap cells of a true run are flagged, the WHOLE
        run is credited; P/R/F1 then computed point-wise on the adjusted mask
        (standard time-series AD protocol).
    event-overlap (EV): a true run is a hit if it's flagged at all; a predicted run
        that touches no true run is a false alarm. P/R/F1 over runs."""
    pred = pred_mask.astype(bool); true = (np.asarray(true_mask) > 0)
    T, N = true.shape
    adj = pred.copy()
    tp_ev = fn_ev = fp_ev = pred_ev = 0
    for s in range(N):
        for a, b in _segments(true[:, s]):
            if pred[a:b, s].sum() >= min_overlap:
                adj[a:b, s] = True; tp_ev += 1
            else:
                fn_ev += 1
        for a, b in _segments(pred[:, s]):
            pred_ev += 1
            if not true[a:b, s].any():
                fp_ev += 1
    p, t = adj.reshape(-1), true.reshape(-1)
    ev_p = (pred_ev - fp_ev) / pred_ev if pred_ev else 0.0
    ev_r = tp_ev / (tp_ev + fn_ev) if (tp_ev + fn_ev) else 0.0
    ev_f1 = 2 * ev_p * ev_r / (ev_p + ev_r) if (ev_p + ev_r) else 0.0
    return {"PA_Precision": round(precision_score(t, p, zero_division=0), 4),
            "PA_Recall":    round(recall_score(t, p, zero_division=0), 4),
            "PA_F1":        round(f1_score(t, p, zero_division=0), 4),
            "EV_Recall":    round(ev_r, 4),
            "EV_F1":        round(ev_f1, 4),
            "Events":       tp_ev + fn_ev}


# =============================================================================
# CLEAN OUTPUT
# =============================================================================
def _sensor_meta(csv_path):
    try:
        df = pd.read_csv(csv_path)
        cols = [c for c in ("thing_name", "thing_ip") if c in df.columns]
        if not cols:
            return {}
        g = df.drop_duplicates("sensor_name").set_index("sensor_name")[cols]
        return {s: row.to_dict() for s, row in g.iterrows()}
    except Exception:
        return {}


def export_clean(corrupted, scaler, pred_mask, dt_index, sensors, meta, out_path):
    """Drop flagged (datetime, sensor) readings; write the rest as long-format CSV."""
    inv = scaler.inverse_transform(corrupted)
    rows = []
    for ti, dt in enumerate(dt_index):
        dt = pd.Timestamp(dt)
        for sj, sname in enumerate(sensors):
            if pred_mask[ti, sj]:
                continue
            m = meta.get(sname, {})
            rows.append({
                "datetime": dt, "date": dt.strftime("%Y-%m-%d"),
                "time": dt.strftime("%H:%M"), "seconds": dt.timestamp(),
                "state": float(inv[ti, sj]), "sensor_name": sname,
                "thing_name": m.get("thing_name"), "thing_ip": m.get("thing_ip"),
            })
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    return df


# =============================================================================
# REGISTRY & ORCHESTRATION
# =============================================================================
def build_registry(N, seq_len, adj, beta):
    return [
        {"name": "TemporalCNN",        "mode": "sup",      "graph": "none",
         "build": lambda: TemporalCNN(N, seq_len)},
        {"name": "StackedLSTM",        "mode": "sup",      "graph": "none",
         "build": lambda: StackedLSTM(N)},
        {"name": "SpatioTemporalGNN",  "mode": "sup",      "graph": "predefined",
         "build": lambda: GCNDetector(N, adj=adj)},
        {"name": "GNN_LearnableGraph", "mode": "sup",      "graph": "learned",
         "build": lambda: GCNDetector(N, adj=None)},
        {"name": "GAT_LearnedGraph",   "mode": "sup",      "graph": "learned",
         "build": lambda: GATDetector(N)},
        {"name": "GraphAnomalyVAE",    "mode": "vae",      "graph": "learned",
         "build": lambda: GraphAnomalyVAE(N, seq_len, beta=beta)},
        {"name": "GNNForecaster",      "mode": "forecast", "graph": "predefined",
         "build": lambda: GNNForecaster(N, adj=adj)},
    ]


def run(csv_path, adj_csv_path=None, groups=None,
        seq_len=24, stride=4, batch_size=64, max_epochs=50, lr=1e-3, patience=15,
        anomaly_frac=0.10, train_anomaly_frac=0.05, anomaly_type="all",
        binary_anomaly=False, run_len=1, binary_mode="flip",
        unsup_percentile=99.0, unsup_threshold="percentile", robust_k=4.0,
        event_min_overlap=1, min_event_len=1, sup_objective="cell",
        pos_weight_cap=10.0, train_frac=0.70, val_frac=0.15,
        out_dir="cleaned_output", models=None, beta=1.0, seed=42):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*64}\n  Sensor-level anomaly cleaning  |  device={device}\n{'='*64}")
    wide = load_and_pivot(csv_path)
    sensors = list(wide.columns); N = len(sensors)
    raw = wide.values.astype(np.float32); T = len(raw)
    n_tr, n_va = int(T * train_frac), int(T * val_frac)

    scaler = MinMaxScaler()
    tr = scaler.fit_transform(raw[:n_tr])
    va = scaler.transform(raw[n_tr:n_tr + n_va])
    te = scaler.transform(raw[n_tr + n_va:])
    dt_test = wide.index[n_tr + n_va:]

    if groups is not None:
        adj = adjacency_from_groups(groups, sensors)
    elif adj_csv_path is not None:
        _, adj = load_edge_index_from_csv(adj_csv_path, sensors)
    else:
        raise ValueError("provide either `groups` (dict) or `adj_csv_path`")

    print("\nInjecting anomalies (val / test, and a small fraction into train):")
    _inj = (lambda v, f: inject_binary(v, f, run_len, binary_mode, adj)) if binary_anomaly \
        else (lambda v, f: inject(v, f, anomaly_type, adj))
    va_c, va_m = _inj(va, anomaly_frac)
    te_c, te_m = _inj(te, anomaly_frac)
    tr_c, tr_m = _inj(tr, train_anomaly_frac)
    meta = _sensor_meta(csv_path)

    # data loaders
    zeros_tr = np.zeros_like(tr, dtype=int)
    sup_tr = DataLoader(WindowDS(tr_c, tr_m, seq_len, stride), batch_size=batch_size, shuffle=True)
    sup_va = DataLoader(WindowDS(va_c, va_m, seq_len, stride), batch_size=batch_size, shuffle=False)
    uns_tr = DataLoader(WindowDS(tr, zeros_tr, seq_len, stride), batch_size=batch_size, shuffle=True)
    uns_va = DataLoader(WindowDS(va_c, np.zeros_like(va, int), seq_len, stride), batch_size=batch_size, shuffle=False)

    pos = tr_m.sum(); neg = tr_m.size - pos
    pw = min(neg / max(pos, 1), pos_weight_cap)
    pos_weight = torch.tensor([pw], dtype=torch.float32)
    print(f"\nTrain cell anomaly rate: {100 * pos / tr_m.size:.2f}%  |  pos_weight={pw:.1f} "
          f"(capped at {pos_weight_cap})")

    registry = build_registry(N, seq_len, adj, beta)
    if models:
        registry = [m for m in registry if m["name"] in models]

    results = []
    for entry in registry:
        name, mode = entry["name"], entry["mode"]
        if mode == "sup" and train_anomaly_frac <= 0:
            print(f"\n[skip] {name}: supervised model needs train_anomaly_frac > 0")
            continue
        print(f"\n{'-'*64}\n  {name}  (mode={mode}, graph={entry['graph']})\n{'-'*64}")
        model = entry["build"]()

        if mode == "sup":
            model = train_model(model, sup_tr, sup_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience, pos_weight=pos_weight)
            val_scores = score_series(model, va_c, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_supervised(val_scores, va_m, test_scores,
                                        objective=sup_objective, min_overlap=event_min_overlap)
        else:
            model = train_model(model, uns_tr, uns_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience)
            clean_scores = score_series(model, tr, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_unsupervised(clean_scores, test_scores,
                                          method=unsup_threshold, pct=unsup_percentile, k=robust_k)
            thr_arr = thr if isinstance(thr, np.ndarray) else np.full(N, thr)
            fpr = ((test_scores > thr_arr) & (te_m == 0)).sum(0) / max((te_m == 0).sum(), 1)
            print(f"   clean-test FPR (mean across sensors): {fpr.mean():.4f}")

        pred = _filter_short_runs(pred, min_event_len)
        m = cell_metrics(pred, te_m)              # metrics on flagged mask (before removal)
        m.update(ranking_metrics(test_scores, te_m))   # threshold-free diagnostics
        m.update(event_metrics(pred, te_m, event_min_overlap))   # run/segment-level
        out_path = os.path.join(out_dir, f"clean_{name}.csv")
        clean_df = export_clean(te_c, scaler, pred, dt_test, sensors, meta, out_path)

        removed = int(pred.sum())
        print(f"   F1={m['F1']:.4f}  Prec={m['Precision']:.4f}  Rec={m['Recall']:.4f}  "
              f"Acc={m['Accuracy']:.4f}  | removed {removed} readings -> {out_path}")
        m.update({"Model": name, "Mode": mode, "Removed": removed, "CleanFile": out_path})
        results.append(m)

    _print_table(results, te_m)
    return results


def _print_table(results, true_mask):
    inj = int((true_mask > 0).sum())
    cols = ["Model", "Mode", "AUPRC", "Cell_F1", "PA_F1", "EV_Recall", "EV_F1", "Removed"]
    key = {"Cell_F1": "F1"}                       # map display name -> result key
    rows = [[r.get(key.get(c, c)) for c in cols] for r in results]
    rows.sort(key=lambda r: (r[4] if r[4] == r[4] else -1), reverse=True)   # by PA_F1
    nev = results[0].get("Events", "?") if results else "?"
    print(f"\n{'='*78}\n  DETECTION  (injected cells: {inj}, anomalous runs/events: {nev})\n{'='*78}")
    if _TAB:
        print(tabulate(rows, headers=cols, tablefmt="fancy_grid", floatfmt=".4f"))
    else:
        print("  ".join(f"{c:<12}" for c in cols))
        for r in rows:
            print("  ".join(f"{str(v):<12}" for v in r))
    print("\nCell_F1  : exact per-(timestep,sensor) match (strictest).")
    print("PA_F1    : point-adjust - a run is credited if any of its cells is flagged.")
    print("EV_Recall/EV_F1 : per-run - did we catch each anomalous segment.")
    print("Event scoring is meaningful for sustained anomalies (run_len > 1).")


# =============================================================================
# ENTRY POINT  (terminal:  python anomaly_cleaning.py --csv ... --adj_csv ...)
#              (jupyter  :  edit the call below and %run the file)
# =============================================================================
# =============================================================================
# ENTRY POINT
#   notebook : edit SENSOR_GROUPS / PARAMS below and run the file (or import `run`)
#   terminal : python anomaly_cleaning.py --csv data.csv --groups_json groups.json \
#                     --binary_anomaly --binary_mode ood --run_len 5
# =============================================================================

# --- domain knowledge: edit this grouping (sensors sharing a group get connected) --
SENSOR_GROUPS = {
    'G1': ['Desk Motion Sensor', 'Office Door Motion Sensor', 'Office Door Sensor'], 
    'G2': ['Office Light Sensor']
}

if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        # ---- NOTEBOOK: all tunables in one place ----
        run(
            csv_path="Data/Office_data/sead_anomaly_free_office_data_13-01-25_to_13-02-25_1h.csv",
            groups=SENSOR_GROUPS,          # use the grouping above (set None to use adj_csv_path)
            adj_csv_path=None,             # e.g. "adjacency.csv" if not using groups
            # windowing / training
            seq_len=24, stride=4, batch_size=64, max_epochs=100, lr=1e-3, patience=20,
            # anomaly injection (binary data)
            binary_anomaly=True, binary_mode="neighbor", run_len=5,   # mode: flip|ood|stuck|neighbor|rate
            anomaly_frac=0.10, train_anomaly_frac=0.05,
            # supervised threshold objective + unsupervised threshold
            sup_objective="event",                               # "cell" or "event"
            unsup_threshold="mean", robust_k=1.5, unsup_percentile=99.0,
            # event scoring / misc
            event_min_overlap=1, min_event_len=1, beta=0.1,
            models=None,                                         # None = all; or e.g. ["TemporalCNN","GraphAnomalyVAE"]
            out_dir="cleaned_output",
        )
    else:
        # ---- TERMINAL ----
        import json
        p = argparse.ArgumentParser(description="Sensor-level anomaly cleaning")
        p.add_argument("--csv", required=True)
        p.add_argument("--adj_csv", default=None, help="labelled adjacency CSV (omit if using --groups_json)")
        p.add_argument("--groups_json", default=None, help="JSON file: {group: [sensor,...]}")
        p.add_argument("--seq_len", type=int, default=24)
        p.add_argument("--stride", type=int, default=4)
        p.add_argument("--batch_size", type=int, default=64)
        p.add_argument("--max_epochs", type=int, default=100)
        p.add_argument("--lr", type=float, default=1e-3)
        p.add_argument("--patience", type=int, default=20)
        p.add_argument("--anomaly_frac", type=float, default=0.10)
        p.add_argument("--train_anomaly_frac", type=float, default=0.08)
        p.add_argument("--anomaly_type", type=str, default="all")
        p.add_argument("--binary_anomaly", action="store_true")
        p.add_argument("--binary_mode", type=str, default="flip",
                       choices=["flip", "ood", "stuck", "neighbor", "rate"])
        p.add_argument("--run_len", type=int, default=5)
        p.add_argument("--sup_objective", type=str, default="cell", choices=["cell", "event"])
        p.add_argument("--unsup_threshold", type=str, default="percentile",
                       choices=["percentile", "median", "mean"])
        p.add_argument("--robust_k", type=float, default=4.0)
        p.add_argument("--unsup_percentile", type=float, default=99.0)
        p.add_argument("--min_event_len", type=int, default=1)
        p.add_argument("--beta", type=float, default=1.0)
        p.add_argument("--out_dir", type=str, default="cleaned_output")
        a = p.parse_args()
        groups = json.load(open(a.groups_json)) if a.groups_json else None
        run(csv_path=a.csv, adj_csv_path=a.adj_csv, groups=groups,
            seq_len=a.seq_len, stride=a.stride, batch_size=a.batch_size, max_epochs=a.max_epochs,
            lr=a.lr, patience=a.patience, anomaly_frac=a.anomaly_frac,
            train_anomaly_frac=a.train_anomaly_frac, anomaly_type=a.anomaly_type,
            binary_anomaly=a.binary_anomaly, run_len=a.run_len, binary_mode=a.binary_mode,
            sup_objective=a.sup_objective, unsup_threshold=a.unsup_threshold, robust_k=a.robust_k,
            unsup_percentile=a.unsup_percentile, min_event_len=a.min_event_len,
            beta=a.beta, out_dir=a.out_dir)


  Sensor-level anomaly cleaning  |  device=cuda
  Loaded 33471 timestamps x 4 sensors
   Sensors: ['Desk Motion Sensor', 'Office Door Motion Sensor', 'Office Door Sensor', 'Office Light Sensor']
   Built graph from 2 groups: 3 undirected edges | density 50.0%
   [groups] isolated sensors (in no group): ['Office Light Sensor']

Injecting anomalies (val / test, and a small fraction into train):

Train cell anomaly rate: 3.62%  |  pos_weight=10.0 (capped at 10.0)

----------------------------------------------------------------
  TemporalCNN  (mode=sup, graph=none)
----------------------------------------------------------------
   trained 24 epoch(s) | best val_AUPRC=0.9965
   F1=0.9828  Prec=0.9738  Rec=0.9921  Acc=0.9976  | removed 1410 readings -> cleaned_output\clean_TemporalCNN.csv

----------------------------------------------------------------
  StackedLSTM  (mode=sup, graph=none)
----------------------------------------------------------------
   trained 54 epoch(s) | best val_

In [ ]:
"""
anomaly_cleaning.py
Sensor-level anomaly detection, removal, and forecasting-based detection.

Reuses data loading / anomaly injection / training utilities from
anomlay_pipeline.py (keep both files in the same folder, or on PYTHONPATH).

Both paradigms flag anomalies at the (timestep, sensor) cell level:
  supervised   : per-cell classifier trained on injected labels
  unsupervised : reconstruction / forecasting error vs a per-sensor threshold

Flow: load -> scale -> inject (val/test, + small frac into train for supervised)
      -> train -> score every cell -> flag -> report metrics on the flagged mask
      -> remove flagged readings -> write clean long-format CSV.

Graph usage:
  predefined adjacency : SpatioTemporalGNN, GNNForecaster
  learned graph        : GNN_LearnableGraph, GAT_LearnedGraph, GraphAnomalyVAE
  no graph             : TemporalCNN, StackedLSTM
"""
import os, sys, argparse, warnings, random
from xml.parsers.expat import model
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score)

warnings.filterwarnings("ignore")

# --- shared pieces from the original pipeline --------------------------------
try:
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)
except ModuleNotFoundError:
    _here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, _here)
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)

try:
    from tabulate import tabulate
    _TAB = True
except ImportError:
    _TAB = False


# =============================================================================
# DATA
# =============================================================================
def _norm_adj(adj_matrix):
    """Symmetric-normalised adjacency with self-loops: D^-1/2 (A+I) D^-1/2."""
    A = (torch.tensor(np.asarray(adj_matrix), dtype=torch.float32) > 0).float()
    A = A + torch.eye(A.shape[0])
    d = A.sum(1).clamp(min=1e-6).pow(-0.5)
    return d[:, None] * A * d[None, :]


def adjacency_from_groups(groups, sensors):
    """Build a symmetric 0/1 adjacency (N x N) from a {group_name: [sensor,...]} dict.
    Every pair of sensors that share a group is connected (each group is a clique).
    Names not present in `sensors` are ignored; sensors in no group stay isolated."""
    idx = {s: i for i, s in enumerate(sensors)}
    N = len(sensors)
    A = np.zeros((N, N), dtype=np.float32)
    missing = set()
    for members in groups.values():
        ids = [idx[s] for s in members if s in idx]
        missing |= {s for s in members if s not in idx}
        for a in ids:
            for b in ids:
                if a != b:
                    A[a, b] = 1.0
    iso = [sensors[i] for i in range(N) if A[i].sum() == 0]
    print(f"   Built graph from {len(groups)} groups: {int(A.sum())//2} undirected edges "
          f"| density {A.sum() / (N * (N - 1)):.1%}")
    if missing:
        print(f"   [groups] ignored {len(missing)} name(s) not in data: {sorted(missing)}")
    if iso:
        print(f"   [groups] isolated sensors (in no group): {iso}")
    return A


def inject(values, frac, atype, adj):
    """Inject anomalies and return (corrupted, cell_mask) where mask[t,s]=1 if
    that exact reading was altered. Mask is derived by diffing the clean array."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr, _ = inject_anomalies(values, frac, atype, "", adj)
    return corr, (corr != values).astype(int)


def inject_binary(values, frac, run_len=1, mode="flip", adj=None, ood_value=1.5, rate=0.7):
    """Binary anomaly injection. Each anomaly spans run_len consecutive steps on one
    sensor. mask marks only cells whose value actually changed. Modes:
      flip     : invert the bit(s) (in-distribution; needs predictable data to detect).
      ood      : set to an out-of-range value (sensor-fault / garbage reading; the
                 binary analogue of a spike - separable in feature space, easiest).
      stuck    : freeze at a constant 0 or 1 over the run (stuck-at fault).
      neighbor : force disagreement with a graph-neighbour (spatial anomaly; detectable
                 from cross-sensor structure even when the series isn't predictable).
      rate     : flip a 'rate' fraction of the run's steps (a burst / activation-rate
                 shift; a sustained statistical anomaly, more robust than one flip)."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr = values.copy()
    mask = np.zeros_like(values, dtype=int)
    T, N = values.shape
    n = max(1, int(T * N * frac / max(run_len, 1)))
    nbrs = None
    if mode == "neighbor" and adj is not None:
        A = np.asarray(adj) > 0
        nbrs = [np.where(A[s])[0] for s in range(N)]
    for _ in range(n):
        t = np.random.randint(0, T - run_len + 1)
        s = np.random.randint(0, N)
        sl = slice(t, t + run_len)
        if mode == "ood":
            corr[sl, s] = ood_value
        elif mode == "stuck":
            corr[sl, s] = float(np.random.randint(0, 2))
        elif mode == "neighbor" and nbrs is not None and len(nbrs[s]):
            corr[sl, s] = 1.0 - values[sl, np.random.choice(nbrs[s])]
        elif mode == "rate":
            pick = np.where(np.random.rand(run_len) < rate)[0] + t
            corr[pick, s] = 1.0 - corr[pick, s]
        else:                                            # flip
            corr[sl, s] = 1.0 - corr[sl, s]
        mask[sl, s] = (corr[sl, s] != values[sl, s]).astype(int)
    return corr, mask


class WindowDS(Dataset):
    """Sliding windows of (seq_len, N) with matching per-cell labels."""
    def __init__(self, values, cell_labels, seq_len, stride):
        self.x, self.y = [], []
        T = len(values)
        starts = list(range(0, T - seq_len + 1, stride))
        if starts and starts[-1] != T - seq_len:
            starts.append(T - seq_len)
        for s in starts:
            self.x.append(torch.tensor(values[s:s + seq_len], dtype=torch.float32))
            self.y.append(torch.tensor(cell_labels[s:s + seq_len], dtype=torch.float32))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]


# =============================================================================
# MODELS  (all output per-cell logits (B,T,N) for sup, or expose cell scores)
# =============================================================================
class TemporalCNN(nn.Module):
    def __init__(self, N, seq_len, hidden=32, levels=4, k=3):
        super().__init__()
        chs = [N] + [hidden] * levels
        self.tcn = nn.Sequential(*[TCNBlock(chs[i], chs[i + 1], k, 2 ** i) for i in range(levels)])
        self.head = nn.Conv1d(hidden, N, 1)

    def forward(self, x):                                   # (B,T,N)
        h = self.tcn(x.permute(0, 2, 1))                    # (B,hidden,T)
        return self.head(h).permute(0, 2, 1)                # (B,T,N)


class StackedLSTM(nn.Module):
    def __init__(self, N, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(N, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, N)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out)                               # (B,T,N)


class GCNDetector(nn.Module):
    """GCN + per-node GRU per-cell classifier. Fixed graph if adj given, else learned."""
    def __init__(self, N, adj=None, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N, self.learn = N, adj is None
        if self.learn:
            self.adj_logits = nn.Parameter(torch.randn(N, N))
        else:
            self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def _A(self, device):
        if not self.learn:
            return self.A_hat
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def forward(self, x):
        B, T, N = x.shape
        feats = self.W(x.unsqueeze(-1))                     # (B,T,N,H)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device), feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)                              # (B*N,T,rnn)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GATDetector(nn.Module):
    """Learned graph: per-timestep multi-head attention over sensors (GAT on a
    fully-connected graph), then per-node GRU, then per-cell head."""
    def __init__(self, N, hidden=16, heads=2, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.proj = nn.Linear(1, hidden)
        self.attn = nn.MultiheadAttention(hidden, heads, batch_first=True, dropout=0.1)
        self.rnn = nn.GRU(hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):
        B, T, N = x.shape
        f = self.proj(x.reshape(B * T, N, 1))               # (B*T,N,hidden)
        a, _ = self.attn(f, f, f)                           # attention = learned graph
        a = torch.relu(a).reshape(B, T, N, -1)
        seq = a.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GraphAnomalyVAE(nn.Module):
    """Unsupervised graph VAE (learned adjacency). Cell score = reconstruction MSE."""
    def __init__(self, N, seq_len, gcn_hidden=16, rnn_hidden=32, latent=16, beta=1.0):
        super().__init__()
        self.N, self.seq_len, self.beta = N, seq_len, beta
        self.adj_logits = nn.Parameter(torch.randn(N, N))
        self.enc_proj = nn.Linear(1, gcn_hidden, bias=False)
        self.enc_rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.fc_mu = nn.Linear(rnn_hidden, latent)
        self.fc_lv = nn.Linear(rnn_hidden, latent)
        self.dec_fc = nn.Linear(latent, rnn_hidden)
        self.dec_rnn = nn.GRU(rnn_hidden, rnn_hidden, batch_first=True)
        self.dec_W = nn.Linear(rnn_hidden, gcn_hidden, bias=False)
        self.dec_out = nn.Linear(gcn_hidden, 1)

    def _A(self, device):
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def encode(self, x):
        B, T, N = x.shape
        f = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device),
                                    self.enc_proj(x.unsqueeze(-1))))
        seq = f.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        _, h = self.enc_rnn(seq)
        h = h.squeeze(0).reshape(B, N, -1).mean(1)
        return self.fc_mu(h), self.fc_lv(h)

    def decode(self, z):
        h = self.dec_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.dec_rnn(h)
        node = self.dec_W(out).unsqueeze(2).repeat(1, 1, self.N, 1)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(z.device), node))
        return self.dec_out(prop).squeeze(-1)               # (B,T,N)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return self.decode(z), mu, lv

    def loss(self, x):
        recon, mu, lv = self.forward(x)
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        return F.mse_loss(recon, x) + self.beta * kl

    def cell_score(self, x):
        recon, _, _ = self.forward(x)
        return (x - recon).pow(2)                           # (B,T,N)


class GNNForecaster(nn.Module):
    """Unsupervised one-step forecaster on the predefined graph. Trained on clean
    data; per-sensor score = squared next-step prediction error."""
    def __init__(self, N, adj, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):                                   # predict step T from steps 0..T-2
        B, T, N = x.shape
        xi = x[:, :-1, :]
        feats = self.W(xi.unsqueeze(-1))
        prop = torch.relu(torch.einsum("ij,btjh->btih", self.A_hat, feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
        out, _ = self.rnn(seq)
        return self.head(out[:, -1, :]).reshape(B, N)       # (B,N)

    def loss(self, x):
        return F.mse_loss(self.forward(x), x[:, -1, :])

    def step_score(self, x):
        return (self.forward(x) - x[:, -1, :]).pow(2)       # (B,N) at last step


# =============================================================================
# TRAIN / SCORE / FLAG / EVAL
# =============================================================================
def train_model(model, train_loader, val_loader, mode, device,
                lr=1e-3, epochs=50, patience=10, pos_weight=None):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device) if pos_weight is not None else None)
    stop = EarlyStopping(patience=patience, mode="f1" if mode == "sup" else "loss")

    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(device)
            opt.zero_grad()
            loss = crit(model(x), y.to(device)) if mode == "sup" else model.loss(x)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            if mode == "sup":
                ps, ys = [], []
                for x, y in val_loader:
                    ps.append(torch.sigmoid(model(x.to(device))).cpu().numpy().reshape(-1))
                    ys.append(y.numpy().reshape(-1))
                ys = np.concatenate(ys)
                # Ranking metric (AUPRC): tracks learning even when no fixed 0.5
                # cut separates the classes. F1@0.5 stays 0 on rare-positive data
                # and would stop training immediately.
                metric = average_precision_score(ys, np.concatenate(ps)) if ys.any() else 0.0
            else:
                metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in val_loader]))
        if stop.step(metric, model):
            stopped = ep + 1
            break
    else:
        stopped = epochs
    stop.restore(model)
    if mode == "sup":
        print(f"   trained {stopped} epoch(s) | val_AUPRC={stop.best:.4f}")
    else:
        model.eval()
        with torch.no_grad():
            train_metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in train_loader]))
        print(f"   trained {stopped} epoch(s) | train_loss={train_metric:.4f} | best {'val_AUPRC' if mode == 'sup' else 'val_loss'}={stop.best:.4f}")
    return model


def score_series(model, values, seq_len, stride, mode, device):
    """Return a (T, N) anomaly-score map, averaging overlapping windows."""
    model.eval()
    T, N = values.shape
    if mode == "forecast":
        stride = 1                                # score every incoming step
    acc = np.zeros((T, N)); cnt = np.zeros((T, N))
    starts = list(range(0, T - seq_len + 1, stride))
    if starts and starts[-1] != T - seq_len:
        starts.append(T - seq_len)
    with torch.no_grad():
        for s in starts:
            x = torch.tensor(values[s:s + seq_len], dtype=torch.float32)[None].to(device)
            if mode == "sup":
                sc = torch.sigmoid(model(x))[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            elif mode == "vae":
                sc = model.cell_score(x)[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            else:                                            # forecast: last step only
                t = s + seq_len - 1
                acc[t] += model.step_score(x)[0].cpu().numpy(); cnt[t] += 1
    cnt[cnt == 0] = 1
    return acc / cnt


def flag_supervised(val_scores, val_mask, test_scores, objective="cell", min_overlap=1):
    """Tune one global probability cut on val. objective='cell' maximises cell F1;
    'event' maximises point-adjust F1 (aligns the cut with run-level scoring)."""
    thr, best = float(np.median(val_scores)), -1.0
    for t in np.linspace(val_scores.min(), val_scores.max(), 100):
        vp = (val_scores >= t).astype(int)
        if objective == "event":
            f = event_metrics(vp, val_mask, min_overlap)["PA_F1"]
        else:
            f = f1_score(val_mask.reshape(-1), vp.reshape(-1).astype(int), zero_division=0)
        if f > best:
            best, thr = f, t
    return (test_scores >= thr).astype(int), float(thr)


def flag_unsupervised(clean_scores, test_scores, method="percentile", pct=99.0, k=4.0):
    """Per-sensor threshold (higher score = more anomalous).
    percentile : pct-th percentile of clean-TRAIN scores (assumes train ~= test).
    median     : median + k*MAD of the scored set itself (adapts to its distribution).
    mean       : mean + k*std of the scored set itself.
    Robust methods need the score to rank anomalies ABOVE normals (AUROC > 0.5)."""
    if method == "percentile":
        thr = np.percentile(clean_scores, pct, axis=0)
    elif method == "median":
        med = np.median(test_scores, axis=0)
        mad = np.median(np.abs(test_scores - med), axis=0)
        thr = med + k * (mad + 1e-9)
    elif method == "mean":
        thr = test_scores.mean(0) + k * (test_scores.std(0) + 1e-9)
    else:
        raise ValueError(f"unknown unsup_threshold '{method}'")
    return (test_scores >= thr).astype(int), thr


def cell_metrics(pred_mask, true_mask):
    p = pred_mask.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    return {"Accuracy":  round(accuracy_score(t, p), 4),
            "Precision": round(precision_score(t, p, zero_division=0), 4),
            "Recall":    round(recall_score(t, p, zero_division=0), 4),
            "F1":        round(f1_score(t, p, zero_division=0), 4)}


def ranking_metrics(scores, true_mask):
    """Threshold-free quality of the score map (does it rank anomalies high?)."""
    s = scores.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    if t.sum() == 0 or t.sum() == len(t):
        return {"AUROC": float("nan"), "AUPRC": float("nan")}
    return {"AUROC": round(roc_auc_score(t, s), 4),
            "AUPRC": round(average_precision_score(t, s), 4)}


def _segments(col):
    """Contiguous runs of 1s in a 1D bool array -> list of (start, end_exclusive)."""
    segs, i, n = [], 0, len(col)
    while i < n:
        if col[i]:
            j = i
            while j < n and col[j]:
                j += 1
            segs.append((i, j)); i = j
        else:
            i += 1
    return segs


def _filter_short_runs(mask, min_len):
    """Drop predicted runs shorter than min_len per sensor (removes isolated false
    flags; useful when real anomalies are sustained)."""
    if min_len <= 1:
        return mask
    out = np.zeros_like(mask)
    for s in range(mask.shape[1]):
        for a, b in _segments(mask[:, s].astype(bool)):
            if b - a >= min_len:
                out[a:b, s] = 1
    return out


def event_metrics(pred_mask, true_mask, min_overlap=1):
    """Score whole anomalous runs instead of individual cells (per sensor).
    point-adjust (PA): if >=min_overlap cells of a true run are flagged, the WHOLE
        run is credited; P/R/F1 then computed point-wise on the adjusted mask
        (standard time-series AD protocol).
    event-overlap (EV): a true run is a hit if it's flagged at all; a predicted run
        that touches no true run is a false alarm. P/R/F1 over runs."""
    pred = pred_mask.astype(bool); true = (np.asarray(true_mask) > 0)
    T, N = true.shape
    adj = pred.copy()
    tp_ev = fn_ev = fp_ev = pred_ev = 0
    for s in range(N):
        for a, b in _segments(true[:, s]):
            if pred[a:b, s].sum() >= min_overlap:
                adj[a:b, s] = True; tp_ev += 1
            else:
                fn_ev += 1
        for a, b in _segments(pred[:, s]):
            pred_ev += 1
            if not true[a:b, s].any():
                fp_ev += 1
    p, t = adj.reshape(-1), true.reshape(-1)
    ev_p = (pred_ev - fp_ev) / pred_ev if pred_ev else 0.0
    ev_r = tp_ev / (tp_ev + fn_ev) if (tp_ev + fn_ev) else 0.0
    ev_f1 = 2 * ev_p * ev_r / (ev_p + ev_r) if (ev_p + ev_r) else 0.0
    return {"PA_Precision": round(precision_score(t, p, zero_division=0), 4),
            "PA_Recall":    round(recall_score(t, p, zero_division=0), 4),
            "PA_F1":        round(f1_score(t, p, zero_division=0), 4),
            "EV_Recall":    round(ev_r, 4),
            "EV_F1":        round(ev_f1, 4),
            "Events":       tp_ev + fn_ev}


# =============================================================================
# CLEAN OUTPUT
# =============================================================================
def _sensor_meta(csv_path):
    try:
        df = pd.read_csv(csv_path)
        cols = [c for c in ("thing_name", "thing_ip") if c in df.columns]
        if not cols:
            return {}
        g = df.drop_duplicates("sensor_name").set_index("sensor_name")[cols]
        return {s: row.to_dict() for s, row in g.iterrows()}
    except Exception:
        return {}


def export_clean(corrupted, scaler, pred_mask, dt_index, sensors, meta, out_path):
    """Drop flagged (datetime, sensor) readings; write the rest as long-format CSV."""
    inv = scaler.inverse_transform(corrupted)
    rows = []
    for ti, dt in enumerate(dt_index):
        dt = pd.Timestamp(dt)
        for sj, sname in enumerate(sensors):
            if pred_mask[ti, sj]:
                continue
            m = meta.get(sname, {})
            rows.append({
                "datetime": dt, "date": dt.strftime("%Y-%m-%d"),
                "time": dt.strftime("%H:%M"), "seconds": dt.timestamp(),
                "state": float(inv[ti, sj]), "sensor_name": sname,
                "thing_name": m.get("thing_name"), "thing_ip": m.get("thing_ip"),
            })
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    return df


# =============================================================================
# REGISTRY & ORCHESTRATION
# =============================================================================
def build_registry(N, seq_len, adj, beta):
    return [
        {"name": "TemporalCNN",        "mode": "sup",      "graph": "none",
         "build": lambda: TemporalCNN(N, seq_len)},
        {"name": "StackedLSTM",        "mode": "sup",      "graph": "none",
         "build": lambda: StackedLSTM(N)},
        {"name": "SpatioTemporalGNN",  "mode": "sup",      "graph": "predefined",
         "build": lambda: GCNDetector(N, adj=adj)},
        {"name": "GNN_LearnableGraph", "mode": "sup",      "graph": "learned",
         "build": lambda: GCNDetector(N, adj=None)},
        {"name": "GAT_LearnedGraph",   "mode": "sup",      "graph": "learned",
         "build": lambda: GATDetector(N)},
        {"name": "GraphAnomalyVAE",    "mode": "vae",      "graph": "learned",
         "build": lambda: GraphAnomalyVAE(N, seq_len, beta=beta)},
        {"name": "GNNForecaster",      "mode": "forecast", "graph": "predefined",
         "build": lambda: GNNForecaster(N, adj=adj)},
    ]


def run(csv_path, adj_csv_path=None, groups=None,
        seq_len=24, stride=4, batch_size=64, max_epochs=50, lr=1e-3, patience=15,
        anomaly_frac=0.10, train_anomaly_frac=0.05, anomaly_type="all",
        binary_anomaly=False, run_len=1, binary_mode="flip",
        unsup_percentile=99.0, unsup_threshold="percentile", robust_k=4.0,
        event_min_overlap=1, min_event_len=1, sup_objective="cell",
        pos_weight_cap=10.0, train_frac=0.70, val_frac=0.15,
        out_dir="cleaned_output", models=None, beta=1.0, seed=42):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*64}\n  Sensor-level anomaly cleaning  |  device={device}\n{'='*64}")
    wide = load_and_pivot(csv_path)
    sensors = list(wide.columns); N = len(sensors)
    raw = wide.values.astype(np.float32); T = len(raw)
    n_tr, n_va = int(T * train_frac), int(T * val_frac)

    scaler = MinMaxScaler()
    tr = scaler.fit_transform(raw[:n_tr])
    va = scaler.transform(raw[n_tr:n_tr + n_va])
    te = scaler.transform(raw[n_tr + n_va:])
    dt_test = wide.index[n_tr + n_va:]

    if groups is not None:
        adj = adjacency_from_groups(groups, sensors)
    elif adj_csv_path is not None:
        _, adj = load_edge_index_from_csv(adj_csv_path, sensors)
    else:
        raise ValueError("provide either `groups` (dict) or `adj_csv_path`")

    print("\nInjecting anomalies (val / test, and a small fraction into train):")
    _inj = (lambda v, f: inject_binary(v, f, run_len, binary_mode, adj)) if binary_anomaly \
        else (lambda v, f: inject(v, f, anomaly_type, adj))
    va_c, va_m = _inj(va, anomaly_frac)
    te_c, te_m = _inj(te, anomaly_frac)
    tr_c, tr_m = _inj(tr, train_anomaly_frac)
    meta = _sensor_meta(csv_path)

    # data loaders
    zeros_tr = np.zeros_like(tr, dtype=int)
    sup_tr = DataLoader(WindowDS(tr_c, tr_m, seq_len, stride), batch_size=batch_size, shuffle=True)
    sup_va = DataLoader(WindowDS(va_c, va_m, seq_len, stride), batch_size=batch_size, shuffle=False)
    uns_tr = DataLoader(WindowDS(tr, zeros_tr, seq_len, stride), batch_size=batch_size, shuffle=True)
    uns_va = DataLoader(WindowDS(va_c, np.zeros_like(va, int), seq_len, stride), batch_size=batch_size, shuffle=False)

    pos = tr_m.sum(); neg = tr_m.size - pos
    pw = min(neg / max(pos, 1), pos_weight_cap)
    pos_weight = torch.tensor([pw], dtype=torch.float32)
    print(f"\nTrain cell anomaly rate: {100 * pos / tr_m.size:.2f}%  |  pos_weight={pw:.1f} "
          f"(capped at {pos_weight_cap})")

    registry = build_registry(N, seq_len, adj, beta)
    if models:
        registry = [m for m in registry if m["name"] in models]

    results = []
    for entry in registry:
        name, mode = entry["name"], entry["mode"]
        if mode == "sup" and train_anomaly_frac <= 0:
            print(f"\n[skip] {name}: supervised model needs train_anomaly_frac > 0")
            continue
        print(f"\n{'-'*64}\n  {name}  (mode={mode}, graph={entry['graph']})\n{'-'*64}")
        model = entry["build"]()

        if mode == "sup":
            model = train_model(model, sup_tr, sup_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience, pos_weight=pos_weight)
            val_scores = score_series(model, va_c, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_supervised(val_scores, va_m, test_scores,
                                        objective=sup_objective, min_overlap=event_min_overlap)
        else:
            model = train_model(model, uns_tr, uns_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience)
            clean_scores = score_series(model, tr, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_unsupervised(clean_scores, test_scores,
                                          method=unsup_threshold, pct=unsup_percentile, k=robust_k)
            thr_arr = thr if isinstance(thr, np.ndarray) else np.full(N, thr)
            fpr = ((test_scores > thr_arr) & (te_m == 0)).sum(0) / max((te_m == 0).sum(), 1)
            print(f"   clean-test FPR (mean across sensors): {fpr.mean():.4f}")

        pred = _filter_short_runs(pred, min_event_len)
        m = cell_metrics(pred, te_m)              # metrics on flagged mask (before removal)
        m.update(ranking_metrics(test_scores, te_m))   # threshold-free diagnostics
        m.update(event_metrics(pred, te_m, event_min_overlap))   # run/segment-level
        m["_scores"] = test_scores  # for debugging / analysis
        out_path = os.path.join(out_dir, f"clean_{name}.csv")
        clean_df = export_clean(te_c, scaler, pred, dt_test, sensors, meta, out_path)

        removed = int(pred.sum())
        print(f"   F1={m['F1']:.4f}  Prec={m['Precision']:.4f}  Rec={m['Recall']:.4f}  "
              f"Acc={m['Accuracy']:.4f}  | removed {removed} readings -> {out_path}")
        m.update({"Model": name, "Mode": mode, "Removed": removed, "CleanFile": out_path})
        results.append(m)

    _print_table(results, te_m)
    return results


def _print_table(results, true_mask):
    inj = int((true_mask > 0).sum())
    cols = ["Model", "Mode", "AUPRC", "Cell_F1", "PA_F1", "EV_Recall", "EV_F1", "Removed"]
    key = {"Cell_F1": "F1"}                       # map display name -> result key
    rows = [[r.get(key.get(c, c)) for c in cols] for r in results]
    rows.sort(key=lambda r: (r[4] if r[4] == r[4] else -1), reverse=True)   # by PA_F1
    nev = results[0].get("Events", "?") if results else "?"
    print(f"\n{'='*78}\n  DETECTION  (injected cells: {inj}, anomalous runs/events: {nev})\n{'='*78}")
    if _TAB:
        print(tabulate(rows, headers=cols, tablefmt="fancy_grid", floatfmt=".4f"))
    else:
        print("  ".join(f"{c:<12}" for c in cols))
        for r in rows:
            print("  ".join(f"{str(v):<12}" for v in r))
    print("\nCell_F1  : exact per-(timestep,sensor) match (strictest).")
    print("PA_F1    : point-adjust - a run is credited if any of its cells is flagged.")
    print("EV_Recall/EV_F1 : per-run - did we catch each anomalous segment.")
    print("Event scoring is meaningful for sustained anomalies (run_len > 1).")


# =============================================================================
# ENTRY POINT  (terminal:  python anomaly_cleaning.py --csv ... --adj_csv ...)
#              (jupyter  :  edit the call below and %run the file)
# =============================================================================
# =============================================================================
# ENTRY POINT
#   notebook : edit SENSOR_GROUPS / PARAMS below and run the file (or import `run`)
#   terminal : python anomaly_cleaning.py --csv data.csv --groups_json groups.json \
#                     --binary_anomaly --binary_mode ood --run_len 5
# =============================================================================

# --- domain knowledge: edit this grouping (sensors sharing a group get connected) --
SENSOR_GROUPS = {
    'G1': ['Front Motion Sensor', 'Lab Door Motion Sensor', 'Sonar Sensor'],
    'G2': ['Lab Door Motion Sensor', 'Right Motion Sensor', 'Sonar Sensor'],
    'G3': ['Lab Door Motion Sensor', 'Lab Light Sensor', 'Sonar Sensor'],
    'G4': ['Lab Door Motion Sensor', 'Lab Door Sensor', 'Sonar Sensor'],
    'G5': ['Lab Door Motion Sensor', 'Left Motion Sensor', 'Sonar Sensor'],
    'G6': ['Back Door Sensor', 'Back Motion Sensor', 'Vibration Sensor'],
}

if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        # ---- NOTEBOOK: all tunables in one place ----
        run(
            csv_path="Data/Lab_Data/sead_anomaly_free_lab_data_1h.csv",
            groups=SENSOR_GROUPS,          # use the grouping above (set None to use adj_csv_path)
            adj_csv_path=None,             # e.g. "adjacency.csv" if not using groups
            # windowing / training
            seq_len=24, stride=4, batch_size=64, max_epochs=100, lr=1e-3, patience=20,
            # anomaly injection (binary data)
            binary_anomaly=True, binary_mode="ood", run_len=5,   # mode: flip|ood|stuck|neighbor|rate
            anomaly_frac=0.10, train_anomaly_frac=0.08,
            # supervised threshold objective + unsupervised threshold
            sup_objective="event",                               # "cell" or "event"
            unsup_threshold="mean", robust_k=3, unsup_percentile=99.0,
            # event scoring / misc
            event_min_overlap=1, min_event_len=1, beta=0.1,
            models=None,                                         # None = all; or e.g. ["TemporalCNN","GraphAnomalyVAE"]
            out_dir="cleaned_output",
        )
    else:
        # ---- TERMINAL ----
        import json
        p = argparse.ArgumentParser(description="Sensor-level anomaly cleaning")
        p.add_argument("--csv", required=True)
        p.add_argument("--adj_csv", default=None, help="labelled adjacency CSV (omit if using --groups_json)")
        p.add_argument("--groups_json", default=None, help="JSON file: {group: [sensor,...]}")
        p.add_argument("--seq_len", type=int, default=24)
        p.add_argument("--stride", type=int, default=4)
        p.add_argument("--batch_size", type=int, default=64)
        p.add_argument("--max_epochs", type=int, default=100)
        p.add_argument("--lr", type=float, default=1e-3)
        p.add_argument("--patience", type=int, default=20)
        p.add_argument("--anomaly_frac", type=float, default=0.10)
        p.add_argument("--train_anomaly_frac", type=float, default=0.08)
        p.add_argument("--anomaly_type", type=str, default="all")
        p.add_argument("--binary_anomaly", action="store_true")
        p.add_argument("--binary_mode", type=str, default="flip",
                       choices=["flip", "ood", "stuck", "neighbor", "rate"])
        p.add_argument("--run_len", type=int, default=5)
        p.add_argument("--sup_objective", type=str, default="cell", choices=["cell", "event"])
        p.add_argument("--unsup_threshold", type=str, default="percentile",
                       choices=["percentile", "median", "mean"])
        p.add_argument("--robust_k", type=float, default=4.0)
        p.add_argument("--unsup_percentile", type=float, default=99.0)
        p.add_argument("--min_event_len", type=int, default=1)
        p.add_argument("--beta", type=float, default=1.0)
        p.add_argument("--out_dir", type=str, default="cleaned_output")
        a = p.parse_args()
        groups = json.load(open(a.groups_json)) if a.groups_json else None
        run(csv_path=a.csv, adj_csv_path=a.adj_csv, groups=groups,
            seq_len=a.seq_len, stride=a.stride, batch_size=a.batch_size, max_epochs=a.max_epochs,
            lr=a.lr, patience=a.patience, anomaly_frac=a.anomaly_frac,
            train_anomaly_frac=a.train_anomaly_frac, anomaly_type=a.anomaly_type,
            binary_anomaly=a.binary_anomaly, run_len=a.run_len, binary_mode=a.binary_mode,
            sup_objective=a.sup_objective, unsup_threshold=a.unsup_threshold, robust_k=a.robust_k,
            unsup_percentile=a.unsup_percentile, min_event_len=a.min_event_len,
            beta=a.beta, out_dir=a.out_dir)


  Sensor-level anomaly cleaning  |  device=cuda
  Loaded 33199 timestamps x 10 sensors
   Sensors: ['Back Door Sensor', 'Back Motion Sensor', 'Front Motion Sensor', 'Lab Door Motion Sensor', 'Lab Door Sensor', 'Lab Light Sensor', 'Left Motion Sensor', 'Right Motion Sensor', 'Sonar Sensor', 'Vibration Sensor']
   Built graph from 6 groups: 14 undirected edges | density 31.1%

Injecting anomalies (val / test, and a small fraction into train):

Train cell anomaly rate: 3.30%  |  pos_weight=10.0 (capped at 10.0)

----------------------------------------------------------------
  TemporalCNN  (mode=sup, graph=none)
----------------------------------------------------------------
   trained 61 epoch(s) | val_AUPRC=0.6185
   F1=0.3652  Prec=0.4802  Rec=0.2946  Acc=0.9516  | removed 1443 readings -> cleaned_output\clean_TemporalCNN.csv

----------------------------------------------------------------
  StackedLSTM  (mode=sup, graph=none)
-------------------------------------------------------

In [24]:
"""
anomaly_cleaning.py
Sensor-level anomaly detection, removal, and forecasting-based detection.

Reuses data loading / anomaly injection / training utilities from
anomlay_pipeline.py (keep both files in the same folder, or on PYTHONPATH).

Both paradigms flag anomalies at the (timestep, sensor) cell level:
  supervised   : per-cell classifier trained on injected labels
  unsupervised : reconstruction / forecasting error vs a per-sensor threshold

Flow: load -> scale -> inject (val/test, + small frac into train for supervised)
      -> train -> score every cell -> flag -> report metrics on the flagged mask
      -> remove flagged readings -> write clean long-format CSV.

Graph usage:
  predefined adjacency : SpatioTemporalGNN, GNNForecaster
  learned graph        : GNN_LearnableGraph, GAT_LearnedGraph, GraphAnomalyVAE
  no graph             : TemporalCNN, StackedLSTM
"""
import os, sys, argparse, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score)

warnings.filterwarnings("ignore")

# --- shared pieces from the original pipeline --------------------------------
try:
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)
except ModuleNotFoundError:
    _here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, _here)
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)

try:
    from tabulate import tabulate
    _TAB = True
except ImportError:
    _TAB = False


# =============================================================================
# DATA
# =============================================================================
def _norm_adj(adj_matrix):
    """Symmetric-normalised adjacency with self-loops: D^-1/2 (A+I) D^-1/2."""
    A = (torch.tensor(np.asarray(adj_matrix), dtype=torch.float32) > 0).float()
    A = A + torch.eye(A.shape[0])
    d = A.sum(1).clamp(min=1e-6).pow(-0.5)
    return d[:, None] * A * d[None, :]


def adjacency_from_groups(groups, sensors):
    """Build a symmetric 0/1 adjacency (N x N) from a {group_name: [sensor,...]} dict.
    Every pair of sensors that share a group is connected (each group is a clique).
    Names not present in `sensors` are ignored; sensors in no group stay isolated."""
    idx = {s: i for i, s in enumerate(sensors)}
    N = len(sensors)
    A = np.zeros((N, N), dtype=np.float32)
    missing = set()
    for members in groups.values():
        ids = [idx[s] for s in members if s in idx]
        missing |= {s for s in members if s not in idx}
        for a in ids:
            for b in ids:
                if a != b:
                    A[a, b] = 1.0
    iso = [sensors[i] for i in range(N) if A[i].sum() == 0]
    print(f"   Built graph from {len(groups)} groups: {int(A.sum())//2} undirected edges "
          f"| density {A.sum() / (N * (N - 1)):.1%}")
    if missing:
        print(f"   [groups] ignored {len(missing)} name(s) not in data: {sorted(missing)}")
    if iso:
        print(f"   [groups] isolated sensors (in no group): {iso}")
    return A


def inject(values, frac, atype, adj):
    """Inject anomalies and return (corrupted, cell_mask) where mask[t,s]=1 if
    that exact reading was altered. Mask is derived by diffing the clean array."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr, _ = inject_anomalies(values, frac, atype, "", adj)
    return corr, (corr != values).astype(int)


def inject_binary(values, frac, run_len=1, mode="flip", adj=None, ood_value=1.5, rate=0.7):
    """Binary anomaly injection. Each anomaly spans run_len consecutive steps on one
    sensor. mask marks only cells whose value actually changed. Modes:
      flip     : invert the bit(s) (in-distribution; needs predictable data to detect).
      ood      : set to an out-of-range value (sensor-fault / garbage reading; the
                 binary analogue of a spike - separable in feature space, easiest).
      stuck    : freeze at a constant 0 or 1 over the run (stuck-at fault).
      neighbor : force disagreement with a graph-neighbour (spatial anomaly; detectable
                 from cross-sensor structure even when the series isn't predictable).
      rate     : flip a 'rate' fraction of the run's steps (a burst / activation-rate
                 shift; a sustained statistical anomaly, more robust than one flip)."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr = values.copy()
    mask = np.zeros_like(values, dtype=int)
    T, N = values.shape
    n = max(1, int(T * N * frac / max(run_len, 1)))
    nbrs = None
    if mode == "neighbor" and adj is not None:
        A = np.asarray(adj) > 0
        nbrs = [np.where(A[s])[0] for s in range(N)]
    for _ in range(n):
        t = np.random.randint(0, T - run_len + 1)
        s = np.random.randint(0, N)
        sl = slice(t, t + run_len)
        if mode == "ood":
            corr[sl, s] = ood_value
        elif mode == "stuck":
            corr[sl, s] = float(np.random.randint(0, 2))
        elif mode == "neighbor" and nbrs is not None and len(nbrs[s]):
            corr[sl, s] = 1.0 - values[sl, np.random.choice(nbrs[s])]
        elif mode == "rate":
            pick = np.where(np.random.rand(run_len) < rate)[0] + t
            corr[pick, s] = 1.0 - corr[pick, s]
        else:                                            # flip
            corr[sl, s] = 1.0 - corr[sl, s]
        mask[sl, s] = (corr[sl, s] != values[sl, s]).astype(int)
    return corr, mask


class WindowDS(Dataset):
    """Sliding windows of (seq_len, N) with matching per-cell labels."""
    def __init__(self, values, cell_labels, seq_len, stride):
        self.x, self.y = [], []
        T = len(values)
        starts = list(range(0, T - seq_len + 1, stride))
        if starts and starts[-1] != T - seq_len:
            starts.append(T - seq_len)
        for s in starts:
            self.x.append(torch.tensor(values[s:s + seq_len], dtype=torch.float32))
            self.y.append(torch.tensor(cell_labels[s:s + seq_len], dtype=torch.float32))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]


# =============================================================================
# MODELS  (all output per-cell logits (B,T,N) for sup, or expose cell scores)
# =============================================================================
class TemporalCNN(nn.Module):
    def __init__(self, N, seq_len, hidden=32, levels=4, k=3):
        super().__init__()
        chs = [N] + [hidden] * levels
        self.tcn = nn.Sequential(*[TCNBlock(chs[i], chs[i + 1], k, 2 ** i) for i in range(levels)])
        self.head = nn.Conv1d(hidden, N, 1)

    def forward(self, x):                                   # (B,T,N)
        h = self.tcn(x.permute(0, 2, 1))                    # (B,hidden,T)
        return self.head(h).permute(0, 2, 1)                # (B,T,N)


class StackedLSTM(nn.Module):
    def __init__(self, N, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(N, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, N)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out)                               # (B,T,N)


class GCNDetector(nn.Module):
    """GCN + per-node GRU per-cell classifier. Fixed graph if adj given, else learned."""
    def __init__(self, N, adj=None, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N, self.learn = N, adj is None
        if self.learn:
            self.adj_logits = nn.Parameter(torch.randn(N, N))
        else:
            self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def _A(self, device):
        if not self.learn:
            return self.A_hat
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def forward(self, x):
        B, T, N = x.shape
        feats = self.W(x.unsqueeze(-1))                     # (B,T,N,H)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device), feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)                              # (B*N,T,rnn)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GATDetector(nn.Module):
    """Learned graph: per-timestep multi-head attention over sensors (GAT on a
    fully-connected graph), then per-node GRU, then per-cell head."""
    def __init__(self, N, hidden=16, heads=2, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.proj = nn.Linear(1, hidden)
        self.attn = nn.MultiheadAttention(hidden, heads, batch_first=True, dropout=0.1)
        self.rnn = nn.GRU(hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):
        B, T, N = x.shape
        f = self.proj(x.reshape(B * T, N, 1))               # (B*T,N,hidden)
        a, _ = self.attn(f, f, f)                           # attention = learned graph
        a = torch.relu(a).reshape(B, T, N, -1)
        seq = a.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GraphAnomalyVAE(nn.Module):
    """Unsupervised graph VAE (learned adjacency). Cell score = reconstruction MSE."""
    def __init__(self, N, seq_len, gcn_hidden=16, rnn_hidden=32, latent=16, beta=1.0):
        super().__init__()
        self.N, self.seq_len, self.beta = N, seq_len, beta
        self.adj_logits = nn.Parameter(torch.randn(N, N))
        self.enc_proj = nn.Linear(1, gcn_hidden, bias=False)
        self.enc_rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.fc_mu = nn.Linear(rnn_hidden, latent)
        self.fc_lv = nn.Linear(rnn_hidden, latent)
        self.dec_fc = nn.Linear(latent, rnn_hidden)
        self.dec_rnn = nn.GRU(rnn_hidden, rnn_hidden, batch_first=True)
        self.dec_W = nn.Linear(rnn_hidden, gcn_hidden, bias=False)
        self.dec_out = nn.Linear(gcn_hidden, 1)

    def _A(self, device):
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def encode(self, x):
        B, T, N = x.shape
        f = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device),
                                    self.enc_proj(x.unsqueeze(-1))))
        seq = f.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        _, h = self.enc_rnn(seq)
        h = h.squeeze(0).reshape(B, N, -1).mean(1)
        return self.fc_mu(h), self.fc_lv(h)

    def decode(self, z):
        h = self.dec_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.dec_rnn(h)
        node = self.dec_W(out).unsqueeze(2).repeat(1, 1, self.N, 1)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(z.device), node))
        return self.dec_out(prop).squeeze(-1)               # (B,T,N)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return self.decode(z), mu, lv

    def loss(self, x):
        recon, mu, lv = self.forward(x)
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        return F.mse_loss(recon, x) + self.beta * kl

    def cell_score(self, x):
        recon, _, _ = self.forward(x)
        return (x - recon).pow(2)                           # (B,T,N)


class GNNForecaster(nn.Module):
    """Unsupervised one-step forecaster on the predefined graph. Trained on clean
    data; per-sensor score = squared next-step prediction error."""
    def __init__(self, N, adj, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):                                   # predict step T from steps 0..T-2
        B, T, N = x.shape
        xi = x[:, :-1, :]
        feats = self.W(xi.unsqueeze(-1))
        prop = torch.relu(torch.einsum("ij,btjh->btih", self.A_hat, feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
        out, _ = self.rnn(seq)
        return self.head(out[:, -1, :]).reshape(B, N)       # (B,N)

    def loss(self, x):
        return F.mse_loss(self.forward(x), x[:, -1, :])

    def step_score(self, x):
        return (self.forward(x) - x[:, -1, :]).pow(2)       # (B,N) at last step


# =============================================================================
# TRAIN / SCORE / FLAG / EVAL
# =============================================================================
def train_model(model, train_loader, val_loader, mode, device,
                lr=1e-3, epochs=50, patience=10, pos_weight=None):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device) if pos_weight is not None else None)
    stop = EarlyStopping(patience=patience, mode="f1" if mode == "sup" else "loss")

    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(device)
            opt.zero_grad()
            loss = crit(model(x), y.to(device)) if mode == "sup" else model.loss(x)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            if mode == "sup":
                ps, ys = [], []
                for x, y in val_loader:
                    ps.append(torch.sigmoid(model(x.to(device))).cpu().numpy().reshape(-1))
                    ys.append(y.numpy().reshape(-1))
                ys = np.concatenate(ys)
                # Ranking metric (AUPRC): tracks learning even when no fixed 0.5
                # cut separates the classes. F1@0.5 stays 0 on rare-positive data
                # and would stop training immediately.
                metric = average_precision_score(ys, np.concatenate(ps)) if ys.any() else 0.0
            else:
                metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in val_loader]))
        if stop.step(metric, model):
            stopped = ep + 1
            break
    else:
        stopped = epochs
    stop.restore(model)
    if mode == "sup":
        print(f"   trained {stopped} epoch(s) | best val_AUPRC={stop.best:.4f}")
    else:
        model.eval()
        with torch.no_grad():
            train_metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in train_loader]))
        print(f"   trained {stopped} epoch(s) | train_loss={train_metric:.4f} | best {'val_AUPRC' if mode == 'sup' else 'val_loss'}={stop.best:.4f}")
    # print(f"   trained {stopped} epoch(s) | best {'val_AUPRC' if mode == 'sup' else 'val_loss'}={stop.best:.4f}")
    return model


def score_series(model, values, seq_len, stride, mode, device):
    """Return a (T, N) anomaly-score map, averaging overlapping windows."""
    model.eval()
    T, N = values.shape
    if mode == "forecast":
        stride = 1                                # score every incoming step
    acc = np.zeros((T, N)); cnt = np.zeros((T, N))
    starts = list(range(0, T - seq_len + 1, stride))
    if starts and starts[-1] != T - seq_len:
        starts.append(T - seq_len)
    with torch.no_grad():
        for s in starts:
            x = torch.tensor(values[s:s + seq_len], dtype=torch.float32)[None].to(device)
            if mode == "sup":
                sc = torch.sigmoid(model(x))[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            elif mode == "vae":
                sc = model.cell_score(x)[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            else:                                            # forecast: last step only
                t = s + seq_len - 1
                acc[t] += model.step_score(x)[0].cpu().numpy(); cnt[t] += 1
    cnt[cnt == 0] = 1
    return acc / cnt


def flag_supervised(val_scores, val_mask, test_scores, objective="cell", min_overlap=1):
    """Tune one global probability cut on val. objective='cell' maximises cell F1;
    'event' maximises point-adjust F1 (aligns the cut with run-level scoring)."""
    thr, best = float(np.median(val_scores)), -1.0
    for t in np.linspace(val_scores.min(), val_scores.max(), 100):
        vp = (val_scores >= t).astype(int)
        if objective == "event":
            f = event_metrics(vp, val_mask, min_overlap)["PA_F1"]
        else:
            f = f1_score(val_mask.reshape(-1), vp.reshape(-1).astype(int), zero_division=0)
        if f > best:
            best, thr = f, t
    return (test_scores >= thr).astype(int), float(thr)


def flag_unsupervised(clean_scores, test_scores, method="percentile", pct=99.0, k=4.0):
    """Per-sensor threshold (higher score = more anomalous).
    percentile : pct-th percentile of clean-TRAIN scores (assumes train ~= test).
    median     : median + k*MAD of the scored set itself (adapts to its distribution).
    mean       : mean + k*std of the scored set itself.
    Robust methods need the score to rank anomalies ABOVE normals (AUROC > 0.5)."""
    if method == "percentile":
        thr = np.percentile(clean_scores, pct, axis=0)
    elif method == "median":
        med = np.median(test_scores, axis=0)
        mad = np.median(np.abs(test_scores - med), axis=0)
        thr = med + k * (mad + 1e-9)
    elif method == "mean":
        thr = test_scores.mean(0) + k * (test_scores.std(0) + 1e-9)
    else:
        raise ValueError(f"unknown unsup_threshold '{method}'")
    return (test_scores >= thr).astype(int), thr


def cell_metrics(pred_mask, true_mask):
    p = pred_mask.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    return {"Accuracy":  round(accuracy_score(t, p), 4),
            "Precision": round(precision_score(t, p, zero_division=0), 4),
            "Recall":    round(recall_score(t, p, zero_division=0), 4),
            "F1":        round(f1_score(t, p, zero_division=0), 4)}


def ranking_metrics(scores, true_mask):
    """Threshold-free quality of the score map (does it rank anomalies high?)."""
    s = scores.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    if t.sum() == 0 or t.sum() == len(t):
        return {"AUROC": float("nan"), "AUPRC": float("nan")}
    return {"AUROC": round(roc_auc_score(t, s), 4),
            "AUPRC": round(average_precision_score(t, s), 4)}


def _segments(col):
    """Contiguous runs of 1s in a 1D bool array -> list of (start, end_exclusive)."""
    segs, i, n = [], 0, len(col)
    while i < n:
        if col[i]:
            j = i
            while j < n and col[j]:
                j += 1
            segs.append((i, j)); i = j
        else:
            i += 1
    return segs


def _filter_short_runs(mask, min_len):
    """Drop predicted runs shorter than min_len per sensor (removes isolated false
    flags; useful when real anomalies are sustained)."""
    if min_len <= 1:
        return mask
    out = np.zeros_like(mask)
    for s in range(mask.shape[1]):
        for a, b in _segments(mask[:, s].astype(bool)):
            if b - a >= min_len:
                out[a:b, s] = 1
    return out


def event_metrics(pred_mask, true_mask, min_overlap=1):
    """Score whole anomalous runs instead of individual cells (per sensor).
    point-adjust (PA): if >=min_overlap cells of a true run are flagged, the WHOLE
        run is credited; P/R/F1 then computed point-wise on the adjusted mask
        (standard time-series AD protocol).
    event-overlap (EV): a true run is a hit if it's flagged at all; a predicted run
        that touches no true run is a false alarm. P/R/F1 over runs."""
    pred = pred_mask.astype(bool); true = (np.asarray(true_mask) > 0)
    T, N = true.shape
    adj = pred.copy()
    tp_ev = fn_ev = fp_ev = pred_ev = 0
    for s in range(N):
        for a, b in _segments(true[:, s]):
            if pred[a:b, s].sum() >= min_overlap:
                adj[a:b, s] = True; tp_ev += 1
            else:
                fn_ev += 1
        for a, b in _segments(pred[:, s]):
            pred_ev += 1
            if not true[a:b, s].any():
                fp_ev += 1
    p, t = adj.reshape(-1), true.reshape(-1)
    ev_p = (pred_ev - fp_ev) / pred_ev if pred_ev else 0.0
    ev_r = tp_ev / (tp_ev + fn_ev) if (tp_ev + fn_ev) else 0.0
    ev_f1 = 2 * ev_p * ev_r / (ev_p + ev_r) if (ev_p + ev_r) else 0.0
    return {"PA_Precision": round(precision_score(t, p, zero_division=0), 4),
            "PA_Recall":    round(recall_score(t, p, zero_division=0), 4),
            "PA_F1":        round(f1_score(t, p, zero_division=0), 4),
            "EV_Recall":    round(ev_r, 4),
            "EV_F1":        round(ev_f1, 4),
            "Events":       tp_ev + fn_ev}


# =============================================================================
# CLEAN OUTPUT
# =============================================================================
def _sensor_meta(csv_path):
    try:
        df = pd.read_csv(csv_path)
        cols = [c for c in ("thing_name", "thing_ip") if c in df.columns]
        if not cols:
            return {}
        g = df.drop_duplicates("sensor_name").set_index("sensor_name")[cols]
        return {s: row.to_dict() for s, row in g.iterrows()}
    except Exception:
        return {}


def export_clean(corrupted, scaler, pred_mask, dt_index, sensors, meta, out_path):
    """Drop flagged (datetime, sensor) readings; write the rest as long-format CSV."""
    inv = scaler.inverse_transform(corrupted)
    rows = []
    for ti, dt in enumerate(dt_index):
        dt = pd.Timestamp(dt)
        for sj, sname in enumerate(sensors):
            if pred_mask[ti, sj]:
                continue
            m = meta.get(sname, {})
            rows.append({
                "datetime": dt, "date": dt.strftime("%Y-%m-%d"),
                "time": dt.strftime("%H:%M"), "seconds": dt.timestamp(),
                "state": float(inv[ti, sj]), "sensor_name": sname,
                "thing_name": m.get("thing_name"), "thing_ip": m.get("thing_ip"),
            })
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    return df


# =============================================================================
# REGISTRY & ORCHESTRATION
# =============================================================================
def build_registry(N, seq_len, adj, beta):
    return [
        {"name": "TemporalCNN",        "mode": "sup",      "graph": "none",
         "build": lambda: TemporalCNN(N, seq_len)},
        {"name": "StackedLSTM",        "mode": "sup",      "graph": "none",
         "build": lambda: StackedLSTM(N)},
        {"name": "GNN_LearnableGraph", "mode": "sup",      "graph": "learned",
         "build": lambda: GCNDetector(N, adj=None)},
        {"name": "GAT_LearnedGraph",   "mode": "sup",      "graph": "learned",
         "build": lambda: GATDetector(N)},
        {"name": "GraphAnomalyVAE",    "mode": "vae",      "graph": "learned",
         "build": lambda: GraphAnomalyVAE(N, seq_len, beta=beta)},
        {"name": "GNNForecaster",      "mode": "forecast", "graph": "predefined",
         "build": lambda: GNNForecaster(N, adj)},
    ]


def run(csv_path, adj_csv_path=None, groups=None,
        seq_len=24, stride=4, batch_size=64, max_epochs=50, lr=1e-3, patience=15,
        anomaly_frac=0.10, train_anomaly_frac=0.05, anomaly_type="all",
        binary_anomaly=False, run_len=1, binary_mode="flip",
        unsup_percentile=99.0, unsup_threshold="percentile", robust_k=4.0,
        event_min_overlap=1, min_event_len=1, sup_objective="cell",
        pos_weight_cap=10.0, train_frac=0.70, val_frac=0.15,
        out_dir="cleaned_output", models=None, beta=1.0, seed=42):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*64}\n  Sensor-level anomaly cleaning  |  device={device}\n{'='*64}")
    wide = load_and_pivot(csv_path)
    sensors = list(wide.columns); N = len(sensors)
    raw = wide.values.astype(np.float32); T = len(raw)
    n_tr, n_va = int(T * train_frac), int(T * val_frac)

    scaler = MinMaxScaler()
    tr = scaler.fit_transform(raw[:n_tr])
    va = scaler.transform(raw[n_tr:n_tr + n_va])
    te = scaler.transform(raw[n_tr + n_va:])
    dt_test = wide.index[n_tr + n_va:]

    if groups is not None:
        adj = adjacency_from_groups(groups, sensors)
    elif adj_csv_path is not None:
        _, adj = load_edge_index_from_csv(adj_csv_path, sensors)
    else:
        raise ValueError("provide either `groups` (dict) or `adj_csv_path`")

    print("\nInjecting anomalies (val / test, and a small fraction into train):")
    _inj = (lambda v, f: inject_binary(v, f, run_len, binary_mode, adj)) if binary_anomaly \
        else (lambda v, f: inject(v, f, anomaly_type, adj))
    va_c, va_m = _inj(va, anomaly_frac)
    te_c, te_m = _inj(te, anomaly_frac)
    tr_c, tr_m = _inj(tr, train_anomaly_frac)
    meta = _sensor_meta(csv_path)

    # data loaders
    zeros_tr = np.zeros_like(tr, dtype=int)
    sup_tr = DataLoader(WindowDS(tr_c, tr_m, seq_len, stride), batch_size=batch_size, shuffle=True)
    sup_va = DataLoader(WindowDS(va_c, va_m, seq_len, stride), batch_size=batch_size, shuffle=False)
    uns_tr = DataLoader(WindowDS(tr, zeros_tr, seq_len, stride), batch_size=batch_size, shuffle=True)
    uns_va = DataLoader(WindowDS(va_c, np.zeros_like(va, int), seq_len, stride), batch_size=batch_size, shuffle=False)

    pos = tr_m.sum(); neg = tr_m.size - pos
    pw = min(neg / max(pos, 1), pos_weight_cap)
    pos_weight = torch.tensor([pw], dtype=torch.float32)
    print(f"\nTrain cell anomaly rate: {100 * pos / tr_m.size:.2f}%  |  pos_weight={pw:.1f} "
          f"(capped at {pos_weight_cap})")

    registry = build_registry(N, seq_len, adj, beta)
    if models:
        registry = [m for m in registry if m["name"] in models]

    results = []
    for entry in registry:
        name, mode = entry["name"], entry["mode"]
        if mode == "sup" and train_anomaly_frac <= 0:
            print(f"\n[skip] {name}: supervised model needs train_anomaly_frac > 0")
            continue
        print(f"\n{'-'*64}\n  {name}  (mode={mode}, graph={entry['graph']})\n{'-'*64}")
        model = entry["build"]()

        if mode == "sup":
            model = train_model(model, sup_tr, sup_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience, pos_weight=pos_weight)
            val_scores = score_series(model, va_c, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_supervised(val_scores, va_m, test_scores,
                                        objective=sup_objective, min_overlap=event_min_overlap)
        else:
            model = train_model(model, uns_tr, uns_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience)
            clean_scores = score_series(model, tr, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_unsupervised(clean_scores, test_scores,
                                          method=unsup_threshold, pct=unsup_percentile, k=robust_k)
            thr_arr = thr if isinstance(thr, np.ndarray) else np.full(N, thr)
            fpr = ((test_scores > thr_arr) & (te_m == 0)).sum(0) / max((te_m == 0).sum(), 1)
            print(f"   clean-test FPR (mean across sensors): {fpr.mean():.4f}"
                  f"   | thresholds: {thr_arr}") 

        # fpr_per_sensor = (test_scores[te_m == 0] >= thr).mean(0)   # on truly clean test cells
        pred = _filter_short_runs(pred, min_event_len)
        m = cell_metrics(pred, te_m)              # metrics on flagged mask (before removal)
        m.update(ranking_metrics(test_scores, te_m))   # threshold-free diagnostics
        m.update(event_metrics(pred, te_m, event_min_overlap))   # run/segment-level
        out_path = os.path.join(out_dir, f"clean_{name}.csv")
        clean_df = export_clean(te_c, scaler, pred, dt_test, sensors, meta, out_path)

        removed = int(pred.sum())
        print(f"   F1={m['F1']:.4f}  Prec={m['Precision']:.4f}  Rec={m['Recall']:.4f}  "
              f"Acc={m['Accuracy']:.4f}  | removed {removed} readings -> {out_path}")
        m.update({"Model": name, "Mode": mode, "Removed": removed, "CleanFile": out_path})
        results.append(m)

    _print_table(results, te_m)
    return results


def _print_table(results, true_mask):
    inj = int((true_mask > 0).sum())
    cols = ["Model", "Mode", "AUPRC", "Cell_F1", "PA_F1", "EV_Recall", "EV_F1", "Removed"]
    key = {"Cell_F1": "F1"}                       # map display name -> result key
    rows = [[r.get(key.get(c, c)) for c in cols] for r in results]
    rows.sort(key=lambda r: (r[4] if r[4] == r[4] else -1), reverse=True)   # by PA_F1
    nev = results[0].get("Events", "?") if results else "?"
    print(f"\n{'='*78}\n  DETECTION  (injected cells: {inj}, anomalous runs/events: {nev})\n{'='*78}")
    if _TAB:
        print(tabulate(rows, headers=cols, tablefmt="fancy_grid", floatfmt=".4f"))
    else:
        print("  ".join(f"{c:<12}" for c in cols))
        for r in rows:
            print("  ".join(f"{str(v):<12}" for v in r))
    print("\nCell_F1  : exact per-(timestep,sensor) match (strictest).")
    print("PA_F1    : point-adjust - a run is credited if any of its cells is flagged.")
    print("EV_Recall/EV_F1 : per-run - did we catch each anomalous segment.")
    print("Event scoring is meaningful for sustained anomalies (run_len > 1).")


# =============================================================================
# ENTRY POINT  (terminal:  python anomaly_cleaning.py --csv ... --adj_csv ...)
#              (jupyter  :  edit the call below and %run the file)
# =============================================================================
# =============================================================================
# ENTRY POINT
#   notebook : edit SENSOR_GROUPS / PARAMS below and run the file (or import `run`)
#   terminal : python anomaly_cleaning.py --csv data.csv --groups_json groups.json \
#                     --binary_anomaly --binary_mode ood --run_len 5
# =============================================================================

# --- domain knowledge: edit this grouping (sensors sharing a group get connected) --
SENSOR_GROUPS = {
    'G1': ['Entrance Motion', 'Kitchen Motion', 'Motion Outside Room'], 
    'G2': ['Desk Left Sonar', 'Desk Right Sonar', 'Motion Outside Room'], 
    'G3': ['Kitchen Humidity', 'Kitchen Motion', 'Motion Outside Room'], 
    'G4': ['Desk Left Sonar', 'Desk Right Motion', 'Motion Outside Room'], 
    'G5': ['Desk Left Sonar', 'Motion Outside Room', 'Washroom Temperature'], 
    'G6': ['Desk Left Sonar', 'Motion Outside Room', 'Washroom Motion'], 
    'G7': ['Desk Left Sonar', 'Kitchen Humidity', 'Motion Outside Room'], 
    'G8': ['Desk Left Sonar', 'Motion Inside Room(East Corner)', 'Motion Outside Room'], 
    'G9': ['Desk Left Sonar', 'Kitchen Light', 'Motion Outside Room'], 
    'G10': ['Motion Inside Room(East Corner)', 'Motion Outside Room', 'Room Door'], 
    'G11': ['Closet Door', 'Motion Inside Room(East Corner)', 'Motion Outside Room'], 
    'G12': ['Entrance Motion', 'Motion Outside Room', 'Washroom Humidity'], 
    'G13': ['Motion Outside Room', 'Washroom Door', 'Washroom Humidity'], 
    'G14': ['Entrance Door', 'Kitchen Motion', 'Motion Outside Room'], 
    'G15': ['Desk Left Motion', 'Desk Left Sonar', 'Motion Outside Room'], 
    'G16': ['Desk Left Sonar', 'Kitchen Temperature', 'Motion Outside Room'], 
    'G17': ['Desk Left Sonar', 'Motion Outside Room', 'Washroom Humidity'], 
    'G18': ['Bedroom Humidity', 'Bedroom Light', 'Motion Inside Room(West Corner)'], 
    'G19': ['Bedroom Humidity', 'Bedroom Temperature', 'Motion Inside Room(West Corner)'], 
    'G20': ['Bedroom Humidity', 'Desk Left Light', 'Motion Inside Room(West Corner)'], 
    'G21': ['Closet Light']
}


if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        # ---- NOTEBOOK: all tunables in one place ----
        run(
            csv_path="Data/Home_Data/sead_anomaly_free_home_data_2024-02-25_to_2024-03-14.csv",
            groups=SENSOR_GROUPS,          # use the grouping above (set None to use adj_csv_path)
            adj_csv_path=None,             # e.g. "adjacency.csv" if not using groups
            # windowing / training
            seq_len=24, stride=4, batch_size=64, max_epochs=100, lr=1e-3, patience=20,
            # anomaly injection (binary data)
            binary_anomaly=True, binary_mode="ood", run_len=5,   # mode: flip|ood|stuck|neighbor|rate
            anomaly_frac=0.10, train_anomaly_frac=0.05,
            # supervised threshold objective + unsupervised threshold
            sup_objective="event",                               # "cell" or "event"
            unsup_threshold="mean", robust_k=1.5, unsup_percentile=99.0,
            # event scoring / misc
            event_min_overlap=1, min_event_len=1, beta=0.1,
            models=None,                                         # None = all; or e.g. ["TemporalCNN","GraphAnomalyVAE"]
            out_dir="cleaned_output",
        )
    else:
        # ---- TERMINAL ----
        import json
        p = argparse.ArgumentParser(description="Sensor-level anomaly cleaning")
        p.add_argument("--csv", required=True)
        p.add_argument("--adj_csv", default=None, help="labelled adjacency CSV (omit if using --groups_json)")
        p.add_argument("--groups_json", default=None, help="JSON file: {group: [sensor,...]}")
        p.add_argument("--seq_len", type=int, default=24)
        p.add_argument("--stride", type=int, default=4)
        p.add_argument("--batch_size", type=int, default=64)
        p.add_argument("--max_epochs", type=int, default=100)
        p.add_argument("--lr", type=float, default=1e-3)
        p.add_argument("--patience", type=int, default=20)
        p.add_argument("--anomaly_frac", type=float, default=0.10)
        p.add_argument("--train_anomaly_frac", type=float, default=0.08)
        p.add_argument("--anomaly_type", type=str, default="all")
        p.add_argument("--binary_anomaly", action="store_true")
        p.add_argument("--binary_mode", type=str, default="flip",
                       choices=["flip", "ood", "stuck", "neighbor", "rate"])
        p.add_argument("--run_len", type=int, default=5)
        p.add_argument("--sup_objective", type=str, default="cell", choices=["cell", "event"])
        p.add_argument("--unsup_threshold", type=str, default="percentile",
                       choices=["percentile", "median", "mean"])
        p.add_argument("--robust_k", type=float, default=4.0)
        p.add_argument("--unsup_percentile", type=float, default=99.0)
        p.add_argument("--min_event_len", type=int, default=1)
        p.add_argument("--beta", type=float, default=1.0)
        p.add_argument("--out_dir", type=str, default="cleaned_output")
        a = p.parse_args()
        groups = json.load(open(a.groups_json)) if a.groups_json else None
        run(csv_path=a.csv, adj_csv_path=a.adj_csv, groups=groups,
            seq_len=a.seq_len, stride=a.stride, batch_size=a.batch_size, max_epochs=a.max_epochs,
            lr=a.lr, patience=a.patience, anomaly_frac=a.anomaly_frac,
            train_anomaly_frac=a.train_anomaly_frac, anomaly_type=a.anomaly_type,
            binary_anomaly=a.binary_anomaly, run_len=a.run_len, binary_mode=a.binary_mode,
            sup_objective=a.sup_objective, unsup_threshold=a.unsup_threshold, robust_k=a.robust_k,
            unsup_percentile=a.unsup_percentile, min_event_len=a.min_event_len,
            beta=a.beta, out_dir=a.out_dir)


  Sensor-level anomaly cleaning  |  device=cuda
  Loaded 169813 timestamps x 23 sensors
   Sensors: ['Bedroom Humidity', 'Bedroom Temperature', 'Closet Door', 'Closet Light', 'Desk Left Light', 'Desk Left Motion', 'Desk Left Sonar', 'Desk Right Motion', 'Desk Right Sonar', 'Entrance Door', 'Entrance Motion', 'Kitchen Humidity', 'Kitchen Light', 'Kitchen Motion', 'Kitchen Temperature', 'Motion Inside Room(East Corner)', 'Motion Inside Room(West Corner)', 'Motion Outside Room', 'Room Door', 'Washroom Door', 'Washroom Humidity', 'Washroom Motion', 'Washroom Temperature']
   Built graph from 21 groups: 39 undirected edges | density 15.4%
   [groups] ignored 1 name(s) not in data: ['Bedroom Light']
   [groups] isolated sensors (in no group): ['Closet Light']

Injecting anomalies (val / test, and a small fraction into train):

Train cell anomaly rate: 4.88%  |  pos_weight=10.0 (capped at 10.0)

----------------------------------------------------------------
  TemporalCNN  (mode=sup, graph=

In [29]:
"""
anomaly_cleaning.py
Sensor-level anomaly detection, removal, and forecasting-based detection.

Reuses data loading / anomaly injection / training utilities from
anomlay_pipeline.py (keep both files in the same folder, or on PYTHONPATH).

Both paradigms flag anomalies at the (timestep, sensor) cell level:
  supervised   : per-cell classifier trained on injected labels
  unsupervised : reconstruction / forecasting error vs a per-sensor threshold

Flow: load -> scale -> inject (val/test, + small frac into train for supervised)
      -> train -> score every cell -> flag -> report metrics on the flagged mask
      -> remove flagged readings -> write clean long-format CSV.

Graph usage:
  predefined adjacency : SpatioTemporalGNN, GNNForecaster
  learned graph        : GNN_LearnableGraph, GAT_LearnedGraph, GraphAnomalyVAE
  no graph             : TemporalCNN, StackedLSTM
"""
import os, sys, argparse, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score)

warnings.filterwarnings("ignore")

# --- shared pieces from the original pipeline --------------------------------
try:
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)
except ModuleNotFoundError:
    _here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, _here)
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)

try:
    from tabulate import tabulate
    _TAB = True
except ImportError:
    _TAB = False


# =============================================================================
# DATA
# =============================================================================
def _norm_adj(adj_matrix):
    """Symmetric-normalised adjacency with self-loops: D^-1/2 (A+I) D^-1/2."""
    A = (torch.tensor(np.asarray(adj_matrix), dtype=torch.float32) > 0).float()
    A = A + torch.eye(A.shape[0])
    d = A.sum(1).clamp(min=1e-6).pow(-0.5)
    return d[:, None] * A * d[None, :]


def adjacency_from_groups(groups, sensors):
    """Build a symmetric 0/1 adjacency (N x N) from a {group_name: [sensor,...]} dict.
    Every pair of sensors that share a group is connected (each group is a clique).
    Names not present in `sensors` are ignored; sensors in no group stay isolated."""
    idx = {s: i for i, s in enumerate(sensors)}
    N = len(sensors)
    A = np.zeros((N, N), dtype=np.float32)
    missing = set()
    for members in groups.values():
        ids = [idx[s] for s in members if s in idx]
        missing |= {s for s in members if s not in idx}
        for a in ids:
            for b in ids:
                if a != b:
                    A[a, b] = 1.0
    iso = [sensors[i] for i in range(N) if A[i].sum() == 0]
    print(f"   Built graph from {len(groups)} groups: {int(A.sum())//2} undirected edges "
          f"| density {A.sum() / (N * (N - 1)):.1%}")
    if missing:
        print(f"   [groups] ignored {len(missing)} name(s) not in data: {sorted(missing)}")
    if iso:
        print(f"   [groups] isolated sensors (in no group): {iso}")
    return A


def inject(values, frac, atype, adj):
    """Inject anomalies and return (corrupted, cell_mask) where mask[t,s]=1 if
    that exact reading was altered. Mask is derived by diffing the clean array."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr, _ = inject_anomalies(values, frac, atype, "", adj)
    return corr, (corr != values).astype(int)


def inject_binary(values, frac, run_len=1, mode="flip", adj=None, ood_value=1.5, rate=0.7):
    """Binary anomaly injection. Each anomaly spans run_len consecutive steps on one
    sensor. mask marks only cells whose value actually changed. Modes:
      flip     : invert the bit(s) (in-distribution; needs predictable data to detect).
      ood      : set to an out-of-range value (sensor-fault / garbage reading; the
                 binary analogue of a spike - separable in feature space, easiest).
      stuck    : freeze at a constant 0 or 1 over the run (stuck-at fault).
      neighbor : force disagreement with a graph-neighbour (spatial anomaly; detectable
                 from cross-sensor structure even when the series isn't predictable).
      rate     : flip a 'rate' fraction of the run's steps (a burst / activation-rate
                 shift; a sustained statistical anomaly, more robust than one flip)."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr = values.copy()
    mask = np.zeros_like(values, dtype=int)
    T, N = values.shape
    n = max(1, int(T * N * frac / max(run_len, 1)))
    nbrs = None
    if mode == "neighbor" and adj is not None:
        A = np.asarray(adj) > 0
        nbrs = [np.where(A[s])[0] for s in range(N)]
    for _ in range(n):
        t = np.random.randint(0, T - run_len + 1)
        s = np.random.randint(0, N)
        sl = slice(t, t + run_len)
        if mode == "ood":
            corr[sl, s] = ood_value
        elif mode == "stuck":
            corr[sl, s] = float(np.random.randint(0, 2))
        elif mode == "neighbor" and nbrs is not None and len(nbrs[s]):
            corr[sl, s] = 1.0 - values[sl, np.random.choice(nbrs[s])]
        elif mode == "rate":
            pick = np.where(np.random.rand(run_len) < rate)[0] + t
            corr[pick, s] = 1.0 - corr[pick, s]
        else:                                            # flip
            corr[sl, s] = 1.0 - corr[sl, s]
        mask[sl, s] = (corr[sl, s] != values[sl, s]).astype(int)
    return corr, mask


class WindowDS(Dataset):
    """Sliding windows of (seq_len, N) with matching per-cell labels."""
    def __init__(self, values, cell_labels, seq_len, stride):
        self.x, self.y = [], []
        T = len(values)
        starts = list(range(0, T - seq_len + 1, stride))
        if starts and starts[-1] != T - seq_len:
            starts.append(T - seq_len)
        for s in starts:
            self.x.append(torch.tensor(values[s:s + seq_len], dtype=torch.float32))
            self.y.append(torch.tensor(cell_labels[s:s + seq_len], dtype=torch.float32))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]


# =============================================================================
# MODELS  (all output per-cell logits (B,T,N) for sup, or expose cell scores)
# =============================================================================
class TemporalCNN(nn.Module):
    def __init__(self, N, seq_len, hidden=32, levels=4, k=3):
        super().__init__()
        chs = [N] + [hidden] * levels
        self.tcn = nn.Sequential(*[TCNBlock(chs[i], chs[i + 1], k, 2 ** i) for i in range(levels)])
        self.head = nn.Conv1d(hidden, N, 1)

    def forward(self, x):                                   # (B,T,N)
        h = self.tcn(x.permute(0, 2, 1))                    # (B,hidden,T)
        return self.head(h).permute(0, 2, 1)                # (B,T,N)


class StackedLSTM(nn.Module):
    def __init__(self, N, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(N, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, N)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out)                               # (B,T,N)


class GCNDetector(nn.Module):
    """GCN + per-node GRU per-cell classifier. Fixed graph if adj given, else learned."""
    def __init__(self, N, adj=None, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N, self.learn = N, adj is None
        if self.learn:
            self.adj_logits = nn.Parameter(torch.randn(N, N))
        else:
            self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def _A(self, device):
        if not self.learn:
            return self.A_hat
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def forward(self, x):
        B, T, N = x.shape
        feats = self.W(x.unsqueeze(-1))                     # (B,T,N,H)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device), feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)                              # (B*N,T,rnn)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GATDetector(nn.Module):
    """Learned graph: per-timestep multi-head attention over sensors (GAT on a
    fully-connected graph), then per-node GRU, then per-cell head."""
    def __init__(self, N, hidden=16, heads=2, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.proj = nn.Linear(1, hidden)
        self.attn = nn.MultiheadAttention(hidden, heads, batch_first=True, dropout=0.1)
        self.rnn = nn.GRU(hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):
        B, T, N = x.shape
        f = self.proj(x.reshape(B * T, N, 1))               # (B*T,N,hidden)
        a, _ = self.attn(f, f, f)                           # attention = learned graph
        a = torch.relu(a).reshape(B, T, N, -1)
        seq = a.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GraphAnomalyVAE(nn.Module):
    """Unsupervised graph VAE (learned adjacency). Cell score = reconstruction MSE."""
    def __init__(self, N, seq_len, gcn_hidden=16, rnn_hidden=32, latent=16, beta=1.0):
        super().__init__()
        self.N, self.seq_len, self.beta = N, seq_len, beta
        self.adj_logits = nn.Parameter(torch.randn(N, N))
        self.enc_proj = nn.Linear(1, gcn_hidden, bias=False)
        self.enc_rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.fc_mu = nn.Linear(rnn_hidden, latent)
        self.fc_lv = nn.Linear(rnn_hidden, latent)
        self.dec_fc = nn.Linear(latent, rnn_hidden)
        self.dec_rnn = nn.GRU(rnn_hidden, rnn_hidden, batch_first=True)
        self.dec_W = nn.Linear(rnn_hidden, gcn_hidden, bias=False)
        self.dec_out = nn.Linear(gcn_hidden, 1)

    def _A(self, device):
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def encode(self, x):
        B, T, N = x.shape
        f = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device),
                                    self.enc_proj(x.unsqueeze(-1))))
        seq = f.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        _, h = self.enc_rnn(seq)
        h = h.squeeze(0).reshape(B, N, -1).mean(1)
        return self.fc_mu(h), self.fc_lv(h)

    def decode(self, z):
        h = self.dec_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.dec_rnn(h)
        node = self.dec_W(out).unsqueeze(2).repeat(1, 1, self.N, 1)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(z.device), node))
        return self.dec_out(prop).squeeze(-1)               # (B,T,N)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return self.decode(z), mu, lv

    def loss(self, x):
        recon, mu, lv = self.forward(x)
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        return F.mse_loss(recon, x) + self.beta * kl

    def cell_score(self, x):
        recon, _, _ = self.forward(x)
        return (x - recon).pow(2)                           # (B,T,N)


class GNNForecaster(nn.Module):
    """Unsupervised one-step forecaster on the predefined graph. Trained on clean
    data; per-sensor score = squared next-step prediction error."""
    def __init__(self, N, adj = None, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N, self.learn = N, adj is None
        if not adj:
            self.adj_logits = nn.Parameter(torch.randn(N, N))
        else:
            self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def _A(self, device):
        if not self.learn:
            return self.A_hat
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def forward(self, x):                                   # predict step T from steps 0..T-2
        B, T, N = x.shape
        xi = x[:, :-1, :]
        feats = self.W(xi.unsqueeze(-1))
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device), feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
        out, _ = self.rnn(seq)
        return self.head(out[:, -1, :]).reshape(B, N)       # (B,N)

    def loss(self, x):
        return F.mse_loss(self.forward(x), x[:, -1, :])

    def step_score(self, x):
        return (self.forward(x) - x[:, -1, :]).pow(2)       # (B,N) at last step


# =============================================================================
# TRAIN / SCORE / FLAG / EVAL
# =============================================================================
def train_model(model, train_loader, val_loader, mode, device,
                lr=1e-3, epochs=50, patience=10, pos_weight=None):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device) if pos_weight is not None else None)
    stop = EarlyStopping(patience=patience, mode="f1" if mode == "sup" else "loss")

    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(device)
            opt.zero_grad()
            loss = crit(model(x), y.to(device)) if mode == "sup" else model.loss(x)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            if mode == "sup":
                ps, ys = [], []
                for x, y in val_loader:
                    ps.append(torch.sigmoid(model(x.to(device))).cpu().numpy().reshape(-1))
                    ys.append(y.numpy().reshape(-1))
                ys = np.concatenate(ys)
                # Ranking metric (AUPRC): tracks learning even when no fixed 0.5
                # cut separates the classes. F1@0.5 stays 0 on rare-positive data
                # and would stop training immediately.
                metric = average_precision_score(ys, np.concatenate(ps)) if ys.any() else 0.0
            else:
                metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in val_loader]))
        if stop.step(metric, model):
            stopped = ep + 1
            break
    else:
        stopped = epochs
    stop.restore(model)
    if mode == "sup":
        print(f"   trained {stopped} epoch(s) | best val_AUPRC={stop.best:.4f}")
    else:
        model.eval()
        with torch.no_grad():
            train_metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in train_loader]))
        print(f"   trained {stopped} epoch(s) | train_loss={train_metric:.4f} | best {'val_AUPRC' if mode == 'sup' else 'val_loss'}={stop.best:.4f}")
    # print(f"   trained {stopped} epoch(s) | best {'val_AUPRC' if mode == 'sup' else 'val_loss'}={stop.best:.4f}")
    return model


def score_series(model, values, seq_len, stride, mode, device):
    """Return a (T, N) anomaly-score map, averaging overlapping windows."""
    model.eval()
    T, N = values.shape
    if mode == "forecast":
        stride = 1                                # score every incoming step
    acc = np.zeros((T, N)); cnt = np.zeros((T, N))
    starts = list(range(0, T - seq_len + 1, stride))
    if starts and starts[-1] != T - seq_len:
        starts.append(T - seq_len)
    with torch.no_grad():
        for s in starts:
            x = torch.tensor(values[s:s + seq_len], dtype=torch.float32)[None].to(device)
            if mode == "sup":
                sc = torch.sigmoid(model(x))[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            elif mode == "vae":
                sc = model.cell_score(x)[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            else:                                            # forecast: last step only
                t = s + seq_len - 1
                acc[t] += model.step_score(x)[0].cpu().numpy(); cnt[t] += 1
    cnt[cnt == 0] = 1
    return acc / cnt


def flag_supervised(val_scores, val_mask, test_scores, objective="cell", min_overlap=1):
    """Tune one global probability cut on val. objective='cell' maximises cell F1;
    'event' maximises point-adjust F1 (aligns the cut with run-level scoring)."""
    thr, best = float(np.median(val_scores)), -1.0
    for t in np.linspace(val_scores.min(), val_scores.max(), 100):
        vp = (val_scores >= t).astype(int)
        if objective == "event":
            f = event_metrics(vp, val_mask, min_overlap)["PA_F1"]
        else:
            f = f1_score(val_mask.reshape(-1), vp.reshape(-1).astype(int), zero_division=0)
        if f > best:
            best, thr = f, t
    return (test_scores >= thr).astype(int), float(thr)


def flag_unsupervised(clean_scores, test_scores, method="percentile", pct=99.0, k=4.0):
    """Per-sensor threshold (higher score = more anomalous).
    percentile : pct-th percentile of clean-TRAIN scores (assumes train ~= test).
    median     : median + k*MAD of the scored set itself (adapts to its distribution).
    mean       : mean + k*std of the scored set itself.
    Robust methods need the score to rank anomalies ABOVE normals (AUROC > 0.5)."""
    if method == "percentile":
        thr = np.percentile(clean_scores, pct, axis=0)
    elif method == "median":
        med = np.median(test_scores, axis=0)
        mad = np.median(np.abs(test_scores - med), axis=0)
        thr = med + k * (mad + 1e-9)
    elif method == "mean":
        thr = test_scores.mean(0) + k * (test_scores.std(0) + 1e-9)
    else:
        raise ValueError(f"unknown unsup_threshold '{method}'")
    return (test_scores >= thr).astype(int), thr


def cell_metrics(pred_mask, true_mask):
    p = pred_mask.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    return {"Accuracy":  round(accuracy_score(t, p), 4),
            "Precision": round(precision_score(t, p, zero_division=0), 4),
            "Recall":    round(recall_score(t, p, zero_division=0), 4),
            "F1":        round(f1_score(t, p, zero_division=0), 4)}


def ranking_metrics(scores, true_mask):
    """Threshold-free quality of the score map (does it rank anomalies high?)."""
    s = scores.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    if t.sum() == 0 or t.sum() == len(t):
        return {"AUROC": float("nan"), "AUPRC": float("nan")}
    return {"AUROC": round(roc_auc_score(t, s), 4),
            "AUPRC": round(average_precision_score(t, s), 4)}


def _segments(col):
    """Contiguous runs of 1s in a 1D bool array -> list of (start, end_exclusive)."""
    segs, i, n = [], 0, len(col)
    while i < n:
        if col[i]:
            j = i
            while j < n and col[j]:
                j += 1
            segs.append((i, j)); i = j
        else:
            i += 1
    return segs


def _filter_short_runs(mask, min_len):
    """Drop predicted runs shorter than min_len per sensor (removes isolated false
    flags; useful when real anomalies are sustained)."""
    if min_len <= 1:
        return mask
    out = np.zeros_like(mask)
    for s in range(mask.shape[1]):
        for a, b in _segments(mask[:, s].astype(bool)):
            if b - a >= min_len:
                out[a:b, s] = 1
    return out


def event_metrics(pred_mask, true_mask, min_overlap=1):
    """Score whole anomalous runs instead of individual cells (per sensor).
    point-adjust (PA): if >=min_overlap cells of a true run are flagged, the WHOLE
        run is credited; P/R/F1 then computed point-wise on the adjusted mask
        (standard time-series AD protocol).
    event-overlap (EV): a true run is a hit if it's flagged at all; a predicted run
        that touches no true run is a false alarm. P/R/F1 over runs."""
    pred = pred_mask.astype(bool); true = (np.asarray(true_mask) > 0)
    T, N = true.shape
    adj = pred.copy()
    tp_ev = fn_ev = fp_ev = pred_ev = 0
    for s in range(N):
        for a, b in _segments(true[:, s]):
            if pred[a:b, s].sum() >= min_overlap:
                adj[a:b, s] = True; tp_ev += 1
            else:
                fn_ev += 1
        for a, b in _segments(pred[:, s]):
            pred_ev += 1
            if not true[a:b, s].any():
                fp_ev += 1
    p, t = adj.reshape(-1), true.reshape(-1)
    ev_p = (pred_ev - fp_ev) / pred_ev if pred_ev else 0.0
    ev_r = tp_ev / (tp_ev + fn_ev) if (tp_ev + fn_ev) else 0.0
    ev_f1 = 2 * ev_p * ev_r / (ev_p + ev_r) if (ev_p + ev_r) else 0.0
    return {"PA_Precision": round(precision_score(t, p, zero_division=0), 4),
            "PA_Recall":    round(recall_score(t, p, zero_division=0), 4),
            "PA_F1":        round(f1_score(t, p, zero_division=0), 4),
            "EV_Recall":    round(ev_r, 4),
            "EV_F1":        round(ev_f1, 4),
            "Events":       tp_ev + fn_ev}


# =============================================================================
# CLEAN OUTPUT
# =============================================================================
def _sensor_meta(csv_path):
    try:
        df = pd.read_csv(csv_path)
        cols = [c for c in ("thing_name", "thing_ip") if c in df.columns]
        if not cols:
            return {}
        g = df.drop_duplicates("sensor_name").set_index("sensor_name")[cols]
        return {s: row.to_dict() for s, row in g.iterrows()}
    except Exception:
        return {}


def export_clean(corrupted, scaler, pred_mask, dt_index, sensors, meta, out_path):
    """Drop flagged (datetime, sensor) readings; write the rest as long-format CSV."""
    inv = scaler.inverse_transform(corrupted)
    rows = []
    for ti, dt in enumerate(dt_index):
        dt = pd.Timestamp(dt)
        for sj, sname in enumerate(sensors):
            if pred_mask[ti, sj]:
                continue
            m = meta.get(sname, {})
            rows.append({
                "datetime": dt, "date": dt.strftime("%Y-%m-%d"),
                "time": dt.strftime("%H:%M"), "seconds": dt.timestamp(),
                "state": float(inv[ti, sj]), "sensor_name": sname,
                "thing_name": m.get("thing_name"), "thing_ip": m.get("thing_ip"),
            })
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    return df


# =============================================================================
# REGISTRY & ORCHESTRATION
# =============================================================================
def build_registry(N, seq_len, adj, beta):
    return [
        # {"name": "TemporalCNN",        "mode": "sup",      "graph": "none",
        #  "build": lambda: TemporalCNN(N, seq_len)},
        # {"name": "StackedLSTM",        "mode": "sup",      "graph": "none",
        #  "build": lambda: StackedLSTM(N)},
        # {"name": "GNN_LearnableGraph", "mode": "sup",      "graph": "learned",
        #  "build": lambda: GCNDetector(N, adj=None)},
        #  {"name": "GCN_PredefinedGraph", "mode": "sup",      "graph": "predefined",
        #  "build": lambda: GCNDetector(N, adj=adj)},
        # {"name": "GAT_LearnedGraph",   "mode": "sup",      "graph": "learned",
        #  "build": lambda: GATDetector(N)},
        # {"name": "GraphAnomalyVAE",    "mode": "vae",      "graph": "learned",
        #  "build": lambda: GraphAnomalyVAE(N, seq_len, beta=beta)},
        {"name": "GNNForecaster",      "mode": "forecast", "graph": "predefined",
         "build": lambda: GNNForecaster(N, adj=None)},
    ]


def run(csv_path, adj_csv_path=None, groups=None,
        seq_len=24, stride=4, batch_size=64, max_epochs=50, lr=1e-3, patience=15,
        anomaly_frac=0.10, train_anomaly_frac=0.05, anomaly_type="all",
        binary_anomaly=False, run_len=1, binary_mode="flip",
        unsup_percentile=99.0, unsup_threshold="percentile", robust_k=4.0,
        event_min_overlap=1, min_event_len=1, sup_objective="cell",
        pos_weight_cap=10.0, train_frac=0.70, val_frac=0.15,
        out_dir="cleaned_output", models=None, beta=1.0, seed=42):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*64}\n  Sensor-level anomaly cleaning  |  device={device}\n{'='*64}")
    wide = load_and_pivot(csv_path)
    sensors = list(wide.columns); N = len(sensors)
    raw = wide.values.astype(np.float32); T = len(raw)
    n_tr, n_va = int(T * train_frac), int(T * val_frac)

    scaler = MinMaxScaler()
    tr = scaler.fit_transform(raw[:n_tr])
    va = scaler.transform(raw[n_tr:n_tr + n_va])
    te = scaler.transform(raw[n_tr + n_va:])
    dt_test = wide.index[n_tr + n_va:]

    if groups is not None:
        adj = adjacency_from_groups(groups, sensors)
    elif adj_csv_path is not None:
        _, adj = load_edge_index_from_csv(adj_csv_path, sensors)
    else:
        raise ValueError("provide either `groups` (dict) or `adj_csv_path`")

    print("\nInjecting anomalies (val / test, and a small fraction into train):")
    _inj = (lambda v, f: inject_binary(v, f, run_len, binary_mode, adj)) if binary_anomaly \
        else (lambda v, f: inject(v, f, anomaly_type, adj))
    va_c, va_m = _inj(va, anomaly_frac)
    te_c, te_m = _inj(te, anomaly_frac)
    tr_c, tr_m = _inj(tr, train_anomaly_frac)
    meta = _sensor_meta(csv_path)

    # data loaders
    zeros_tr = np.zeros_like(tr, dtype=int)
    sup_tr = DataLoader(WindowDS(tr_c, tr_m, seq_len, stride), batch_size=batch_size, shuffle=True)
    sup_va = DataLoader(WindowDS(va_c, va_m, seq_len, stride), batch_size=batch_size, shuffle=False)
    uns_tr = DataLoader(WindowDS(tr, zeros_tr, seq_len, stride), batch_size=batch_size, shuffle=True)
    uns_va = DataLoader(WindowDS(va_c, np.zeros_like(va, int), seq_len, stride), batch_size=batch_size, shuffle=False)

    pos = tr_m.sum(); neg = tr_m.size - pos
    pw = min(neg / max(pos, 1), pos_weight_cap)
    pos_weight = torch.tensor([pw], dtype=torch.float32)
    print(f"\nTrain cell anomaly rate: {100 * pos / tr_m.size:.2f}%  |  pos_weight={pw:.1f} "
          f"(capped at {pos_weight_cap})")

    registry = build_registry(N, seq_len, adj, beta)
    if models:
        registry = [m for m in registry if m["name"] in models]

    results = []
    for entry in registry:
        name, mode = entry["name"], entry["mode"]
        if mode == "sup" and train_anomaly_frac <= 0:
            print(f"\n[skip] {name}: supervised model needs train_anomaly_frac > 0")
            continue
        print(f"\n{'-'*64}\n  {name}  (mode={mode}, graph={entry['graph']})\n{'-'*64}")
        model = entry["build"]()

        if mode == "sup":
            model = train_model(model, sup_tr, sup_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience, pos_weight=pos_weight)
            val_scores = score_series(model, va_c, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_supervised(val_scores, va_m, test_scores,
                                        objective=sup_objective, min_overlap=event_min_overlap)
        else:
            model = train_model(model, uns_tr, uns_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience)
            clean_scores = score_series(model, tr, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_unsupervised(clean_scores, test_scores,
                                          method=unsup_threshold, pct=unsup_percentile, k=robust_k)
            thr_arr = thr if isinstance(thr, np.ndarray) else np.full(N, thr)
            fpr = ((test_scores > thr_arr) & (te_m == 0)).sum(0) / max((te_m == 0).sum(), 1)
            print(f"   clean-test FPR (mean across sensors): {fpr.mean():.4f}")

        pred = _filter_short_runs(pred, min_event_len)
        m = cell_metrics(pred, te_m)              # metrics on flagged mask (before removal)
        m.update(ranking_metrics(test_scores, te_m))   # threshold-free diagnostics
        m.update(event_metrics(pred, te_m, event_min_overlap))   # run/segment-level
        out_path = os.path.join(out_dir, f"clean_{name}.csv")
        clean_df = export_clean(te_c, scaler, pred, dt_test, sensors, meta, out_path)

        removed = int(pred.sum())
        print(f"   F1={m['F1']:.4f}  Prec={m['Precision']:.4f}  Rec={m['Recall']:.4f}  "
              f"Acc={m['Accuracy']:.4f}  | removed {removed} readings -> {out_path}")
        m.update({"Model": name, "Mode": mode, "Removed": removed, "CleanFile": out_path})
        results.append(m)

    _print_table(results, te_m)
    return results


def _print_table(results, true_mask):
    inj = int((true_mask > 0).sum())
    cols = ["Model", "Mode", "AUPRC", "Cell_F1", "PA_F1", "EV_Recall", "EV_F1", "Removed"]
    key = {"Cell_F1": "F1"}                       # map display name -> result key
    rows = [[r.get(key.get(c, c)) for c in cols] for r in results]
    rows.sort(key=lambda r: (r[4] if r[4] == r[4] else -1), reverse=True)   # by PA_F1
    nev = results[0].get("Events", "?") if results else "?"
    print(f"\n{'='*78}\n  DETECTION  (injected cells: {inj}, anomalous runs/events: {nev})\n{'='*78}")
    if _TAB:
        print(tabulate(rows, headers=cols, tablefmt="fancy_grid", floatfmt=".4f"))
    else:
        print("  ".join(f"{c:<12}" for c in cols))
        for r in rows:
            print("  ".join(f"{str(v):<12}" for v in r))
    print("\nCell_F1  : exact per-(timestep,sensor) match (strictest).")
    print("PA_F1    : point-adjust - a run is credited if any of its cells is flagged.")
    print("EV_Recall/EV_F1 : per-run - did we catch each anomalous segment.")
    print("Event scoring is meaningful for sustained anomalies (run_len > 1).")


# =============================================================================
# ENTRY POINT  (terminal:  python anomaly_cleaning.py --csv ... --adj_csv ...)
#              (jupyter  :  edit the call below and %run the file)
# =============================================================================
# =============================================================================
# ENTRY POINT
#   notebook : edit SENSOR_GROUPS / PARAMS below and run the file (or import `run`)
#   terminal : python anomaly_cleaning.py --csv data.csv --groups_json groups.json \
#                     --binary_anomaly --binary_mode ood --run_len 5
# =============================================================================

# --- domain knowledge: edit this grouping (sensors sharing a group get connected) --
SENSOR_GROUPS = {
    'G1': ['D001', 'D002', 'LS010', 'BATP010', 'M010', 'LS001', 'BATP001', 'M001', 'LS005', 'BATP005', 'M005', 'MA013', 'LS013', 'BATP013'], 
    'G2': ['T102', 'BATP102', 'T103', 'BATP103', 'T101', 'MA016', 'M009', 'LS011', 'BATP011', 'LS008', 'BATP008', 'M008', 'LS009', 'BATP009', 'M011', 'LS012', 'BATP012', 'T105', 'T104', 'BATP007', 'M012', 'MA015', 'LS015', 'BATP015', 'MA014', 'LS014', 'BATP014'], 
    'G3': ['D003', 'LS002', 'BATP002', 'LS016', 'BATP016', 'M002', 'LS004', 'BATP004', 'M004'],
    'G4': ['M007', 'LS007'],
    'G5': ['LS003', 'BATP003', 'M003', 'LS006', 'BATP006', 'M006']
}


if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        # ---- NOTEBOOK: all tunables in one place ----
        run(
            csv_path="Data/Casas_Data/sead_anomaly_free_casas_data_2012-07-18_to_2012-08-18.csv",
            groups=SENSOR_GROUPS,          # use the grouping above (set None to use adj_csv_path)
            adj_csv_path=None,             # e.g. "adjacency.csv" if not using groups
            # windowing / training
            seq_len=24, stride=4, batch_size=64, max_epochs=100, lr=1e-3, patience=20,
            # anomaly injection (binary data)
            binary_anomaly=True, binary_mode="neighbor", run_len=5,   # mode: flip|ood|stuck|neighbor|rate
            anomaly_frac=0.10, train_anomaly_frac=0.08,
            # supervised threshold objective + unsupervised threshold
            sup_objective="event",                               # "cell" or "event"
            unsup_threshold="mean", robust_k=1.5, unsup_percentile=99.0,
            # event scoring / misc
            event_min_overlap=1, min_event_len=1, beta=0.1,
            models=None,                                         # None = all; or e.g. ["TemporalCNN","GraphAnomalyVAE"]
            out_dir="cleaned_output",
        )
    else:
        # ---- TERMINAL ----
        import json
        p = argparse.ArgumentParser(description="Sensor-level anomaly cleaning")
        p.add_argument("--csv", required=True)
        p.add_argument("--adj_csv", default=None, help="labelled adjacency CSV (omit if using --groups_json)")
        p.add_argument("--groups_json", default=None, help="JSON file: {group: [sensor,...]}")
        p.add_argument("--seq_len", type=int, default=24)
        p.add_argument("--stride", type=int, default=4)
        p.add_argument("--batch_size", type=int, default=64)
        p.add_argument("--max_epochs", type=int, default=100)
        p.add_argument("--lr", type=float, default=1e-3)
        p.add_argument("--patience", type=int, default=20)
        p.add_argument("--anomaly_frac", type=float, default=0.10)
        p.add_argument("--train_anomaly_frac", type=float, default=0.08)
        p.add_argument("--anomaly_type", type=str, default="all")
        p.add_argument("--binary_anomaly", action="store_true")
        p.add_argument("--binary_mode", type=str, default="flip",
                       choices=["flip", "ood", "stuck", "neighbor", "rate"])
        p.add_argument("--run_len", type=int, default=5)
        p.add_argument("--sup_objective", type=str, default="cell", choices=["cell", "event"])
        p.add_argument("--unsup_threshold", type=str, default="percentile",
                       choices=["percentile", "median", "mean"])
        p.add_argument("--robust_k", type=float, default=4.0)
        p.add_argument("--unsup_percentile", type=float, default=99.0)
        p.add_argument("--min_event_len", type=int, default=1)
        p.add_argument("--beta", type=float, default=1.0)
        p.add_argument("--out_dir", type=str, default="cleaned_output")
        a = p.parse_args()
        groups = json.load(open(a.groups_json)) if a.groups_json else None
        run(csv_path=a.csv, adj_csv_path=a.adj_csv, groups=groups,
            seq_len=a.seq_len, stride=a.stride, batch_size=a.batch_size, max_epochs=a.max_epochs,
            lr=a.lr, patience=a.patience, anomaly_frac=a.anomaly_frac,
            train_anomaly_frac=a.train_anomaly_frac, anomaly_type=a.anomaly_type,
            binary_anomaly=a.binary_anomaly, run_len=a.run_len, binary_mode=a.binary_mode,
            sup_objective=a.sup_objective, unsup_threshold=a.unsup_threshold, robust_k=a.robust_k,
            unsup_percentile=a.unsup_percentile, min_event_len=a.min_event_len,
            beta=a.beta, out_dir=a.out_dir)


  Sensor-level anomaly cleaning  |  device=cuda
  Loaded 78870 timestamps x 58 sensors
   Sensors: ['BATP001', 'BATP002', 'BATP003', 'BATP004', 'BATP005', 'BATP006', 'BATP007', 'BATP008', 'BATP009', 'BATP010', 'BATP011', 'BATP012', 'BATP013', 'BATP014', 'BATP015', 'BATP016', 'BATP102', 'BATP103', 'D001', 'D002', 'D003', 'LS001', 'LS002', 'LS003', 'LS004', 'LS005', 'LS006', 'LS007', 'LS008', 'LS009', 'LS010', 'LS011', 'LS012', 'LS013', 'LS014', 'LS015', 'LS016', 'M001', 'M002', 'M003', 'M004', 'M005', 'M006', 'M007', 'M008', 'M009', 'M010', 'M011', 'M012', 'MA013', 'MA014', 'MA015', 'MA016', 'T101', 'T102', 'T103', 'T104', 'T105']
   Built graph from 5 groups: 494 undirected edges | density 29.9%

Injecting anomalies (val / test, and a small fraction into train):

Train cell anomaly rate: 4.25%  |  pos_weight=10.0 (capped at 10.0)

----------------------------------------------------------------
  GNNForecaster  (mode=forecast, graph=predefined)
----------------------------------------

In [3]:
"""
anomaly_cleaning.py
Sensor-level anomaly detection, removal, and forecasting-based detection.

Reuses data loading / anomaly injection / training utilities from
anomlay_pipeline.py (keep both files in the same folder, or on PYTHONPATH).

Both paradigms flag anomalies at the (timestep, sensor) cell level:
  supervised   : per-cell classifier trained on injected labels
  unsupervised : reconstruction / forecasting error vs a per-sensor threshold

Flow: load -> scale -> inject (val/test, + small frac into train for supervised)
      -> train -> score every cell -> flag -> report metrics on the flagged mask
      -> remove flagged readings -> write clean long-format CSV.

Graph usage:
  predefined adjacency : SpatioTemporalGNN, GNNForecaster
  learned graph        : GNN_LearnableGraph, GAT_LearnedGraph, GraphAnomalyVAE
  no graph             : TemporalCNN, StackedLSTM
"""
import os, sys, argparse, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score)

warnings.filterwarnings("ignore")

# --- shared pieces from the original pipeline --------------------------------
try:
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)
except ModuleNotFoundError:
    _here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, _here)
    from anomlay_pipeline import (load_and_pivot, load_edge_index_from_csv,
                                  inject_anomalies, TCNBlock, EarlyStopping)

try:
    from tabulate import tabulate
    _TAB = True
except ImportError:
    _TAB = False


# =============================================================================
# DATA
# =============================================================================
def _norm_adj(adj_matrix):
    """Symmetric-normalised adjacency with self-loops: D^-1/2 (A+I) D^-1/2."""
    A = (torch.tensor(np.asarray(adj_matrix), dtype=torch.float32) > 0).float()
    A = A + torch.eye(A.shape[0])
    d = A.sum(1).clamp(min=1e-6).pow(-0.5)
    return d[:, None] * A * d[None, :]


def adjacency_from_groups(groups, sensors):
    """Build a symmetric 0/1 adjacency (N x N) from a {group_name: [sensor,...]} dict.
    Every pair of sensors that share a group is connected (each group is a clique).
    Names not present in `sensors` are ignored; sensors in no group stay isolated."""
    idx = {s: i for i, s in enumerate(sensors)}
    N = len(sensors)
    A = np.zeros((N, N), dtype=np.float32)
    missing = set()
    for members in groups.values():
        ids = [idx[s] for s in members if s in idx]
        missing |= {s for s in members if s not in idx}
        for a in ids:
            for b in ids:
                if a != b:
                    A[a, b] = 1.0
    iso = [sensors[i] for i in range(N) if A[i].sum() == 0]
    print(f"   Built graph from {len(groups)} groups: {int(A.sum())//2} undirected edges "
          f"| density {A.sum() / (N * (N - 1)):.1%}")
    if missing:
        print(f"   [groups] ignored {len(missing)} name(s) not in data: {sorted(missing)}")
    if iso:
        print(f"   [groups] isolated sensors (in no group): {iso}")
    return A


def inject(values, frac, atype, adj):
    """Inject anomalies and return (corrupted, cell_mask) where mask[t,s]=1 if
    that exact reading was altered. Mask is derived by diffing the clean array."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr, _ = inject_anomalies(values, frac, atype, "", adj)
    return corr, (corr != values).astype(int)


def inject_binary(values, frac, run_len=1, mode="flip", adj=None, ood_value=1.5, rate=0.7):
    """Binary anomaly injection. Each anomaly spans run_len consecutive steps on one
    sensor. mask marks only cells whose value actually changed. Modes:
      flip     : invert the bit(s) (in-distribution; needs predictable data to detect).
      ood      : set to an out-of-range value (sensor-fault / garbage reading; the
                 binary analogue of a spike - separable in feature space, easiest).
      stuck    : freeze at a constant 0 or 1 over the run (stuck-at fault).
      neighbor : force disagreement with a graph-neighbour (spatial anomaly; detectable
                 from cross-sensor structure even when the series isn't predictable).
      rate     : flip a 'rate' fraction of the run's steps (a burst / activation-rate
                 shift; a sustained statistical anomaly, more robust than one flip).
      contextual : force a sensor to contradict its group's majority vote over the run.
                 Marginally plausible, jointly wrong -> only graph-aware models can catch it.
      swap     : swap two grouped sensors' values over the run. Each marginal is preserved
                 exactly; only the joint pattern is violated (pure graph-detectable test)."""
    if frac <= 0:
        return values.copy(), np.zeros_like(values, dtype=int)
    corr = values.copy()
    mask = np.zeros_like(values, dtype=int)
    T, N = values.shape
    n = max(1, int(T * N * frac / max(run_len, 1)))
    nbrs = None
    if mode in ("neighbor", "contextual", "swap") and adj is not None:
        A = np.asarray(adj) > 0
        nbrs = [np.where(A[s])[0] for s in range(N)]
    for _ in range(n):
        t = np.random.randint(0, T - run_len + 1)
        s = np.random.randint(0, N)
        sl = slice(t, t + run_len)
        if mode == "ood":
            corr[sl, s] = ood_value
        elif mode == "stuck":
            corr[sl, s] = float(np.random.randint(0, 2))
        elif mode == "neighbor" and nbrs is not None and len(nbrs[s]):
            corr[sl, s] = 1.0 - values[sl, np.random.choice(nbrs[s])]
        elif mode == "contextual" and nbrs is not None and len(nbrs[s]):
            grp = nbrs[s]
            majority = (values[sl][:, grp].mean(axis=1) >= 0.5).astype(float)
            corr[sl, s] = 1.0 - majority                 # contradict the group consensus
        elif mode == "swap" and nbrs is not None and len(nbrs[s]):
            j = int(np.random.choice(nbrs[s]))
            corr[sl, s], corr[sl, j] = values[sl, j].copy(), values[sl, s].copy()
            mask[sl, j] = (corr[sl, j] != values[sl, j]).astype(int)
        elif mode == "rate":
            pick = np.where(np.random.rand(run_len) < rate)[0] + t
            corr[pick, s] = 1.0 - corr[pick, s]
        else:                                            # flip
            corr[sl, s] = 1.0 - corr[sl, s]
        mask[sl, s] = np.maximum(mask[sl, s], (corr[sl, s] != values[sl, s]).astype(int))
    return corr, mask


class WindowDS(Dataset):
    """Sliding windows of (seq_len, N) with matching per-cell labels."""
    def __init__(self, values, cell_labels, seq_len, stride):
        self.x, self.y = [], []
        T = len(values)
        starts = list(range(0, T - seq_len + 1, stride))
        if starts and starts[-1] != T - seq_len:
            starts.append(T - seq_len)
        for s in starts:
            self.x.append(torch.tensor(values[s:s + seq_len], dtype=torch.float32))
            self.y.append(torch.tensor(cell_labels[s:s + seq_len], dtype=torch.float32))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]


# =============================================================================
# MODELS  (all output per-cell logits (B,T,N) for sup, or expose cell scores)
# =============================================================================
class TemporalCNN(nn.Module):
    def __init__(self, N, seq_len, hidden=32, levels=4, k=3):
        super().__init__()
        chs = [N] + [hidden] * levels
        self.tcn = nn.Sequential(*[TCNBlock(chs[i], chs[i + 1], k, 2 ** i) for i in range(levels)])
        self.head = nn.Conv1d(hidden, N, 1)

    def forward(self, x):                                   # (B,T,N)
        h = self.tcn(x.permute(0, 2, 1))                    # (B,hidden,T)
        return self.head(h).permute(0, 2, 1)                # (B,T,N)


class StackedLSTM(nn.Module):
    def __init__(self, N, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(N, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, N)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out)                               # (B,T,N)


class UnivariateBaseline(nn.Module):
    """Per-sensor temporal detector with NO cross-sensor information (one GRU shared
    across sensors, each sensor processed independently). Control: anomalies that are
    only detectable via OTHER sensors (contextual / swap) should stay invisible here,
    while the multivariate/graph models catch them - that gap is the evidence the graph
    helps."""
    def __init__(self, N, hidden=32, layers=1):
        super().__init__()
        self.gru = nn.GRU(1, hidden, layers, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):                                   # (B,T,N)
        B, T, N = x.shape
        seq = x.permute(0, 2, 1).reshape(B * N, T, 1)       # each sensor on its own
        out, _ = self.gru(seq)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GCNDetector(nn.Module):
    """GCN + per-node GRU per-cell classifier. Fixed graph if adj given, else learned."""
    def __init__(self, N, adj=None, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N, self.learn = N, adj is None
        if self.learn:
            self.adj_logits = nn.Parameter(torch.randn(N, N))
        else:
            self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def _A(self, device):
        if not self.learn:
            return self.A_hat
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def forward(self, x):
        B, T, N = x.shape
        feats = self.W(x.unsqueeze(-1))                     # (B,T,N,H)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device), feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)                              # (B*N,T,rnn)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GATDetector(nn.Module):
    """Learned graph: per-timestep multi-head attention over sensors (GAT on a
    fully-connected graph), then per-node GRU, then per-cell head."""
    def __init__(self, N, hidden=16, heads=2, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.proj = nn.Linear(1, hidden)
        self.attn = nn.MultiheadAttention(hidden, heads, batch_first=True, dropout=0.1)
        self.rnn = nn.GRU(hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):
        B, T, N = x.shape
        f = self.proj(x.reshape(B * T, N, 1))               # (B*T,N,hidden)
        a, _ = self.attn(f, f, f)                           # attention = learned graph
        a = torch.relu(a).reshape(B, T, N, -1)
        seq = a.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.rnn(seq)
        return self.head(out).reshape(B, N, T).permute(0, 2, 1)   # (B,T,N)


class GraphAnomalyVAE(nn.Module):
    """Unsupervised graph VAE (learned adjacency). Cell score = reconstruction MSE."""
    def __init__(self, N, seq_len, gcn_hidden=16, rnn_hidden=32, latent=16, beta=1.0):
        super().__init__()
        self.N, self.seq_len, self.beta = N, seq_len, beta
        self.adj_logits = nn.Parameter(torch.randn(N, N))
        self.enc_proj = nn.Linear(1, gcn_hidden, bias=False)
        self.enc_rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.fc_mu = nn.Linear(rnn_hidden, latent)
        self.fc_lv = nn.Linear(rnn_hidden, latent)
        self.dec_fc = nn.Linear(latent, rnn_hidden)
        self.dec_rnn = nn.GRU(rnn_hidden, rnn_hidden, batch_first=True)
        self.dec_W = nn.Linear(rnn_hidden, gcn_hidden, bias=False)
        self.dec_out = nn.Linear(gcn_hidden, 1)

    def _A(self, device):
        a = torch.softmax(self.adj_logits, dim=1)
        return a * (1 - torch.eye(self.N, device=device))

    def encode(self, x):
        B, T, N = x.shape
        f = torch.relu(torch.einsum("ij,btjh->btih", self._A(x.device),
                                    self.enc_proj(x.unsqueeze(-1))))
        seq = f.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        _, h = self.enc_rnn(seq)
        h = h.squeeze(0).reshape(B, N, -1).mean(1)
        return self.fc_mu(h), self.fc_lv(h)

    def decode(self, z):
        h = self.dec_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.dec_rnn(h)
        node = self.dec_W(out).unsqueeze(2).repeat(1, 1, self.N, 1)
        prop = torch.relu(torch.einsum("ij,btjh->btih", self._A(z.device), node))
        return self.dec_out(prop).squeeze(-1)               # (B,T,N)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return self.decode(z), mu, lv

    def loss(self, x):
        recon, mu, lv = self.forward(x)
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        return F.mse_loss(recon, x) + self.beta * kl

    def cell_score(self, x):
        recon, _, _ = self.forward(x)
        return (x - recon).pow(2)                           # (B,T,N)


class GNNForecaster(nn.Module):
    """Unsupervised one-step forecaster on the predefined graph. Trained on clean
    data; per-sensor score = squared next-step prediction error."""
    def __init__(self, N, adj, gcn_hidden=16, rnn_hidden=32):
        super().__init__()
        self.N = N
        self.register_buffer("A_hat", _norm_adj(adj))
        self.W = nn.Linear(1, gcn_hidden, bias=False)
        self.bias = nn.Parameter(torch.zeros(gcn_hidden))
        self.rnn = nn.GRU(gcn_hidden, rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, 1)

    def forward(self, x):                                   # predict step T from steps 0..T-2
        B, T, N = x.shape
        xi = x[:, :-1, :]
        feats = self.W(xi.unsqueeze(-1))
        prop = torch.relu(torch.einsum("ij,btjh->btih", self.A_hat, feats) + self.bias)
        seq = prop.permute(0, 2, 1, 3).reshape(B * N, T - 1, -1)
        out, _ = self.rnn(seq)
        return self.head(out[:, -1, :]).reshape(B, N)       # (B,N)

    def loss(self, x):
        return F.mse_loss(self.forward(x), x[:, -1, :])

    def step_score(self, x):
        return (self.forward(x) - x[:, -1, :]).pow(2)       # (B,N) at last step


# =============================================================================
# TRAIN / SCORE / FLAG / EVAL
# =============================================================================
def train_model(model, train_loader, val_loader, mode, device,
                lr=1e-3, epochs=50, patience=10, pos_weight=None):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device) if pos_weight is not None else None)
    stop = EarlyStopping(patience=patience, mode="f1" if mode == "sup" else "loss")

    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(device)
            opt.zero_grad()
            loss = crit(model(x), y.to(device)) if mode == "sup" else model.loss(x)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            if mode == "sup":
                ps, ys = [], []
                for x, y in val_loader:
                    ps.append(torch.sigmoid(model(x.to(device))).cpu().numpy().reshape(-1))
                    ys.append(y.numpy().reshape(-1))
                ys = np.concatenate(ys)
                # Ranking metric (AUPRC): tracks learning even when no fixed 0.5
                # cut separates the classes. F1@0.5 stays 0 on rare-positive data
                # and would stop training immediately.
                metric = average_precision_score(ys, np.concatenate(ps)) if ys.any() else 0.0
            else:
                metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in val_loader]))
        if stop.step(metric, model):
            stopped = ep + 1
            break
    else:
        stopped = epochs
    stop.restore(model)
    model.eval()
    with torch.no_grad():
        train_metric = float(np.mean([model.loss(x.to(device)).item() for x, _ in train_loader]))
    print(f"   trained {stopped} epoch(s) | train_loss={train_metric:.4f} | best {'val_AUPRC' if mode == 'sup' else 'val_loss'}={stop.best:.4f}")
    return model


def score_series(model, values, seq_len, stride, mode, device):
    """Return a (T, N) anomaly-score map, averaging overlapping windows."""
    model.eval()
    T, N = values.shape
    if mode == "forecast":
        stride = 1                                # score every incoming step
    acc = np.zeros((T, N)); cnt = np.zeros((T, N))
    starts = list(range(0, T - seq_len + 1, stride))
    if starts and starts[-1] != T - seq_len:
        starts.append(T - seq_len)
    with torch.no_grad():
        for s in starts:
            x = torch.tensor(values[s:s + seq_len], dtype=torch.float32)[None].to(device)
            if mode == "sup":
                sc = torch.sigmoid(model(x))[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            elif mode == "vae":
                sc = model.cell_score(x)[0].cpu().numpy()
                acc[s:s + seq_len] += sc; cnt[s:s + seq_len] += 1
            else:                                            # forecast: last step only
                t = s + seq_len - 1
                acc[t] += model.step_score(x)[0].cpu().numpy(); cnt[t] += 1
    cnt[cnt == 0] = 1
    return acc / cnt


def flag_supervised(val_scores, val_mask, test_scores, objective="cell", min_overlap=1):
    """Tune one global probability cut on val. objective='cell' maximises cell F1;
    'event' maximises point-adjust F1 (aligns the cut with run-level scoring)."""
    thr, best = float(np.median(val_scores)), -1.0
    for t in np.linspace(val_scores.min(), val_scores.max(), 100):
        vp = (val_scores >= t).astype(int)
        if objective == "event":
            f = event_metrics(vp, val_mask, min_overlap)["PA_F1"]
        else:
            f = f1_score(val_mask.reshape(-1), vp.reshape(-1).astype(int), zero_division=0)
        if f > best:
            best, thr = f, t
    return (test_scores >= thr).astype(int), float(thr)


def flag_unsupervised(clean_scores, test_scores, method="percentile", pct=99.0, k=4.0):
    """Per-sensor threshold (higher score = more anomalous).
    percentile : pct-th percentile of clean-TRAIN scores (assumes train ~= test).
    median     : median + k*MAD of the scored set itself (adapts to its distribution).
    mean       : mean + k*std of the scored set itself.
    Robust methods need the score to rank anomalies ABOVE normals (AUROC > 0.5)."""
    if method == "percentile":
        thr = np.percentile(clean_scores, pct, axis=0)
    elif method == "median":
        med = np.median(test_scores, axis=0)
        mad = np.median(np.abs(test_scores - med), axis=0)
        thr = med + k * (mad + 1e-9)
    elif method == "mean":
        thr = test_scores.mean(0) + k * (test_scores.std(0) + 1e-9)
    else:
        raise ValueError(f"unknown unsup_threshold '{method}'")
    return (test_scores >= thr).astype(int), thr


def cell_metrics(pred_mask, true_mask):
    p = pred_mask.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    return {"Accuracy":  round(accuracy_score(t, p), 4),
            "Precision": round(precision_score(t, p, zero_division=0), 4),
            "Recall":    round(recall_score(t, p, zero_division=0), 4),
            "F1":        round(f1_score(t, p, zero_division=0), 4)}


def ranking_metrics(scores, true_mask):
    """Threshold-free quality of the score map (does it rank anomalies high?)."""
    s = scores.reshape(-1)
    t = (np.asarray(true_mask) > 0).reshape(-1).astype(int)
    if t.sum() == 0 or t.sum() == len(t):
        return {"AUROC": float("nan"), "AUPRC": float("nan")}
    return {"AUROC": round(roc_auc_score(t, s), 4),
            "AUPRC": round(average_precision_score(t, s), 4)}


def _segments(col):
    """Contiguous runs of 1s in a 1D bool array -> list of (start, end_exclusive)."""
    segs, i, n = [], 0, len(col)
    while i < n:
        if col[i]:
            j = i
            while j < n and col[j]:
                j += 1
            segs.append((i, j)); i = j
        else:
            i += 1
    return segs


def _filter_short_runs(mask, min_len):
    """Drop predicted runs shorter than min_len per sensor (removes isolated false
    flags; useful when real anomalies are sustained)."""
    if min_len <= 1:
        return mask
    out = np.zeros_like(mask)
    for s in range(mask.shape[1]):
        for a, b in _segments(mask[:, s].astype(bool)):
            if b - a >= min_len:
                out[a:b, s] = 1
    return out


def event_metrics(pred_mask, true_mask, min_overlap=1):
    """Score whole anomalous runs instead of individual cells (per sensor).
    point-adjust (PA): if >=min_overlap cells of a true run are flagged, the WHOLE
        run is credited; P/R/F1 then computed point-wise on the adjusted mask
        (standard time-series AD protocol).
    event-overlap (EV): a true run is a hit if it's flagged at all; a predicted run
        that touches no true run is a false alarm. P/R/F1 over runs."""
    pred = pred_mask.astype(bool); true = (np.asarray(true_mask) > 0)
    T, N = true.shape
    adj = pred.copy()
    tp_ev = fn_ev = fp_ev = pred_ev = 0
    for s in range(N):
        for a, b in _segments(true[:, s]):
            if pred[a:b, s].sum() >= min_overlap:
                adj[a:b, s] = True; tp_ev += 1
            else:
                fn_ev += 1
        for a, b in _segments(pred[:, s]):
            pred_ev += 1
            if not true[a:b, s].any():
                fp_ev += 1
    p, t = adj.reshape(-1), true.reshape(-1)
    ev_p = (pred_ev - fp_ev) / pred_ev if pred_ev else 0.0
    ev_r = tp_ev / (tp_ev + fn_ev) if (tp_ev + fn_ev) else 0.0
    ev_f1 = 2 * ev_p * ev_r / (ev_p + ev_r) if (ev_p + ev_r) else 0.0
    return {"PA_Precision": round(precision_score(t, p, zero_division=0), 4),
            "PA_Recall":    round(recall_score(t, p, zero_division=0), 4),
            "PA_F1":        round(f1_score(t, p, zero_division=0), 4),
            "EV_Recall":    round(ev_r, 4),
            "EV_F1":        round(ev_f1, 4),
            "Events":       tp_ev + fn_ev}


# =============================================================================
# CLEAN OUTPUT
# =============================================================================
def _sensor_meta(csv_path):
    try:
        df = pd.read_csv(csv_path)
        cols = [c for c in ("thing_name", "thing_ip") if c in df.columns]
        if not cols:
            return {}
        g = df.drop_duplicates("sensor_name").set_index("sensor_name")[cols]
        return {s: row.to_dict() for s, row in g.iterrows()}
    except Exception:
        return {}


def export_clean(corrupted, scaler, pred_mask, dt_index, sensors, meta, out_path):
    """Drop flagged (datetime, sensor) readings; write the rest as long-format CSV."""
    inv = scaler.inverse_transform(corrupted)
    rows = []
    for ti, dt in enumerate(dt_index):
        dt = pd.Timestamp(dt)
        for sj, sname in enumerate(sensors):
            if pred_mask[ti, sj]:
                continue
            m = meta.get(sname, {})
            rows.append({
                "datetime": dt, "date": dt.strftime("%Y-%m-%d"),
                "time": dt.strftime("%H:%M"), "seconds": dt.timestamp(),
                "state": float(inv[ti, sj]), "sensor_name": sname,
                "thing_name": m.get("thing_name"), "thing_ip": m.get("thing_ip"),
            })
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    return df


# =============================================================================
# REGISTRY & ORCHESTRATION
# =============================================================================
def build_registry(N, seq_len, adj, beta):
    return [
        {"name": "UnivariateBaseline", "mode": "sup",      "graph": "none (per-sensor)",
         "build": lambda: UnivariateBaseline(N)},
        {"name": "TemporalCNN",        "mode": "sup",      "graph": "none",
         "build": lambda: TemporalCNN(N, seq_len)},
        {"name": "StackedLSTM",        "mode": "sup",      "graph": "none",
         "build": lambda: StackedLSTM(N)},
        {"name": "SpatioTemporalGNN",  "mode": "sup",      "graph": "predefined",
         "build": lambda: GCNDetector(N, adj=adj)},
        {"name": "GNN_LearnableGraph", "mode": "sup",      "graph": "learned",
         "build": lambda: GCNDetector(N, adj=None)},
        {"name": "GAT_LearnedGraph",   "mode": "sup",      "graph": "learned",
         "build": lambda: GATDetector(N)},
        {"name": "GraphAnomalyVAE",    "mode": "vae",      "graph": "learned",
         "build": lambda: GraphAnomalyVAE(N, seq_len, beta=beta)},
        {"name": "GNNForecaster",      "mode": "forecast", "graph": "predefined",
         "build": lambda: GNNForecaster(N, adj=adj)},
    ]


def run(csv_path, adj_csv_path=None, groups=None,
        seq_len=24, stride=4, batch_size=64, max_epochs=50, lr=1e-3, patience=15,
        anomaly_frac=0.10, train_anomaly_frac=0.05, anomaly_type="all",
        binary_anomaly=False, run_len=1, binary_mode="flip",
        unsup_percentile=99.0, unsup_threshold="percentile", robust_k=4.0,
        event_min_overlap=1, min_event_len=1, sup_objective="cell",
        plot_residuals=False,
        pos_weight_cap=10.0, train_frac=0.70, val_frac=0.15,
        out_dir="cleaned_output", models=None, beta=1.0, seed=42):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*64}\n  Sensor-level anomaly cleaning  |  device={device}\n{'='*64}")
    wide = load_and_pivot(csv_path)
    sensors = list(wide.columns); N = len(sensors)
    raw = wide.values.astype(np.float32); T = len(raw)
    n_tr, n_va = int(T * train_frac), int(T * val_frac)

    scaler = MinMaxScaler()
    tr = scaler.fit_transform(raw[:n_tr])
    va = scaler.transform(raw[n_tr:n_tr + n_va])
    te = scaler.transform(raw[n_tr + n_va:])
    dt_test = wide.index[n_tr + n_va:]

    if groups is not None:
        adj = adjacency_from_groups(groups, sensors)
    elif adj_csv_path is not None:
        _, adj = load_edge_index_from_csv(adj_csv_path, sensors)
    else:
        raise ValueError("provide either `groups` (dict) or `adj_csv_path`")

    print("\nInjecting anomalies (val / test, and a small fraction into train):")
    _inj = (lambda v, f: inject_binary(v, f, run_len, binary_mode, adj)) if binary_anomaly \
        else (lambda v, f: inject(v, f, anomaly_type, adj))
    va_c, va_m = _inj(va, anomaly_frac)
    te_c, te_m = _inj(te, anomaly_frac)
    tr_c, tr_m = _inj(tr, train_anomaly_frac)
    meta = _sensor_meta(csv_path)

    # data loaders
    zeros_tr = np.zeros_like(tr, dtype=int)
    sup_tr = DataLoader(WindowDS(tr_c, tr_m, seq_len, stride), batch_size=batch_size, shuffle=True)
    sup_va = DataLoader(WindowDS(va_c, va_m, seq_len, stride), batch_size=batch_size, shuffle=False)
    uns_tr = DataLoader(WindowDS(tr, zeros_tr, seq_len, stride), batch_size=batch_size, shuffle=True)
    uns_va = DataLoader(WindowDS(va_c, np.zeros_like(va, int), seq_len, stride), batch_size=batch_size, shuffle=False)

    pos = tr_m.sum(); neg = tr_m.size - pos
    pw = min(neg / max(pos, 1), pos_weight_cap)
    pos_weight = torch.tensor([pw], dtype=torch.float32)
    print(f"\nTrain cell anomaly rate: {100 * pos / tr_m.size:.2f}%  |  pos_weight={pw:.1f} "
          f"(capped at {pos_weight_cap})")

    registry = build_registry(N, seq_len, adj, beta)
    if models:
        registry = [m for m in registry if m["name"] in models]

    results = []
    for entry in registry:
        name, mode = entry["name"], entry["mode"]
        if mode == "sup" and train_anomaly_frac <= 0:
            print(f"\n[skip] {name}: supervised model needs train_anomaly_frac > 0")
            continue
        print(f"\n{'-'*64}\n  {name}  (mode={mode}, graph={entry['graph']})\n{'-'*64}")
        model = entry["build"]()

        if mode == "sup":
            model = train_model(model, sup_tr, sup_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience, pos_weight=pos_weight)
            val_scores = score_series(model, va_c, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_supervised(val_scores, va_m, test_scores,
                                        objective=sup_objective, min_overlap=event_min_overlap)
        else:
            model = train_model(model, uns_tr, uns_va, mode, device,
                                lr=lr, epochs=max_epochs, patience=patience)
            clean_scores = score_series(model, tr, seq_len, stride, mode, device)
            test_scores = score_series(model, te_c, seq_len, stride, mode, device)
            pred, thr = flag_unsupervised(clean_scores, test_scores,
                                          method=unsup_threshold, pct=unsup_percentile, k=robust_k)
            thr_arr = thr if isinstance(thr, np.ndarray) else np.full(N, thr)
            fpr = ((test_scores > thr_arr) & (te_m == 0)).sum(0) / max((te_m == 0).sum(), 1)
            print(f"   clean-test FPR (mean across sensors): {fpr.mean():.4f}")

        pred = _filter_short_runs(pred, min_event_len)
        m = cell_metrics(pred, te_m)              # metrics on flagged mask (before removal)
        m.update(ranking_metrics(test_scores, te_m))   # threshold-free diagnostics
        m.update(event_metrics(pred, te_m, event_min_overlap))   # run/segment-level
        m["_scores"] = test_scores                    # kept for residual plots
        out_path = os.path.join(out_dir, f"clean_{name}.csv")
        clean_df = export_clean(te_c, scaler, pred, dt_test, sensors, meta, out_path)

        removed = int(pred.sum())
        print(f"   F1={m['F1']:.4f}  Prec={m['Precision']:.4f}  Rec={m['Recall']:.4f}  "
              f"Acc={m['Accuracy']:.4f}  | removed {removed} readings -> {out_path}")
        m.update({"Model": name, "Mode": mode, "Removed": removed, "CleanFile": out_path})
        results.append(m)

    if plot_residuals:
        try:
            import matplotlib.pyplot as plt
            for r in results:
                s = r.get("_scores")
                if s is None: continue
                fig, ax = plt.subplots(figsize=(7, 3))
                ax.hist(s[te_m == 0].reshape(-1), bins=80, alpha=0.6, label="normal", density=True)
                ax.hist(s[te_m == 1].reshape(-1), bins=80, alpha=0.6, label="anomalous", density=True)
                ax.set_title(r["Model"]); ax.set_xlabel("score S"); ax.legend()
                plt.tight_layout(); plt.show()
        except ImportError:
            print("[plot_residuals] matplotlib not available")

    _print_table(results, te_m)
    return results


def _print_table(results, true_mask):
    inj = int((true_mask > 0).sum())
    cols = ["Model", "Mode", "AUPRC", "Cell_F1", "PA_F1", "EV_Recall", "EV_F1", "Removed"]
    key = {"Cell_F1": "F1"}                       # map display name -> result key
    rows = [[r.get(key.get(c, c)) for c in cols] for r in results]
    rows.sort(key=lambda r: (r[4] if r[4] == r[4] else -1), reverse=True)   # by PA_F1
    nev = results[0].get("Events", "?") if results else "?"
    print(f"\n{'='*78}\n  DETECTION  (injected cells: {inj}, anomalous runs/events: {nev})\n{'='*78}")
    if _TAB:
        print(tabulate(rows, headers=cols, tablefmt="fancy_grid", floatfmt=".4f"))
    else:
        print("  ".join(f"{c:<12}" for c in cols))
        for r in rows:
            print("  ".join(f"{str(v):<12}" for v in r))
    print("\nCell_F1  : exact per-(timestep,sensor) match (strictest).")
    print("PA_F1    : point-adjust - a run is credited if any of its cells is flagged.")
    print("EV_Recall/EV_F1 : per-run - did we catch each anomalous segment.")
    print("Event scoring is meaningful for sustained anomalies (run_len > 1).")


# =============================================================================
# ENTRY POINT  (terminal:  python anomaly_cleaning.py --csv ... --adj_csv ...)
#              (jupyter  :  edit the call below and %run the file)
# =============================================================================
# =============================================================================
# ENTRY POINT
#   notebook : edit SENSOR_GROUPS / PARAMS below and run the file (or import `run`)
#   terminal : python anomaly_cleaning.py --csv data.csv --groups_json groups.json \
#                     --binary_anomaly --binary_mode ood --run_len 5
# =============================================================================

# --- domain knowledge: edit this grouping (sensors sharing a group get connected) --
SENSOR_GROUPS = {
    'G1': ['Front Motion Sensor', 'Lab Door Motion Sensor', 'Sonar Sensor'],
    'G2': ['Lab Door Motion Sensor', 'Right Motion Sensor', 'Sonar Sensor'],
    'G3': ['Lab Door Motion Sensor', 'Lab Light Sensor', 'Sonar Sensor'],
    'G4': ['Lab Door Motion Sensor', 'Lab Door Sensor', 'Sonar Sensor'],
    'G5': ['Lab Door Motion Sensor', 'Left Motion Sensor', 'Sonar Sensor'],
    'G6': ['Back Door Sensor', 'Back Motion Sensor', 'Vibration Sensor'],
}

if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        # ---- NOTEBOOK: all tunables in one place ----
        run(
            csv_path="Data/Lab_Data/sead_anomaly_free_lab_data_1h.csv",
            groups=SENSOR_GROUPS,          # use the grouping above (set None to use adj_csv_path)
            adj_csv_path=None,             # e.g. "adjacency.csv" if not using groups
            # windowing / training
            seq_len=24, stride=4, batch_size=64, max_epochs=100, lr=1e-3, patience=20,
            # anomaly injection (binary data)
            binary_anomaly=True, binary_mode="ood", run_len=5,   # mode: flip|ood|stuck|neighbor|rate
            anomaly_frac=0.10, train_anomaly_frac=0.08,
            # supervised threshold objective + unsupervised threshold
            sup_objective="event",                               # "cell" or "event"
            unsup_threshold="mean", robust_k=1.5, unsup_percentile=99.0,
            # event scoring / misc
            event_min_overlap=1, min_event_len=1, beta=0.1,
            models=None,                                         # None = all; or e.g. ["TemporalCNN","GraphAnomalyVAE"]
            out_dir="cleaned_output",plot_residuals=True,
        )
    else:
        # ---- TERMINAL ----
        import json
        p = argparse.ArgumentParser(description="Sensor-level anomaly cleaning")
        p.add_argument("--csv", required=True)
        p.add_argument("--adj_csv", default=None, help="labelled adjacency CSV (omit if using --groups_json)")
        p.add_argument("--groups_json", default=None, help="JSON file: {group: [sensor,...]}")
        p.add_argument("--seq_len", type=int, default=24)
        p.add_argument("--stride", type=int, default=4)
        p.add_argument("--batch_size", type=int, default=64)
        p.add_argument("--max_epochs", type=int, default=100)
        p.add_argument("--lr", type=float, default=1e-3)
        p.add_argument("--patience", type=int, default=20)
        p.add_argument("--anomaly_frac", type=float, default=0.10)
        p.add_argument("--train_anomaly_frac", type=float, default=0.08)
        p.add_argument("--anomaly_type", type=str, default="all")
        p.add_argument("--binary_anomaly", action="store_true")
        p.add_argument("--binary_mode", type=str, default="flip",
                       choices=["flip", "ood", "stuck", "neighbor", "rate", "contextual", "swap"])
        p.add_argument("--run_len", type=int, default=5)
        p.add_argument("--sup_objective", type=str, default="cell", choices=["cell", "event"])
        p.add_argument("--unsup_threshold", type=str, default="percentile",
                       choices=["percentile", "median", "mean"])
        p.add_argument("--robust_k", type=float, default=4.0)
        p.add_argument("--unsup_percentile", type=float, default=99.0)
        p.add_argument("--min_event_len", type=int, default=1)
        p.add_argument("--beta", type=float, default=1.0)
        p.add_argument("--out_dir", type=str, default="cleaned_output")
        a = p.parse_args()
        groups = json.load(open(a.groups_json)) if a.groups_json else None
        run(csv_path=a.csv, adj_csv_path=a.adj_csv, groups=groups,
            seq_len=a.seq_len, stride=a.stride, batch_size=a.batch_size, max_epochs=a.max_epochs,
            lr=a.lr, patience=a.patience, anomaly_frac=a.anomaly_frac,
            train_anomaly_frac=a.train_anomaly_frac, anomaly_type=a.anomaly_type,
            binary_anomaly=a.binary_anomaly, run_len=a.run_len, binary_mode=a.binary_mode,
            sup_objective=a.sup_objective, unsup_threshold=a.unsup_threshold, robust_k=a.robust_k,
            unsup_percentile=a.unsup_percentile, min_event_len=a.min_event_len,
            beta=a.beta, out_dir=a.out_dir)


  Sensor-level anomaly cleaning  |  device=cuda
  Loaded 33199 timestamps x 10 sensors
   Sensors: ['Back Door Sensor', 'Back Motion Sensor', 'Front Motion Sensor', 'Lab Door Motion Sensor', 'Lab Door Sensor', 'Lab Light Sensor', 'Left Motion Sensor', 'Right Motion Sensor', 'Sonar Sensor', 'Vibration Sensor']
   Built graph from 6 groups: 14 undirected edges | density 31.1%

Injecting anomalies (val / test, and a small fraction into train):

Train cell anomaly rate: 7.71%  |  pos_weight=10.0 (capped at 10.0)

----------------------------------------------------------------
  UnivariateBaseline  (mode=sup, graph=none (per-sensor))
----------------------------------------------------------------


AttributeError: 'UnivariateBaseline' object has no attribute 'loss'